## (old) Synthetic data instead of reading pkl file

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("synthetic_wave_keogram_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Spatial grid: 0 to 500 km in both image directions.
# This matches the coordinate convention used when plotting the original images.
DX_KM = 1.0
DY_KM = 1.0
FIELD_SIZE_X_KM = 500.0
FIELD_SIZE_Y_KM = 500.0

# Temporal grid
DT_SECONDS = 120
TOTAL_TIME_MIN = 300

# Synthetic Gaussian wave packet
WAVELENGTH_KM = 4000.0
GAUSSIAN_WIDTH_KM = 6000.0  # Gaussian sigma
MAX_AMPLITUDE = 0.3
PROPAGATION_ANGLE_DEG = 45.0

# A speed is required to animate the moving wave.
# Change this value as needed.
WAVE_SPEED_M_S = -600.0

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = TOTAL_TIME_MIN
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Distance rows to plot from each keogram
PROFILE_DISTANCES_KM = [-250, 0, 250]

# Percentile used to determine the color range separately
# for each directional keogram.
KEOGRAM_COLOR_PERCENTILE = 99

# Fractional padding added to the automatically determined
# y-range of each profile plot.
PROFILE_Y_PADDING_FRACTION = 0.05


# ==================================================
# Coordinate and time grids
# ==================================================
NX = int(round(FIELD_SIZE_X_KM / DX_KM)) + 1
NY = int(round(FIELD_SIZE_Y_KM / DY_KM)) + 1

x_km = np.linspace(
    0.0,
    FIELD_SIZE_X_KM,
    NX,
)
y_km = np.linspace(
    0.0,
    FIELD_SIZE_Y_KM,
    NY,
)

times_s = np.arange(
    0,
    TOTAL_TIME_MIN * 60 + DT_SECONDS,
    DT_SECONDS,
)
times_min = times_s / 60.0
nt = len(times_s)

X_KM, Y_KM = np.meshgrid(x_km, y_km)

# Centered coordinates used only for constructing the wave.
# X_CENTERED_KM increases toward the right (east).
# Y_CENTERED_KM increases downward on the displayed image (south),
# matching imshow(origin="upper") and the original keogram-cut convention.
cx_km = 0.5 * (x_km.min() + x_km.max())
cy_km = 0.5 * (y_km.min() + y_km.max())

X_CENTERED_KM = X_KM - cx_km
Y_CENTERED_KM = Y_KM - cy_km


# ==================================================
# Generate synthetic moving Gaussian wave packet
# ==================================================
theta = np.deg2rad(PROPAGATION_ANGLE_DEG)

# The propagation angle follows the displayed image convention:
#   0 deg   = right/east
#   45 deg  = down-right/southeast
#   90 deg  = down/south
# Because image y increases downward, +sin(theta) is consistent
# with the directional keogram cuts in the original image script.
parallel_distance_km = (
    X_CENTERED_KM * np.cos(theta)
    + Y_CENTERED_KM * np.sin(theta)
)

wavenumber_rad_per_km = 2.0 * np.pi / WAVELENGTH_KM
wave_speed_km_s = WAVE_SPEED_M_S / 1000.0

wave_cube = np.empty(
    (nt, NY, NX),
    dtype=np.float32,
)

for it, time_s in enumerate(times_s):
    packet_center_km = wave_speed_km_s * time_s
    moving_coordinate_km = (
        parallel_distance_km - packet_center_km
    )

    gaussian_envelope = np.exp(
        -0.5
        * (
            moving_coordinate_km
            / GAUSSIAN_WIDTH_KM
        ) ** 2
    )

    wave_phase = (
        wavenumber_rad_per_km
        * moving_coordinate_km
    )

    wave_cube[it] = (
        MAX_AMPLITUDE
        * gaussian_envelope
        * np.cos(wave_phase)
    ).astype(np.float32)

print("Synthetic wave-cube shape:", wave_cube.shape)
print(
    f"Field size: {FIELD_SIZE_X_KM:.0f} km x "
    f"{FIELD_SIZE_Y_KM:.0f} km"
)
print(f"Wavelength: {WAVELENGTH_KM:.0f} km")
print(
    f"Gaussian width (sigma): "
    f"{GAUSSIAN_WIDTH_KM:.0f} km"
)
print(
    f"Propagation direction: "
    f"{PROPAGATION_ANGLE_DEG:.1f} degrees"
)
print(f"Maximum amplitude: {MAX_AMPLITUDE}")
print(f"Wave speed: {WAVE_SPEED_M_S:.1f} m/s")


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)
cbar.set_label("Perturbation amplitude")

title = ax.set_title("")

ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])

    title.set_text(
        f"Synthetic Wave Field | "
        f"Frame {frame}/{nt - 1} | "
        f"t = {times_min[frame]:.1f} min"
    )

    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "synthetic_wave_animation.mp4"

writer = FFMpegWriter(
    fps=FPS,
    bitrate=3000,
)

ani.save(
    out_mp4,
    writer=writer,
    dpi=DPI,
)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# Only show every 15 degrees, but still create
# keograms every 5 degrees later
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_first = fig_first.colorbar(
    im_first,
    ax=ax_first,
    fraction=0.046,
    pad=0.04,
)
cbar_first.set_label("Perturbation amplitude")

cx_km = 0.5 * (x_km.min() + x_km.max())
cy_km = 0.5 * (y_km.min() + y_km.max())

half_length_km = 0.5 * min(
    x_km.max() - x_km.min(),
    y_km.max() - y_km.min(),
)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [
        cx_km - dx_line,
        cx_km + dx_line,
    ]

    y_line = [
        cy_km - dy_line,
        cy_km + dy_line,
    ]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title(
    "First Synthetic Wave Frame with Directional Keogram Cuts"
)
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = (
    OUT_DIR
    / "first_perturbation_frame_with_keogram_cuts_every_15deg.png"
)

fig_first.savefig(
    out_first,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(
    wave_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the center
    of the image cube.

    angle_deg:
        0 degrees  = E-W slice
        90 degrees = N-S slice
    """

    nt, NY, NX = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    # Unit vector along the keogram direction
    ux = np.cos(theta)
    uy = np.sin(theta)

    # Unit vector perpendicular to the keogram direction
    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2

    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels)
        - width_pixels // 2
    )

    keogram = np.full(
        (nt, len(distances_pix)),
        np.nan,
        dtype=float,
    )

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = (
                cx
                + distances_pix * ux
                + off * px
            )

            y_sample = (
                cy
                + distances_pix * uy
                + off * py
            )

            valid = (
                (x_sample >= 0)
                & (x_sample <= NX - 1)
                & (y_sample >= 0)
                & (y_sample <= NY - 1)
            )

            values = np.full(
                len(distances_pix),
                np.nan,
            )

            values[valid] = map_coordinates(
                wave_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(
            sampled_slices,
            axis=0,
        )

    # This assumes square 1-km pixels.
    # Because DX_KM = DY_KM = 1 km here,
    # one pixel along any angle equals 1 km.
    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Create and save directional keograms
#
# For each keogram:
#   1. Save the keogram image
#   2. Save the full keogram array
#   3. Plot the -250, 0, and +250 km time series
#      together on one plot
# ==================================================
saved_keograms = []
saved_profile_plots = []

for angle_deg in KEOGRAM_ANGLES_DEG:

    # ----------------------------------------------
    # Make keogram
    # ----------------------------------------------
    keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    # Determine a separate symmetric color range for this keogram,
    # using only the time/distance region that will be displayed.
    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]
    finite_keogram_values = visible_keogram[
        np.isfinite(visible_keogram)
    ]

    if finite_keogram_values.size > 0:
        keogram_vmax = np.nanpercentile(
            np.abs(finite_keogram_values),
            KEOGRAM_COLOR_PERCENTILE,
        )
    else:
        keogram_vmax = 1.0

    if not np.isfinite(keogram_vmax) or keogram_vmax == 0:
        keogram_vmax = 1.0

    keogram_vmin = -keogram_vmax

    # ----------------------------------------------
    # Plot keogram
    # ----------------------------------------------
    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=keogram_vmin,
        vmax=keogram_vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(
        im_k,
        ax=ax_k,
        fraction=0.046,
        pad=0.04,
    )
    cbar_k.set_label("Perturbation amplitude")

    ax_k.set_title(
        f"Synthetic Directional Keogram: {angle_deg:.0f}°"
    )
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    # Keep the requested time and distance display ranges.
    ax_k.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax_k.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    # Show the three selected rows on the keogram
    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        ax_k.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    plt.tight_layout()

    out_png = (
        OUT_DIR
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.png"
    )

    out_npy = (
        OUT_DIR
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.npy"
    )

    fig_k.savefig(
        out_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_k)

    np.save(
        out_npy,
        keogram,
    )

    saved_keograms.append(out_png)

    # ----------------------------------------------
    # Plot -250, 0, and +250 km rows
    # on the same time-series plot
    # ----------------------------------------------
    fig_profile, ax_profile = plt.subplots(
        figsize=(10, 5)
    )

    profile_data = {}

    for requested_distance in PROFILE_DISTANCES_KM:

        # Find nearest available keogram distance
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        # Keogram shape is:
        # time x distance
        profile = keogram[:, distance_index]

        profile_data[
            f"{actual_distance:.1f}_km"
        ] = profile

        ax_profile.plot(
            times_min,
            profile,
            linewidth=1.3,
            label=(
                f"{actual_distance:+.1f} km"
            ),
        )

    ax_profile.axhline(
        0,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    ax_profile.set_title(
        f"Synthetic Keogram Profiles at "
        f"-250, 0, and +250 km: {angle_deg:.0f}°"
    )

    ax_profile.set_xlabel("Time (min)")
    ax_profile.set_ylabel("Perturbation amplitude")

    # Keep the requested time range and calculate the amplitude
    # range independently for this profile figure.
    ax_profile.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

    finite_profile_parts = [
        profile[
            visible_time_mask & np.isfinite(profile)
        ]
        for profile in profile_data.values()
        if np.any(
            visible_time_mask & np.isfinite(profile)
        )
    ]

    if finite_profile_parts:
        finite_profile_values = np.concatenate(finite_profile_parts)
        profile_ymin = np.min(finite_profile_values)
        profile_ymax = np.max(finite_profile_values)
        profile_span = profile_ymax - profile_ymin

        if profile_span == 0:
            profile_span = max(abs(profile_ymax), 1.0)

        profile_padding = (
            PROFILE_Y_PADDING_FRACTION * profile_span
        )

        ax_profile.set_ylim(
            profile_ymin - profile_padding,
            profile_ymax + profile_padding,
        )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
    )

    plt.tight_layout()

    out_profile_png = (
        OUT_DIR
        / (
            f"synthetic_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.png"
        )
    )

    fig_profile.savefig(
        out_profile_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_profile)

    saved_profile_plots.append(
        out_profile_png
    )

    # ----------------------------------------------
    # Save the three profiles as a NumPy file
    # ----------------------------------------------
    out_profile_npy = (
        OUT_DIR
        / (
            f"synthetic_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.npy"
        )
    )

    profile_array = np.column_stack(
        [
            times_min,
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km + 250))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 0))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 250))]:.1f}_km"
            ],
        ]
    )

    np.save(
        out_profile_npy,
        profile_array,
    )


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(
    OUT_DIR / "times_s.npy",
    times_s,
)

np.save(
    OUT_DIR / "times_min.npy",
    times_min,
)

np.save(
    OUT_DIR / "x_km.npy",
    x_km,
)

np.save(
    OUT_DIR / "y_km.npy",
    y_km,
)

np.save(
    OUT_DIR / "keogram_angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "first_frame_cut_angles_deg.npy",
    PLOT_CUT_ANGLES_DEG,
)

np.save(
    OUT_DIR / "profile_distances_km.npy",
    np.asarray(PROFILE_DISTANCES_KM),
)

with open(
    OUT_DIR / "synthetic_wave_parameters.txt",
    "w",
) as f:
    f.write(f"Field size X (km): {FIELD_SIZE_X_KM}\n")
    f.write(f"Field size Y (km): {FIELD_SIZE_Y_KM}\n")
    f.write(f"Grid spacing X (km): {DX_KM}\n")
    f.write(f"Grid spacing Y (km): {DY_KM}\n")
    f.write(f"Time step (s): {DT_SECONDS}\n")
    f.write(f"Total time (min): {TOTAL_TIME_MIN}\n")
    f.write(f"Wavelength (km): {WAVELENGTH_KM}\n")
    f.write(
        f"Gaussian width sigma (km): "
        f"{GAUSSIAN_WIDTH_KM}\n"
    )
    f.write(f"Maximum amplitude: {MAX_AMPLITUDE}\n")
    f.write(
        f"Propagation angle (deg): "
        f"{PROPAGATION_ANGLE_DEG}\n"
    )
    f.write(f"Wave speed (m/s): {WAVE_SPEED_M_S}\n")


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_last = fig_last.colorbar(
    im_last,
    ax=ax_last,
    fraction=0.046,
    pad=0.04,
)
cbar_last.set_label("Perturbation amplitude")

ax_last.set_title("Final Synthetic Wave Frame")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_perturbation_frame.png"

fig_last.savefig(
    out_last,
    dpi=DPI,
    bbox_inches="tight",
)

plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for file_path in saved_keograms:
    print(file_path)

print("\nKeogram distance-profile plots:")
for file_path in saved_profile_plots:
    print(file_path)

print(
    f"\nNumber of directional keograms made: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    f"Number of profile plots made: "
    f"{len(saved_profile_plots)}"
)

print(
    f"First-frame cut lines shown: every 15 degrees, "
    f"{len(PLOT_CUT_ANGLES_DEG)} lines"
)

print(
    f"Keogram/profile time range: 0 to "
    f"{KEOGRAM_TIME_MIN_MAX} min"
)

print(
    f"Keogram distance range: {KEOGRAM_DISTANCE_MIN} to "
    f"{KEOGRAM_DISTANCE_MAX} km"
)

print(
    "Each keogram and profile plot uses its own "
    "amplitude range."
)

print(
    "Profile distances requested:",
    PROFILE_DISTANCES_KM,
)

print("Output folder:", OUT_DIR)


Synthetic wave-cube shape: (151, 501, 501)
Field size: 500 km x 500 km
Wavelength: 4000 km
Gaussian width (sigma): 6000 km
Propagation direction: 45.0 degrees
Maximum amplitude: 0.3
Wave speed: -600.0 m/s
Saved:
synthetic_wave_keogram_outputs/synthetic_wave_animation.mp4
synthetic_wave_keogram_outputs/first_perturbation_frame_with_keogram_cuts_every_15deg.png
synthetic_wave_keogram_outputs/final_perturbation_frame.png

Directional keograms:
synthetic_wave_keogram_outputs/synthetic_keogram_angle_000.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_005.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_010.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_015.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_020.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_025.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_030.png
synthetic_wave_keogram_outputs/synthetic_keogram_angle_035.png
synthetic_wave_keogram_outputs/synthetic_keogram_an

## (old) Introduce bias

In [31]:
# ==================================================
# Version note
# This file includes:
#   1. Two moving Gaussian wave packets
#   2. Time-varying directional bias using spline interpolation
#   3. Spatially uniform linear temporal slope
#   4. Configurable synthetic-wave time shift
#   5. Configurable additive Gaussian noise
# Final field:
#   total_cube = (
#       wave_1_cube
#       + wave_2_cube
#       + bias_cube
#       + temporal_slope_cube
#       + noise_cube
#   )
# ==================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates
from scipy.interpolate import CubicSpline


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("synthetic_wave_keogram_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Spatial grid: 0 to 500 km in both image directions.
# This matches the coordinate convention used when plotting the original images.
DX_KM = 1.0
DY_KM = 1.0
FIELD_SIZE_X_KM = 500.0
FIELD_SIZE_Y_KM = 500.0

# Temporal grid
DT_SECONDS = 120
TOTAL_TIME_MIN = 300

# Synthetic Gaussian wave packet 1
WAVE_1_WAVELENGTH_KM = 4800.0
WAVE_1_GAUSSIAN_SIGMA_KM = 8000.0
WAVE_1_MAX_AMPLITUDE = 0.18
WAVE_1_PROPAGATION_ANGLE_DEG = 46.0
WAVE_1_SPEED_M_S = -600.0

# Synthetic Gaussian wave packet 2
# The requested width is given as FWHM, so it is converted to Gaussian sigma.
WAVE_2_WAVELENGTH_KM = 850.0
WAVE_2_GAUSSIAN_SIGMA_KM = 1500
WAVE_2_MAX_AMPLITUDE = 0.20
WAVE_2_PROPAGATION_ANGLE_DEG = 15.0
WAVE_2_SPEED_M_S = -650.0

# Additive Gaussian noise applied independently to every pixel and frame.
# NOISE_STD is the standard deviation in perturbation-amplitude units.
# Set NOISE_STD = 0.0 to disable noise.
# The fixed seed makes the synthetic noise reproducible.
NOISE_MEAN = 0.0
NOISE_STD = 0.02
NOISE_RANDOM_SEED = 42

# Shift only the synthetic waves in time.
# Positive = advance the wave to a later wave state.
# Negative = delay the wave to an earlier wave state.
# Examples: +10.0 advances by 10 minutes; -10.0 delays by 10 minutes.
WAVE_TIME_SHIFT_MIN = 50.0  # applied to both waves

# Time-varying directional linear bias control points.
# Angle convention matches the displayed image and keogram convention:
#   0 deg   = right/east
#   90 deg  = down/south
#  -90 deg  = up/north
# Gradient units are perturbation amplitude per 100 km.
BIAS_CONTROL_TIMES_MIN = np.array([0.0, 100.0, 150.0, 200.0, 300.0])
BIAS_CONTROL_ANGLES_DEG = np.array([-80.0, -90.0, -100.0, -110.0, -120])
BIAS_CONTROL_GRADIENTS_PER_100KM = np.array([0.10, 0.03, 0.01, -0.002, -0.01])

# Spatially uniform temporal slope added to every pixel.
# It changes linearly from +1.0 at 0 minutes to -0.2 at 300 minutes.
TEMPORAL_SLOPE_START = .8
TEMPORAL_SLOPE_END = -0.2

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = TOTAL_TIME_MIN
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Distance rows to plot from each keogram
PROFILE_DISTANCES_KM = [-250, 0, 250]

# Percentile used to determine the color range separately
# for each directional keogram.
KEOGRAM_COLOR_PERCENTILE = 99

# Fractional padding added to the automatically determined
# y-range of each profile plot.
PROFILE_Y_PADDING_FRACTION = 0.05


# ==================================================
# Coordinate and time grids
# ==================================================
NX = int(round(FIELD_SIZE_X_KM / DX_KM)) + 1
NY = int(round(FIELD_SIZE_Y_KM / DY_KM)) + 1

x_km = np.linspace(
    0.0,
    FIELD_SIZE_X_KM,
    NX,
)
y_km = np.linspace(
    0.0,
    FIELD_SIZE_Y_KM,
    NY,
)

times_s = np.arange(
    0,
    TOTAL_TIME_MIN * 60 + DT_SECONDS,
    DT_SECONDS,
)
times_min = times_s / 60.0
nt = len(times_s)

X_KM, Y_KM = np.meshgrid(x_km, y_km)

# Centered coordinates used only for constructing the wave.
# X_CENTERED_KM increases toward the right (east).
# Y_CENTERED_KM increases downward on the displayed image (south),
# matching imshow(origin="upper") and the original keogram-cut convention.
cx_km = 0.5 * (x_km.min() + x_km.max())
cy_km = 0.5 * (y_km.min() + y_km.max())

X_CENTERED_KM = X_KM - cx_km
Y_CENTERED_KM = Y_KM - cy_km


# ==================================================
# Generate two synthetic moving Gaussian wave packets
# ==================================================
def generate_gaussian_wave_packet(
    wavelength_km,
    gaussian_sigma_km,
    max_amplitude,
    propagation_angle_deg,
    wave_speed_m_s,
    wave_time_shift_min,
):
    """Generate one moving Gaussian-envelope cosine wave packet."""
    theta = np.deg2rad(propagation_angle_deg)

    # Displayed-image angle convention:
    #   0 deg   = right/east
    #   45 deg  = down-right/southeast
    #   90 deg  = down/south
    parallel_distance_km = (
        X_CENTERED_KM * np.cos(theta)
        + Y_CENTERED_KM * np.sin(theta)
    )

    wavenumber_rad_per_km = 2.0 * np.pi / wavelength_km
    wave_speed_km_s = wave_speed_m_s / 1000.0

    wave_cube = np.empty((nt, NY, NX), dtype=np.float32)

    for it, time_s in enumerate(times_s):
        shifted_wave_time_s = time_s + wave_time_shift_min * 60.0
        packet_center_km = wave_speed_km_s * shifted_wave_time_s
        moving_coordinate_km = parallel_distance_km - packet_center_km

        gaussian_envelope = np.exp(
            -0.5 * (moving_coordinate_km / gaussian_sigma_km) ** 2
        )
        wave_phase = wavenumber_rad_per_km * moving_coordinate_km

        wave_cube[it] = (
            max_amplitude
            * gaussian_envelope
            * np.cos(wave_phase)
        ).astype(np.float32)

    return wave_cube


wave_1_cube = generate_gaussian_wave_packet(
    wavelength_km=WAVE_1_WAVELENGTH_KM,
    gaussian_sigma_km=WAVE_1_GAUSSIAN_SIGMA_KM,
    max_amplitude=WAVE_1_MAX_AMPLITUDE,
    propagation_angle_deg=WAVE_1_PROPAGATION_ANGLE_DEG,
    wave_speed_m_s=WAVE_1_SPEED_M_S,
    wave_time_shift_min=WAVE_TIME_SHIFT_MIN,
)

wave_2_cube = generate_gaussian_wave_packet(
    wavelength_km=WAVE_2_WAVELENGTH_KM,
    gaussian_sigma_km=WAVE_2_GAUSSIAN_SIGMA_KM,
    max_amplitude=WAVE_2_MAX_AMPLITUDE,
    propagation_angle_deg=WAVE_2_PROPAGATION_ANGLE_DEG,
    wave_speed_m_s=WAVE_2_SPEED_M_S,
    wave_time_shift_min=WAVE_TIME_SHIFT_MIN,
)

# Combined wave-only field before adding bias, slope, and noise.
original_wave_cube = wave_1_cube + wave_2_cube


# ==================================================
# Generate additive Gaussian noise
# ==================================================
rng = np.random.default_rng(NOISE_RANDOM_SEED)

noise_cube = rng.normal(
    loc=NOISE_MEAN,
    scale=NOISE_STD,
    size=(nt, NY, NX),
).astype(np.float32)


# ==================================================
# Construct time-varying directional bias with splines
# ==================================================
# Two independent cubic splines are used:
#   1. bias direction versus time
#   2. bias gradient versus time
#
# The bias is zero at the field center. At each time, it changes linearly
# along the interpolated direction. For example, +0.10 per 100 km means
# the bias increases by 0.10 for every 100 km traveled in that direction.
bias_angle_spline = CubicSpline(
    BIAS_CONTROL_TIMES_MIN,
    BIAS_CONTROL_ANGLES_DEG,
    bc_type="natural",
)

bias_gradient_spline = CubicSpline(
    BIAS_CONTROL_TIMES_MIN,
    BIAS_CONTROL_GRADIENTS_PER_100KM,
    bc_type="natural",
)

bias_angles_deg = bias_angle_spline(times_min)
bias_gradients_per_100km = bias_gradient_spline(times_min)

bias_cube = np.empty((nt, NY, NX), dtype=np.float32)

for it, (angle_deg, gradient_per_100km) in enumerate(
    zip(bias_angles_deg, bias_gradients_per_100km)
):
    angle_rad = np.deg2rad(angle_deg)

    # Signed distance from the image center along the bias direction.
    bias_distance_km = (
        X_CENTERED_KM * np.cos(angle_rad)
        + Y_CENTERED_KM * np.sin(angle_rad)
    )

    bias_cube[it] = (
        gradient_per_100km * bias_distance_km / 100.0
    ).astype(np.float32)

# ==================================================
# Construct spatially uniform linear temporal slope
# ==================================================
# The value is the same at every spatial pixel for a given frame.
# It varies linearly in time from TEMPORAL_SLOPE_START at 0 minutes
# to TEMPORAL_SLOPE_END at TOTAL_TIME_MIN.
temporal_slope_values = np.linspace(
    TEMPORAL_SLOPE_START,
    TEMPORAL_SLOPE_END,
    nt,
    dtype=np.float32,
)

temporal_slope_cube = np.broadcast_to(
    temporal_slope_values[:, None, None],
    (nt, NY, NX),
).copy()

# Total synthetic field used for animation, directional cuts, and keograms.
total_cube = (
    original_wave_cube
    + bias_cube
    + temporal_slope_cube
    + noise_cube
)

print("Synthetic cube shape:", total_cube.shape)
print(
    f"Field size: {FIELD_SIZE_X_KM:.0f} km x "
    f"{FIELD_SIZE_Y_KM:.0f} km"
)
print("Wave 1:")
print(f"  Wavelength: {WAVE_1_WAVELENGTH_KM:.0f} km")
print(f"  Gaussian sigma: {WAVE_1_GAUSSIAN_SIGMA_KM:.1f} km")
print(f"  Propagation direction: {WAVE_1_PROPAGATION_ANGLE_DEG:.1f} degrees")
print(f"  Maximum amplitude: {WAVE_1_MAX_AMPLITUDE}")
print(f"  Speed: {WAVE_1_SPEED_M_S:.1f} m/s")
print("Wave 2:")
print(f"  Wavelength: {WAVE_2_WAVELENGTH_KM:.0f} km")
print(f"  Gaussian FWHM: {WAVE_2_GAUSSIAN_FWHM_KM:.1f} km")
print(f"  Gaussian sigma: {WAVE_2_GAUSSIAN_SIGMA_KM:.1f} km")
print(f"  Propagation direction: {WAVE_2_PROPAGATION_ANGLE_DEG:.1f} degrees")
print(f"  Maximum amplitude: {WAVE_2_MAX_AMPLITUDE}")
print(f"  Speed: {WAVE_2_SPEED_M_S:.1f} m/s")
print(
    f"Gaussian noise: mean={NOISE_MEAN:+.4f}, "
    f"standard deviation={NOISE_STD:.4f}, "
    f"seed={NOISE_RANDOM_SEED}"
)
print(f"Wave time shift: {WAVE_TIME_SHIFT_MIN:+.1f} min")
print(
    f"Temporal slope: {TEMPORAL_SLOPE_START:+.3f} at 0 min to "
    f"{TEMPORAL_SLOPE_END:+.3f} at {TOTAL_TIME_MIN:.0f} min"
)

print("\nBias control points:")
for t, angle, gradient in zip(
    BIAS_CONTROL_TIMES_MIN,
    BIAS_CONTROL_ANGLES_DEG,
    BIAS_CONTROL_GRADIENTS_PER_100KM,
):
    print(
        f"  t={t:6.1f} min: direction={angle:6.1f} deg, "
        f"gradient={gradient:+.3f} per 100 km"
    )


# ==================================================
# Color-limit helper
# ==================================================
def symmetric_limits(data, percentile=99.0):
    vmax = np.nanpercentile(np.abs(data), percentile)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return -vmax, vmax


wave_1_vmin, wave_1_vmax = symmetric_limits(wave_1_cube)
wave_2_vmin, wave_2_vmax = symmetric_limits(wave_2_cube)
original_vmin, original_vmax = symmetric_limits(original_wave_cube)
bias_vmin, bias_vmax = symmetric_limits(bias_cube)
slope_vmin, slope_vmax = symmetric_limits(temporal_slope_cube)
noise_vmin, noise_vmax = symmetric_limits(noise_cube)
total_vmin, total_vmax = symmetric_limits(total_cube)


# ==================================================
# Plot the spline-interpolated bias parameters
# ==================================================
fig_bias_parameters, (ax_angle, ax_gradient) = plt.subplots(
    2,
    1,
    figsize=(9, 7),
    sharex=True,
)

ax_angle.plot(times_min, bias_angles_deg, linewidth=2.0)
ax_angle.scatter(
    BIAS_CONTROL_TIMES_MIN,
    BIAS_CONTROL_ANGLES_DEG,
    zorder=3,
    label="Specified control points",
)
ax_angle.set_ylabel("Bias direction (deg)")
ax_angle.set_title("Spline-interpolated directional bias")
ax_angle.grid(True, alpha=0.3)
ax_angle.legend(loc="best")

ax_gradient.plot(times_min, bias_gradients_per_100km, linewidth=2.0)
ax_gradient.scatter(
    BIAS_CONTROL_TIMES_MIN,
    BIAS_CONTROL_GRADIENTS_PER_100KM,
    zorder=3,
    label="Specified control points",
)
ax_gradient.axhline(0.0, linewidth=0.8, linestyle="--", alpha=0.7)
ax_gradient.set_xlabel("Time (min)")
ax_gradient.set_ylabel("Gradient per 100 km")
ax_gradient.grid(True, alpha=0.3)
ax_gradient.legend(loc="best")

plt.tight_layout()
out_bias_parameters = OUT_DIR / "bias_direction_and_gradient_splines.png"
fig_bias_parameters.savefig(out_bias_parameters, dpi=DPI, bbox_inches="tight")
plt.close(fig_bias_parameters)


# ==================================================
# Seven-panel animation: both waves, combined wave, bias, slope, noise, total
# ==================================================
fig, axes = plt.subplots(1, 7, figsize=(38, 5.6), constrained_layout=True)

animation_specs = [
    (wave_1_cube, "Wave 1", wave_1_vmin, wave_1_vmax),
    (wave_2_cube, "Wave 2", wave_2_vmin, wave_2_vmax),
    (original_wave_cube, "Wave 1 + Wave 2", original_vmin, original_vmax),
    (bias_cube, "Directional bias", bias_vmin, bias_vmax),
    (temporal_slope_cube, "Uniform temporal slope", slope_vmin, slope_vmax),
    (noise_cube, "Gaussian noise", noise_vmin, noise_vmax),
    (total_cube, "Total field", total_vmin, total_vmax),
]

animation_images = []
for ax, (cube, panel_title, panel_vmin, panel_vmax) in zip(axes, animation_specs):
    image = ax.imshow(
        cube[0],
        origin="upper",
        cmap=CMAP,
        vmin=panel_vmin,
        vmax=panel_vmax,
        extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
    )
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04).set_label(
        "Perturbation amplitude"
    )
    ax.set_title(panel_title)
    ax.set_xlabel("E-W distance (km)")
    ax.set_ylabel("N-S distance (km)")
    ax.set_aspect("equal")
    animation_images.append(image)

animation_title = fig.suptitle("")


def update(frame):
    for image, (cube, _, _, _) in zip(animation_images, animation_specs):
        image.set_data(cube[frame])

    animation_title.set_text(
        f"Two Synthetic Waves with Bias, Slope, and Noise | "
        f"t = {times_min[frame]:.1f} min | "
        f"bias angle = {bias_angles_deg[frame]:.1f} deg | "
        f"gradient = {bias_gradients_per_100km[frame]:+.4f} per 100 km | "
        f"slope = {temporal_slope_values[frame]:+.4f}"
    )
    return (*animation_images, animation_title)


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "synthetic_two_waves_bias_slope_noise_total_animation.mp4"
writer = FFMpegWriter(fps=FPS, bitrate=5000)
ani.save(out_mp4, writer=writer, dpi=DPI)
plt.close(fig)


# ==================================================
# Save seven-panel snapshots at the bias control times
# ==================================================
def save_seven_panel_snapshot(frame_index, output_path):
    fig_snapshot, axes_snapshot = plt.subplots(
        1, 7, figsize=(38, 5.6), constrained_layout=True
    )

    snapshot_specs = [
        (wave_1_cube[frame_index], "Wave 1", wave_1_vmin, wave_1_vmax),
        (wave_2_cube[frame_index], "Wave 2", wave_2_vmin, wave_2_vmax),
        (original_wave_cube[frame_index], "Wave 1 + Wave 2", original_vmin, original_vmax),
        (bias_cube[frame_index], "Directional bias", bias_vmin, bias_vmax),
        (temporal_slope_cube[frame_index], "Uniform temporal slope", slope_vmin, slope_vmax),
        (noise_cube[frame_index], "Gaussian noise", noise_vmin, noise_vmax),
        (total_cube[frame_index], "Total field", total_vmin, total_vmax),
    ]

    for ax, (field, panel_title, panel_vmin, panel_vmax) in zip(
        axes_snapshot, snapshot_specs
    ):
        image = ax.imshow(
            field,
            origin="upper",
            cmap=CMAP,
            vmin=panel_vmin,
            vmax=panel_vmax,
            extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
        )
        fig_snapshot.colorbar(
            image, ax=ax, fraction=0.046, pad=0.04
        ).set_label("Perturbation amplitude")
        ax.set_title(panel_title)
        ax.set_xlabel("E-W distance (km)")
        ax.set_ylabel("N-S distance (km)")
        ax.set_aspect("equal")

    fig_snapshot.suptitle(
        f"t = {times_min[frame_index]:.1f} min | "
        f"bias angle = {bias_angles_deg[frame_index]:.1f} deg | "
        f"gradient = {bias_gradients_per_100km[frame_index]:+.4f} per 100 km | "
        f"slope = {temporal_slope_values[frame_index]:+.4f}"
    )
    fig_snapshot.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close(fig_snapshot)


saved_bias_snapshots = []
for control_time_min in BIAS_CONTROL_TIMES_MIN:
    frame_index = int(np.argmin(np.abs(times_min - control_time_min)))
    output_path = OUT_DIR / (
        f"two_waves_bias_slope_noise_total_at_{control_time_min:03.0f}min.png"
    )
    save_seven_panel_snapshot(frame_index, output_path)
    saved_bias_snapshots.append(output_path)


# ==================================================
# Save first total-field frame with directional keogram cuts
# Only show every 15 degrees, but still create keograms every 5 degrees later
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    total_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=total_vmin,
    vmax=total_vmax,
    extent=[
        x_km.min(),
        x_km.max(),
        y_km.max(),
        y_km.min(),
    ],
)

cbar_first = fig_first.colorbar(
    im_first,
    ax=ax_first,
    fraction=0.046,
    pad=0.04,
)
cbar_first.set_label("Perturbation amplitude")

half_length_km = 0.5 * min(
    x_km.max() - x_km.min(),
    y_km.max() - y_km.min(),
)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [cx_km - dx_line, cx_km + dx_line]
    y_line = [cy_km - dy_line, cy_km + dy_line]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title(
    "First Total Field Frame with Directional Keogram Cuts"
)
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")
ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()
out_first = OUT_DIR / "first_total_field_with_keogram_cuts_every_15deg.png"
fig_first.savefig(out_first, dpi=DPI, bbox_inches="tight")
plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(
    wave_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the center
    of the image cube.

    angle_deg:
        0 degrees  = E-W slice
        90 degrees = N-S slice
    """

    nt, NY, NX = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    # Unit vector along the keogram direction
    ux = np.cos(theta)
    uy = np.sin(theta)

    # Unit vector perpendicular to the keogram direction
    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2

    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels)
        - width_pixels // 2
    )

    keogram = np.full(
        (nt, len(distances_pix)),
        np.nan,
        dtype=float,
    )

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = (
                cx
                + distances_pix * ux
                + off * px
            )

            y_sample = (
                cy
                + distances_pix * uy
                + off * py
            )

            valid = (
                (x_sample >= 0)
                & (x_sample <= NX - 1)
                & (y_sample >= 0)
                & (y_sample <= NY - 1)
            )

            values = np.full(
                len(distances_pix),
                np.nan,
            )

            values[valid] = map_coordinates(
                wave_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(
            sampled_slices,
            axis=0,
        )

    # This assumes square 1-km pixels.
    # Because DX_KM = DY_KM = 1 km here,
    # one pixel along any angle equals 1 km.
    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Create and save directional keograms
#
# For each keogram:
#   1. Save the keogram image
#   2. Save the full keogram array
#   3. Plot the -250, 0, and +250 km time series
#      together on one plot
# ==================================================
saved_keograms = []
saved_profile_plots = []

for angle_deg in KEOGRAM_ANGLES_DEG:

    # ----------------------------------------------
    # Make keogram
    # ----------------------------------------------
    keogram, distance_km = make_directional_keogram(
        total_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    # Determine a separate symmetric color range for this keogram,
    # using only the time/distance region that will be displayed.
    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]
    finite_keogram_values = visible_keogram[
        np.isfinite(visible_keogram)
    ]

    if finite_keogram_values.size > 0:
        keogram_vmax = np.nanpercentile(
            np.abs(finite_keogram_values),
            KEOGRAM_COLOR_PERCENTILE,
        )
    else:
        keogram_vmax = 1.0

    if not np.isfinite(keogram_vmax) or keogram_vmax == 0:
        keogram_vmax = 1.0

    keogram_vmin = -keogram_vmax

    # ----------------------------------------------
    # Plot keogram
    # ----------------------------------------------
    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=keogram_vmin,
        vmax=keogram_vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(
        im_k,
        ax=ax_k,
        fraction=0.046,
        pad=0.04,
    )
    cbar_k.set_label("Perturbation amplitude")

    ax_k.set_title(
        f"Synthetic Directional Keogram: {angle_deg:.0f}°"
    )
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    # Keep the requested time and distance display ranges.
    ax_k.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax_k.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    # Show the three selected rows on the keogram
    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        ax_k.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    plt.tight_layout()

    out_png = (
        OUT_DIR
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.png"
    )

    out_npy = (
        OUT_DIR
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.npy"
    )

    fig_k.savefig(
        out_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_k)

    np.save(
        out_npy,
        keogram,
    )

    saved_keograms.append(out_png)

    # ----------------------------------------------
    # Plot -250, 0, and +250 km rows
    # on the same time-series plot
    # ----------------------------------------------
    fig_profile, ax_profile = plt.subplots(
        figsize=(10, 5)
    )

    profile_data = {}

    for requested_distance in PROFILE_DISTANCES_KM:

        # Find nearest available keogram distance
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )

        actual_distance = distance_km[distance_index]

        # Keogram shape is:
        # time x distance
        profile = keogram[:, distance_index]

        profile_data[
            f"{actual_distance:.1f}_km"
        ] = profile

        ax_profile.plot(
            times_min,
            profile,
            linewidth=1.3,
            label=(
                f"{actual_distance:+.1f} km"
            ),
        )

    ax_profile.axhline(
        0,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    ax_profile.set_title(
        f"Synthetic Keogram Profiles at "
        f"-250, 0, and +250 km: {angle_deg:.0f}°"
    )

    ax_profile.set_xlabel("Time (min)")
    ax_profile.set_ylabel("Perturbation amplitude")

    # Keep the requested time range and calculate the amplitude
    # range independently for this profile figure.
    ax_profile.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

    finite_profile_parts = [
        profile[
            visible_time_mask & np.isfinite(profile)
        ]
        for profile in profile_data.values()
        if np.any(
            visible_time_mask & np.isfinite(profile)
        )
    ]

    if finite_profile_parts:
        finite_profile_values = np.concatenate(finite_profile_parts)
        profile_ymin = np.min(finite_profile_values)
        profile_ymax = np.max(finite_profile_values)
        profile_span = profile_ymax - profile_ymin

        if profile_span == 0:
            profile_span = max(abs(profile_ymax), 1.0)

        profile_padding = (
            PROFILE_Y_PADDING_FRACTION * profile_span
        )

        ax_profile.set_ylim(
            profile_ymin - profile_padding,
            profile_ymax + profile_padding,
        )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
    )

    plt.tight_layout()

    out_profile_png = (
        OUT_DIR
        / (
            f"synthetic_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.png"
        )
    )

    fig_profile.savefig(
        out_profile_png,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig_profile)

    saved_profile_plots.append(
        out_profile_png
    )

    # ----------------------------------------------
    # Save the three profiles as a NumPy file
    # ----------------------------------------------
    out_profile_npy = (
        OUT_DIR
        / (
            f"synthetic_keogram_profiles_"
            f"minus250_0_plus250km_"
            f"angle_{angle_deg:03.0f}.npy"
        )
    )

    profile_array = np.column_stack(
        [
            times_min,
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km + 250))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 0))]:.1f}_km"
            ],
            profile_data[
                f"{distance_km[np.argmin(np.abs(distance_km - 250))]:.1f}_km"
            ],
        ]
    )

    np.save(
        out_profile_npy,
        profile_array,
    )


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(
    OUT_DIR / "times_s.npy",
    times_s,
)

np.save(
    OUT_DIR / "times_min.npy",
    times_min,
)

np.save(
    OUT_DIR / "x_km.npy",
    x_km,
)

np.save(
    OUT_DIR / "y_km.npy",
    y_km,
)

np.save(
    OUT_DIR / "keogram_angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)

np.save(
    OUT_DIR / "first_frame_cut_angles_deg.npy",
    PLOT_CUT_ANGLES_DEG,
)

np.save(
    OUT_DIR / "profile_distances_km.npy",
    np.asarray(PROFILE_DISTANCES_KM),
)

np.save(OUT_DIR / "wave_1_cube.npy", wave_1_cube)
np.save(OUT_DIR / "wave_2_cube.npy", wave_2_cube)
np.save(OUT_DIR / "original_wave_cube.npy", original_wave_cube)
np.save(OUT_DIR / "bias_cube.npy", bias_cube)
np.save(OUT_DIR / "temporal_slope_values.npy", temporal_slope_values)
np.save(OUT_DIR / "temporal_slope_cube.npy", temporal_slope_cube)
np.save(OUT_DIR / "noise_cube.npy", noise_cube)
np.save(OUT_DIR / "total_cube.npy", total_cube)
np.save(OUT_DIR / "bias_angles_deg.npy", bias_angles_deg)
np.save(
    OUT_DIR / "bias_gradients_per_100km.npy",
    bias_gradients_per_100km,
)
np.save(
    OUT_DIR / "bias_control_times_min.npy",
    BIAS_CONTROL_TIMES_MIN,
)
np.save(
    OUT_DIR / "bias_control_angles_deg.npy",
    BIAS_CONTROL_ANGLES_DEG,
)
np.save(
    OUT_DIR / "bias_control_gradients_per_100km.npy",
    BIAS_CONTROL_GRADIENTS_PER_100KM,
)

with open(
    OUT_DIR / "synthetic_wave_parameters.txt",
    "w",
) as f:
    f.write(f"Field size X (km): {FIELD_SIZE_X_KM}\n")
    f.write(f"Field size Y (km): {FIELD_SIZE_Y_KM}\n")
    f.write(f"Grid spacing X (km): {DX_KM}\n")
    f.write(f"Grid spacing Y (km): {DY_KM}\n")
    f.write(f"Time step (s): {DT_SECONDS}\n")
    f.write(f"Total time (min): {TOTAL_TIME_MIN}\n")
    f.write("Wave 1:\n")
    f.write(f"  Wavelength (km): {WAVE_1_WAVELENGTH_KM}\n")
    f.write(f"  Gaussian sigma (km): {WAVE_1_GAUSSIAN_SIGMA_KM}\n")
    f.write(f"  Maximum amplitude: {WAVE_1_MAX_AMPLITUDE}\n")
    f.write(f"  Propagation angle (deg): {WAVE_1_PROPAGATION_ANGLE_DEG}\n")
    f.write(f"  Speed (m/s): {WAVE_1_SPEED_M_S}\n")
    f.write("Wave 2:\n")
    f.write(f"  Wavelength (km): {WAVE_2_WAVELENGTH_KM}\n")
    f.write(f"  Gaussian FWHM (km): {WAVE_2_GAUSSIAN_FWHM_KM}\n")
    f.write(f"  Gaussian sigma (km): {WAVE_2_GAUSSIAN_SIGMA_KM}\n")
    f.write(f"  Maximum amplitude: {WAVE_2_MAX_AMPLITUDE}\n")
    f.write(f"  Propagation angle (deg): {WAVE_2_PROPAGATION_ANGLE_DEG}\n")
    f.write(f"  Speed (m/s): {WAVE_2_SPEED_M_S}\n")
    f.write(f"Noise distribution: Gaussian\n")
    f.write(f"Noise mean: {NOISE_MEAN}\n")
    f.write(f"Noise standard deviation: {NOISE_STD}\n")
    f.write(f"Noise random seed: {NOISE_RANDOM_SEED}\n")
    f.write(f"Wave time shift (min): {WAVE_TIME_SHIFT_MIN}\n")
    f.write(f"Temporal slope at 0 min: {TEMPORAL_SLOPE_START}\n")
    f.write(
        f"Temporal slope at {TOTAL_TIME_MIN} min: "
        f"{TEMPORAL_SLOPE_END}\n"
    )
    f.write("Bias interpolation: natural cubic spline\n")
    f.write("Bias is zero at the field center\n")
    f.write("Bias angle convention: 0 deg east, +90 deg south\n")
    f.write("Bias control points (time_min, angle_deg, gradient_per_100km):\n")
    for t, angle, gradient in zip(
        BIAS_CONTROL_TIMES_MIN,
        BIAS_CONTROL_ANGLES_DEG,
        BIAS_CONTROL_GRADIENTS_PER_100KM,
    ):
        f.write(f"  {t}, {angle}, {gradient}\n")


# ==================================================
# Save final seven-panel frame
# ==================================================
out_last = OUT_DIR / "final_two_waves_bias_slope_noise_total_frame.png"
save_seven_panel_snapshot(nt - 1, out_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)
print(out_bias_parameters)

print("\nBias-control snapshots:")
for file_path in saved_bias_snapshots:
    print(file_path)

print("\nDirectional keograms:")
for file_path in saved_keograms:
    print(file_path)

print("\nKeogram distance-profile plots:")
for file_path in saved_profile_plots:
    print(file_path)

print(
    f"\nNumber of directional keograms made: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print(
    f"Number of profile plots made: "
    f"{len(saved_profile_plots)}"
)

print(
    f"First-frame cut lines shown: every 15 degrees, "
    f"{len(PLOT_CUT_ANGLES_DEG)} lines"
)

print(
    f"Keogram/profile time range: 0 to "
    f"{KEOGRAM_TIME_MIN_MAX} min"
)

print(
    f"Keogram distance range: {KEOGRAM_DISTANCE_MIN} to "
    f"{KEOGRAM_DISTANCE_MAX} km"
)

print(
    "Each keogram and profile plot uses its own "
    "amplitude range."
)

print(
    "Profile distances requested:",
    PROFILE_DISTANCES_KM,
)

print("Output folder:", OUT_DIR)

print("\nBias definition:")
print("  Spatial bias is zero at the image center.")
print("  Positive gradient increases along the listed direction.")
print("  Negative gradient decreases along the listed direction.")
print("  Direction and gradient are independently interpolated with natural cubic splines.")
print(
    "  Directional keograms are calculated from total_cube = "
    "wave_1_cube + wave_2_cube + bias_cube + temporal_slope_cube + noise_cube."
)


Synthetic cube shape: (151, 501, 501)
Field size: 500 km x 500 km
Wave 1:
  Wavelength: 4800 km
  Gaussian sigma: 8000.0 km
  Propagation direction: 46.0 degrees
  Maximum amplitude: 0.18
  Speed: -600.0 m/s
Wave 2:
  Wavelength: 850 km
  Gaussian FWHM: 2000.0 km
  Gaussian sigma: 1500.0 km
  Propagation direction: 15.0 degrees
  Maximum amplitude: 0.2
  Speed: -650.0 m/s
Gaussian noise: mean=+0.0000, standard deviation=0.0200, seed=42
Wave time shift: +50.0 min
Temporal slope: +0.800 at 0 min to -0.200 at 300 min

Bias control points:
  t=   0.0 min: direction= -80.0 deg, gradient=+0.100 per 100 km
  t= 100.0 min: direction= -90.0 deg, gradient=+0.030 per 100 km
  t= 150.0 min: direction=-100.0 deg, gradient=+0.010 per 100 km
  t= 200.0 min: direction=-110.0 deg, gradient=-0.002 per 100 km
  t= 300.0 min: direction=-120.0 deg, gradient=-0.010 per 100 km
Saved:
synthetic_wave_keogram_outputs/synthetic_two_waves_bias_slope_noise_total_animation.mp4
synthetic_wave_keogram_outputs/first_t

## (old) Synthetic Data, 6 variants

In [42]:
# ==================================================
# Synthetic-wave directional keograms: six variants
#
# This script creates and saves:
#   1. Waves only
#   2. Bias only
#   3. Waves + temporal slope
#   4. Waves + directional bias
#   5. Waves + temporal slope + directional bias
#   6. Waves + temporal slope + directional bias + noise
#
# Each variant is saved in a separate folder. For every keogram
# angle, one comparison figure is also saved with six vertically
# stacked profile subplots. Each subplot contains the profiles at
# -250, 0, and +250 km.
# ==================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("synthetic_wave_keogram_outputs_six_variants")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Spatial grid
DX_KM = 1.0
DY_KM = 1.0
FIELD_SIZE_X_KM = 500.0
FIELD_SIZE_Y_KM = 500.0

# Temporal grid
DT_SECONDS = 120
TOTAL_TIME_MIN = 500

# Synthetic Gaussian wave packet
WAVELENGTH_KM = 3150.0
GAUSSIAN_WIDTH_KM = 6000.0  # Gaussian sigma
MAX_AMPLITUDE = 0.15
PROPAGATION_ANGLE_DEG = 40.0
WAVE_SPEED_M_S = -350.0

# Shift only the wave state in time.
# Positive = advance; negative = delay.
WAVE_TIME_SHIFT_MIN = 90.0

# Spatially uniform temporal slope.
# The same value is added to every pixel in a frame.
TEMPORAL_SLOPE_START = 1.0
TEMPORAL_SLOPE_END = -0.4

# Time-varying directional bias anchor points derived from the
# rounded-path plane-fit vector plot.
#
# Component convention:
#   +x = right/east
#   +y = down/south
#
# Units below are perturbation amplitude per 100 km.
BIAS_PATH_START_TIME_MIN = 120.0
BIAS_PATH_END_TIME_MIN = 500.0

BIAS_ANCHOR_TIMES_MIN = np.array(
    [140.0, 230.0, 370.0, 480.0],
    dtype=np.float64,
)

BIAS_ANCHOR_X_PER_100KM = np.array(
    [0.003743, -0.015397, -0.016520, -0.006721],
    dtype=np.float64,
)

BIAS_ANCHOR_Y_PER_100KM = np.array(
    [-0.014641, 0.005227, 0.011898, 0.020024],
    dtype=np.float64,
)

# Use a mostly straight rounded polyline, similar to the rounded-path
# vector plot. Smaller corner_fraction makes the path straighter.
BIAS_CORNER_FRACTION = 0.10
BIAS_POINTS_PER_STRAIGHT = 150
BIAS_POINTS_PER_CORNER = 60

# Additive Gaussian noise
NOISE_MEAN = 0.0
NOISE_STD = 0.02
NOISE_RANDOM_SEED = 42

DPI = 200
CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = TOTAL_TIME_MIN
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Distance rows shown in each profile subplot
PROFILE_DISTANCES_KM = [-250, 0, 250]

# Automatic plot limits
KEOGRAM_COLOR_PERCENTILE = 99.0
PROFILE_Y_PADDING_FRACTION = 0.05

# When True, all six profile subplots use one common y range.
# This makes the amplitudes directly comparable between variants.
# Set False to scale each subplot independently.
PROFILE_USE_SHARED_Y_LIMITS = True

# Plane-fit analysis
# Each frame is fitted as z = intercept + slope_x*x + slope_y*y.
# The direction points toward increasing perturbation.
# Angle convention: 0 deg = +x/east, +90 deg = +y/south.
# Angles are wrapped to [-180, 180).
# Scalar slope units are perturbation amplitude per 100 km.
PLANE_FIT_OUTPUT_DIR = OUT_DIR / "08_plane_fit_analysis"
PLANE_FIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Slope-vector trajectory plots
# Plot every Nth vector arrow to reduce crowding.
SLOPE_VECTOR_ARROW_STEP = 10
SLOPE_VECTOR_CMAP = "viridis"


# ==================================================
# Output folders and labels
# ==================================================
VARIANT_SPECS = [
    (
        "01_waves_only",
        "Waves only",
    ),
    (
        "02_bias_only",
        "Bias only",
    ),
    (
        "03_waves_plus_slope",
        "Waves + slope",
    ),
    (
        "04_waves_plus_bias",
        "Waves + bias",
    ),
    (
        "05_waves_plus_slope_plus_bias",
        "Waves + slope + bias",
    ),
    (
        "06_waves_plus_slope_plus_bias_plus_noise",
        "Waves + slope + bias + noise",
    ),
]

VARIANT_DIRS = {}
for folder_name, label in VARIANT_SPECS:
    variant_dir = OUT_DIR / folder_name
    (variant_dir / "keogram_images").mkdir(parents=True, exist_ok=True)
    (variant_dir / "keogram_arrays").mkdir(parents=True, exist_ok=True)
    (variant_dir / "profile_arrays").mkdir(parents=True, exist_ok=True)
    VARIANT_DIRS[folder_name] = variant_dir

COMPARISON_PROFILE_DIR = OUT_DIR / "07_profile_comparisons_six_subplots"
COMPARISON_PROFILE_DIR.mkdir(parents=True, exist_ok=True)


# ==================================================
# Coordinate and time grids
# ==================================================
NX = int(round(FIELD_SIZE_X_KM / DX_KM)) + 1
NY = int(round(FIELD_SIZE_Y_KM / DY_KM)) + 1

x_km = np.linspace(0.0, FIELD_SIZE_X_KM, NX)
y_km = np.linspace(0.0, FIELD_SIZE_Y_KM, NY)

times_s = np.arange(
    0,
    TOTAL_TIME_MIN * 60 + DT_SECONDS,
    DT_SECONDS,
)
times_min = times_s / 60.0
nt = len(times_s)

X_KM, Y_KM = np.meshgrid(x_km, y_km)

cx_km = 0.5 * (x_km.min() + x_km.max())
cy_km = 0.5 * (y_km.min() + y_km.max())

X_CENTERED_KM = X_KM - cx_km
Y_CENTERED_KM = Y_KM - cy_km


# ==================================================
# Generate the moving Gaussian wave packet
# ==================================================
theta = np.deg2rad(PROPAGATION_ANGLE_DEG)

parallel_distance_km = (
    X_CENTERED_KM * np.cos(theta)
    + Y_CENTERED_KM * np.sin(theta)
)

wavenumber_rad_per_km = 2.0 * np.pi / WAVELENGTH_KM
wave_speed_km_s = WAVE_SPEED_M_S / 1000.0

wave_cube = np.empty((nt, NY, NX), dtype=np.float32)

for it, time_s in enumerate(times_s):
    shifted_time_s = time_s + WAVE_TIME_SHIFT_MIN * 60.0
    packet_center_km = wave_speed_km_s * shifted_time_s
    moving_coordinate_km = parallel_distance_km - packet_center_km

    gaussian_envelope = np.exp(
        -0.5
        * (moving_coordinate_km / GAUSSIAN_WIDTH_KM) ** 2
    )

    wave_phase = (
        wavenumber_rad_per_km * moving_coordinate_km
    )

    wave_cube[it] = (
        MAX_AMPLITUDE
        * gaussian_envelope
        * np.cos(wave_phase)
    ).astype(np.float32)


# ==================================================
# Construct the spatially uniform temporal slope
# ==================================================
temporal_slope_values = np.linspace(
    TEMPORAL_SLOPE_START,
    TEMPORAL_SLOPE_END,
    nt,
    dtype=np.float32,
)

temporal_slope_cube = np.broadcast_to(
    temporal_slope_values[:, None, None],
    (nt, NY, NX),
)


def make_rounded_polyline(
    times,
    x_values,
    y_values,
    corner_fraction=0.15,
    points_per_straight=150,
    points_per_corner=60,
):
    """
    Connect anchor points using mostly straight segments with rounded corners.

    The curve is constructed in (time, x, y) space so the resulting
    time-varying x and y components follow the same rounded-path geometry.
    """
    times = np.asarray(times, dtype=float)
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)

    if len(times) < 2:
        return times.copy(), x_values.copy(), y_values.copy()

    corner_fraction = np.clip(corner_fraction, 0.0, 0.49)

    points = np.column_stack([times, x_values, y_values])
    curve_parts = []
    current_point = points[0]

    for index in range(1, len(points) - 1):
        previous_anchor = points[index - 1]
        corner_anchor = points[index]
        next_anchor = points[index + 1]

        corner_start = corner_anchor - corner_fraction * (
            corner_anchor - previous_anchor
        )
        corner_end = corner_anchor + corner_fraction * (
            next_anchor - corner_anchor
        )

        straight_parameter = np.linspace(
            0.0,
            1.0,
            points_per_straight,
            endpoint=False,
        )
        straight_section = (
            current_point[None, :]
            + straight_parameter[:, None]
            * (corner_start - current_point)[None, :]
        )
        curve_parts.append(straight_section)

        corner_parameter = np.linspace(
            0.0,
            1.0,
            points_per_corner,
            endpoint=False,
        )
        one_minus = 1.0 - corner_parameter

        rounded_corner = (
            one_minus[:, None] ** 2 * corner_start[None, :]
            + 2.0
            * one_minus[:, None]
            * corner_parameter[:, None]
            * corner_anchor[None, :]
            + corner_parameter[:, None] ** 2
            * corner_end[None, :]
        )
        curve_parts.append(rounded_corner)

        current_point = corner_end

    final_parameter = np.linspace(
        0.0,
        1.0,
        points_per_straight,
    )
    final_section = (
        current_point[None, :]
        + final_parameter[:, None]
        * (points[-1] - current_point)[None, :]
    )
    curve_parts.append(final_section)

    full_curve = np.vstack(curve_parts)

    return full_curve[:, 0], full_curve[:, 1], full_curve[:, 2]


# ==================================================
# Construct the time-varying directional bias
# ==================================================
# Build a rounded polyline in (time, x, y) space so the synthetic bias
# follows the same anchor points and similar curvature as the rounded-path
# vector plot.
(
    bias_curve_time_min,
    bias_curve_x_per_100km,
    bias_curve_y_per_100km,
) = make_rounded_polyline(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_X_PER_100KM,
    BIAS_ANCHOR_Y_PER_100KM,
    corner_fraction=BIAS_CORNER_FRACTION,
    points_per_straight=BIAS_POINTS_PER_STRAIGHT,
    points_per_corner=BIAS_POINTS_PER_CORNER,
)

# Hold the first and last anchor values constant outside the anchor interval.
bias_evaluation_times_min = np.clip(
    times_min,
    BIAS_ANCHOR_TIMES_MIN[0],
    BIAS_ANCHOR_TIMES_MIN[-1],
)

bias_x_per_100km = np.interp(
    bias_evaluation_times_min,
    bias_curve_time_min,
    bias_curve_x_per_100km,
)
bias_y_per_100km = np.interp(
    bias_evaluation_times_min,
    bias_curve_time_min,
    bias_curve_y_per_100km,
)

bias_gradients_per_100km = np.hypot(
    bias_x_per_100km,
    bias_y_per_100km,
)

bias_angles_deg = np.degrees(
    np.arctan2(
        bias_y_per_100km,
        bias_x_per_100km,
    )
)
bias_angles_deg = (
    bias_angles_deg + 180.0
) % 360.0 - 180.0

bias_cube = np.empty((nt, NY, NX), dtype=np.float32)

for it, (gradient_x, gradient_y) in enumerate(
    zip(
        bias_x_per_100km,
        bias_y_per_100km,
    )
):
    # A planar field whose gradient components are exactly
    # gradient_x and gradient_y per 100 km.
    bias_cube[it] = (
        gradient_x * X_CENTERED_KM / 100.0
        + gradient_y * Y_CENTERED_KM / 100.0
    ).astype(np.float32)


# ==================================================
# Construct reproducible Gaussian noise
# ==================================================
rng = np.random.default_rng(NOISE_RANDOM_SEED)

noise_cube = rng.normal(
    loc=NOISE_MEAN,
    scale=NOISE_STD,
    size=(nt, NY, NX),
).astype(np.float32)


# ==================================================
# Build the six data variants
# ==================================================
variant_cubes = {
    "01_waves_only": wave_cube,
    "02_bias_only": bias_cube,
    "03_waves_plus_slope": (
        wave_cube + temporal_slope_cube
    ),
    "04_waves_plus_bias": (
        wave_cube + bias_cube
    ),
    "05_waves_plus_slope_plus_bias": (
        wave_cube
        + temporal_slope_cube
        + bias_cube
    ),
    "06_waves_plus_slope_plus_bias_plus_noise": (
        wave_cube
        + temporal_slope_cube
        + bias_cube
        + noise_cube
    ),
}

variant_labels = dict(VARIANT_SPECS)

print("Synthetic data shape:", wave_cube.shape)
print("Six variants created:")
for folder_name, label in VARIANT_SPECS:
    print(f"  {folder_name}: {label}")


# ==================================================
# Helper functions
# ==================================================
def symmetric_limits(data, percentile=99.0):
    finite_values = np.asarray(data)[np.isfinite(data)]

    if finite_values.size == 0:
        return -1.0, 1.0

    vmax = np.nanpercentile(
        np.abs(finite_values),
        percentile,
    )

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    return -vmax, vmax


def padded_limits(values, padding_fraction=0.05):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return -1.0, 1.0

    ymin = np.min(values)
    ymax = np.max(values)
    span = ymax - ymin

    if span == 0:
        span = max(abs(ymax), 1.0)

    padding = padding_fraction * span
    return ymin - padding, ymax + padding


def fit_plane_to_cube(data_cube, x_centered_km, y_centered_km):
    """Fit z = intercept + slope_x*x + slope_y*y to every frame."""
    nt_local = data_cube.shape[0]
    x_flat = np.asarray(x_centered_km, dtype=np.float64).ravel()
    y_flat = np.asarray(y_centered_km, dtype=np.float64).ravel()
    design = np.column_stack([np.ones(x_flat.size), x_flat, y_flat])

    intercept = np.full(nt_local, np.nan)
    slope_x_per_km = np.full(nt_local, np.nan)
    slope_y_per_km = np.full(nt_local, np.nan)

    for it in range(nt_local):
        z_flat = np.asarray(data_cube[it], dtype=np.float64).ravel()
        valid = np.isfinite(x_flat) & np.isfinite(y_flat) & np.isfinite(z_flat)
        if np.count_nonzero(valid) < 3:
            continue
        coefficients, _, _, _ = np.linalg.lstsq(
            design[valid], z_flat[valid], rcond=None
        )
        intercept[it], slope_x_per_km[it], slope_y_per_km[it] = coefficients

    slope_per_100km = 100.0 * np.hypot(slope_x_per_km, slope_y_per_km)
    angle_deg = np.rad2deg(np.arctan2(slope_y_per_km, slope_x_per_km))
    angle_deg = (angle_deg + 180.0) % 360.0 - 180.0

    return {
        "intercept": intercept,
        "slope_x_per_km": slope_x_per_km,
        "slope_y_per_km": slope_y_per_km,
        "slope_x_per_100km": 100.0 * slope_x_per_km,
        "slope_y_per_100km": 100.0 * slope_y_per_km,
        "slope_per_100km": slope_per_100km,
        "angle_deg": angle_deg,
    }


def save_plane_fit_results(plane_fit_results, variant_folder_name):
    """Save plane-fit arrays and angle/slope versus frame plots."""
    variant_label = variant_labels[variant_folder_name]
    variant_output_dir = PLANE_FIT_OUTPUT_DIR / variant_folder_name
    variant_output_dir.mkdir(parents=True, exist_ok=True)
    frame_numbers = np.arange(nt)

    output_array = np.column_stack([
        frame_numbers,
        times_min,
        plane_fit_results["angle_deg"],
        plane_fit_results["slope_per_100km"],
        plane_fit_results["slope_x_per_km"],
        plane_fit_results["slope_y_per_km"],
        plane_fit_results["intercept"],
    ])

    np.save(variant_output_dir / "plane_fit_vs_frame.npy", output_array)
    np.savetxt(
        variant_output_dir / "plane_fit_vs_frame.csv",
        output_array,
        delimiter=",",
        header=(
            "frame,time_min,angle_deg_increasing,slope_per_100km,"
            "slope_x_per_km,slope_y_per_km,intercept"
        ),
        comments="",
    )

    fig, (ax_angle, ax_slope) = plt.subplots(
        2, 1, figsize=(10, 8), sharex=True, constrained_layout=True
    )
    ax_angle.plot(frame_numbers, plane_fit_results["angle_deg"], linewidth=1.5)
    ax_angle.set_ylabel("Plane angle (deg)")
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks([-180, -120, -60, 0, 60, 120, 180])
    ax_angle.grid(True, alpha=0.3)
    ax_angle.set_title(f"{variant_label}: plane fit versus frame")

    ax_slope.plot(frame_numbers, plane_fit_results["slope_per_100km"], linewidth=1.5)
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Plane slope per 100 km")
    ax_slope.grid(True, alpha=0.3)

    fig.savefig(
        variant_output_dir / "plane_angle_and_slope_vs_frame.png",
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_slope_vector_plot(
    plane_fit_results,
    variant_folder_name,
):
    """
    Plot the plane-fit slope vector endpoint in component space.

    Coordinates:
        x = slope_x in amplitude per 100 km
        y = slope_y in amplitude per 100 km

    Color represents time. The y-axis is inverted so positive y points
    downward, consistent with the image-coordinate angle convention.
    """
    variant_label = variant_labels[variant_folder_name]
    variant_output_dir = PLANE_FIT_OUTPUT_DIR / variant_folder_name
    variant_output_dir.mkdir(parents=True, exist_ok=True)

    slope_x = plane_fit_results["slope_x_per_100km"]
    slope_y = plane_fit_results["slope_y_per_100km"]

    valid = (
        np.isfinite(slope_x)
        & np.isfinite(slope_y)
        & np.isfinite(times_min)
    )

    fig, ax = plt.subplots(figsize=(8, 8))

    if np.any(valid):
        x_valid = slope_x[valid]
        y_valid = slope_y[valid]
        time_valid = times_min[valid]

        # Connect consecutive vector endpoints to show time evolution.
        ax.plot(
            x_valid,
            y_valid,
            linewidth=0.9,
            alpha=0.35,
        )

        scatter = ax.scatter(
            x_valid,
            y_valid,
            c=time_valid,
            cmap=SLOPE_VECTOR_CMAP,
            s=28,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=0.02,
        )
        cbar.set_label("Time (min)")

        selected_indices = np.flatnonzero(valid)[
            ::SLOPE_VECTOR_ARROW_STEP
        ]

        if selected_indices.size > 0:
            ax.quiver(
                np.zeros(selected_indices.size),
                np.zeros(selected_indices.size),
                slope_x[selected_indices],
                slope_y[selected_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        # Overlay the prescribed rounded bias path and anchors for variants
        # that contain bias.
        if "bias" in variant_folder_name:
            ax.plot(
                bias_curve_x_per_100km,
                bias_curve_y_per_100km,
                color="black",
                linewidth=2.0,
                alpha=0.9,
                label="Prescribed rounded bias path",
                zorder=4,
            )

            ax.scatter(
                BIAS_ANCHOR_X_PER_100KM,
                BIAS_ANCHOR_Y_PER_100KM,
                marker="x",
                s=80,
                linewidths=2.0,
                c="black",
                label="Bias anchors",
                zorder=5,
            )

            for anchor_number, (
                anchor_x,
                anchor_y,
                anchor_time,
            ) in enumerate(
                zip(
                    BIAS_ANCHOR_X_PER_100KM,
                    BIAS_ANCHOR_Y_PER_100KM,
                    BIAS_ANCHOR_TIMES_MIN,
                ),
                start=1,
            ):
                ax.annotate(
                    f"{anchor_number}: {anchor_time:.0f} min",
                    xy=(anchor_x, anchor_y),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=8,
                )

            ax.legend(loc="best")

        component_limit = np.nanmax(
            np.abs(
                np.concatenate(
                    [
                        x_valid,
                        y_valid,
                        bias_curve_x_per_100km
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        bias_curve_y_per_100km
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        BIAS_ANCHOR_X_PER_100KM
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        BIAS_ANCHOR_Y_PER_100KM
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                    ]
                )
            )
        )
    else:
        component_limit = 1.0

    if not np.isfinite(component_limit) or component_limit <= 0:
        component_limit = 1.0

    component_limit *= 1.10

    ax.axhline(
        0.0,
        color="0.5",
        linewidth=0.8,
        alpha=0.6,
    )
    ax.axvline(
        0.0,
        color="0.5",
        linewidth=0.8,
        alpha=0.6,
    )
    ax.scatter(
        [0.0],
        [0.0],
        c="black",
        s=25,
        zorder=6,
    )

    ax.set_xlim(-component_limit, component_limit)
    ax.set_ylim(component_limit, -component_limit)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{variant_label}: Plane-fit Slope Vector Colored by Time"
    )
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        variant_output_dir
        / "plane_slope_vector_colored_by_time.png",
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def make_directional_keogram(
    data_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the image center.

    angle_deg:
        0 degrees  = E-W slice
        90 degrees = N-S slice
    """
    nt_local, ny_local, nx_local = data_cube.shape

    theta_local = np.deg2rad(angle_deg)

    # Unit vector along the cut
    ux = np.cos(theta_local)
    uy = np.sin(theta_local)

    # Perpendicular unit vector used for strip averaging
    px = -np.sin(theta_local)
    py = np.cos(theta_local)

    cx = (nx_local - 1) / 2.0
    cy = (ny_local - 1) / 2.0

    half_length_pix = min(nx_local, ny_local) // 2
    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels)
        - width_pixels // 2
    )

    keogram = np.full(
        (nt_local, len(distances_pix)),
        np.nan,
        dtype=np.float32,
    )

    # Sampling coordinates do not change with time, so calculate
    # them once for each perpendicular offset.
    sampling_coordinates = []

    for off in offsets:
        x_sample = cx + distances_pix * ux + off * px
        y_sample = cy + distances_pix * uy + off * py

        valid = (
            (x_sample >= 0)
            & (x_sample <= nx_local - 1)
            & (y_sample >= 0)
            & (y_sample <= ny_local - 1)
        )

        sampling_coordinates.append(
            (x_sample, y_sample, valid)
        )

    for it in range(nt_local):
        sampled_slices = []

        for x_sample, y_sample, valid in sampling_coordinates:
            values = np.full(
                len(distances_pix),
                np.nan,
                dtype=np.float32,
            )

            values[valid] = map_coordinates(
                data_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(
            sampled_slices,
            axis=0,
        )

    # DX_KM = DY_KM in this synthetic grid.
    distance_km = distances_pix * DX_KM

    return keogram, distance_km


def extract_profiles(keogram, distance_km):
    """Return profiles at all requested distances."""
    profiles = {}

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )
        actual_distance = float(distance_km[distance_index])
        profiles[actual_distance] = keogram[:, distance_index]

    return profiles


def save_keogram_figure(
    keogram,
    distance_km,
    angle_deg,
    variant_folder_name,
):
    """Save one directional keogram image."""
    variant_dir = VARIANT_DIRS[variant_folder_name]
    variant_label = variant_labels[variant_folder_name]

    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]

    keogram_vmin, keogram_vmax = symmetric_limits(
        visible_keogram,
        percentile=KEOGRAM_COLOR_PERCENTILE,
    )

    fig, ax = plt.subplots(figsize=(9, 5))

    image = ax.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=keogram_vmin,
        vmax=keogram_vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Perturbation amplitude")

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )
        actual_distance = distance_km[distance_index]

        ax.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    ax.set_title(
        f"{variant_label} | Directional keogram: "
        f"{angle_deg:.0f}°"
    )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Distance along slice (km)")
    ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    plt.tight_layout()

    output_path = (
        variant_dir
        / "keogram_images"
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.png"
    )

    fig.savefig(
        output_path,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_profile_array(
    profiles,
    angle_deg,
    variant_folder_name,
):
    """Save time and the selected profiles in one NumPy array."""
    ordered_profiles = []

    for requested_distance in PROFILE_DISTANCES_KM:
        actual_distance = min(
            profiles.keys(),
            key=lambda value: abs(value - requested_distance),
        )
        ordered_profiles.append(profiles[actual_distance])

    profile_array = np.column_stack(
        [times_min] + ordered_profiles
    )

    output_path = (
        VARIANT_DIRS[variant_folder_name]
        / "profile_arrays"
        / (
            "synthetic_keogram_profiles_"
            f"angle_{angle_deg:03.0f}.npy"
        )
    )

    np.save(output_path, profile_array)


# ==================================================
# Fit one plane to every frame of every data variant
# ==================================================
plane_fit_results_by_variant = {}
print("\nCalculating plane fits for all variants...")

for variant_folder_name, variant_label in VARIANT_SPECS:
    print(f"  Plane fitting: {variant_label}")
    results = fit_plane_to_cube(
        variant_cubes[variant_folder_name],
        X_CENTERED_KM,
        Y_CENTERED_KM,
    )
    plane_fit_results_by_variant[variant_folder_name] = results
    save_plane_fit_results(results, variant_folder_name)
    save_slope_vector_plot(results, variant_folder_name)

# Comparison figure containing all six variants.
frame_numbers = np.arange(nt)
fig_plane_compare, (ax_angle_compare, ax_slope_compare) = plt.subplots(
    2, 1, figsize=(11, 9), sharex=True, constrained_layout=True
)

for variant_folder_name, variant_label in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    ax_angle_compare.plot(
        frame_numbers, results["angle_deg"], linewidth=1.2, label=variant_label
    )
    ax_slope_compare.plot(
        frame_numbers, results["slope_per_100km"], linewidth=1.2, label=variant_label
    )

ax_angle_compare.set_title("Plane-fit direction and slope for all variants")
ax_angle_compare.set_ylabel("Plane angle (deg)")
ax_angle_compare.set_ylim(-180, 180)
ax_angle_compare.set_yticks([-180, -120, -60, 0, 60, 120, 180])
ax_angle_compare.grid(True, alpha=0.3)
ax_angle_compare.legend(loc="best", fontsize=8, ncol=2)

ax_slope_compare.set_xlabel("Frame")
ax_slope_compare.set_ylabel("Plane slope per 100 km")
ax_slope_compare.grid(True, alpha=0.3)
ax_slope_compare.legend(loc="best", fontsize=8, ncol=2)

fig_plane_compare.savefig(
    PLANE_FIT_OUTPUT_DIR / "plane_angle_and_slope_vs_frame_all_variants.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_plane_compare)


# ==================================================
# Slope-vector trajectories for all variants
# ==================================================
fig_vector_compare, axes_vector = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(15, 10),
    constrained_layout=True,
)

axes_vector = axes_vector.ravel()

# Use one common symmetric component limit for all variants.
all_vector_components = []

for variant_folder_name, _ in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    all_vector_components.extend(
        [
            results["slope_x_per_100km"],
            results["slope_y_per_100km"],
        ]
    )

all_vector_components.extend(
    [
        bias_curve_x_per_100km,
        bias_curve_y_per_100km,
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ]
)

common_vector_limit = np.nanmax(
    np.abs(np.concatenate(all_vector_components))
)

if not np.isfinite(common_vector_limit) or common_vector_limit <= 0:
    common_vector_limit = 1.0

common_vector_limit *= 1.10

shared_scatter = None

for ax, (variant_folder_name, variant_label) in zip(
    axes_vector,
    VARIANT_SPECS,
):
    results = plane_fit_results_by_variant[variant_folder_name]
    slope_x = results["slope_x_per_100km"]
    slope_y = results["slope_y_per_100km"]

    valid = (
        np.isfinite(slope_x)
        & np.isfinite(slope_y)
        & np.isfinite(times_min)
    )

    if np.any(valid):
        ax.plot(
            slope_x[valid],
            slope_y[valid],
            linewidth=0.8,
            alpha=0.30,
        )

        shared_scatter = ax.scatter(
            slope_x[valid],
            slope_y[valid],
            c=times_min[valid],
            cmap=SLOPE_VECTOR_CMAP,
            vmin=times_min.min(),
            vmax=times_min.max(),
            s=18,
            zorder=3,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.7, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.7, alpha=0.6)
    ax.set_xlim(-common_vector_limit, common_vector_limit)
    ax.set_ylim(common_vector_limit, -common_vector_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(variant_label)
    ax.grid(True, alpha=0.3)

    if "bias" in variant_folder_name:
        ax.plot(
            bias_curve_x_per_100km,
            bias_curve_y_per_100km,
            color="black",
            linewidth=1.8,
            alpha=0.9,
            zorder=4,
        )
        ax.scatter(
            BIAS_ANCHOR_X_PER_100KM,
            BIAS_ANCHOR_Y_PER_100KM,
            marker="x",
            s=55,
            linewidths=1.7,
            c="black",
            zorder=5,
        )

for ax in axes_vector:
    ax.set_xlabel("X slope (amp/100 km)")
    ax.set_ylabel("Y slope (amp/100 km)")

fig_vector_compare.suptitle(
    "Plane-fit Slope Vector Trajectories Colored by Time",
    fontsize=15,
)

if shared_scatter is not None:
    cbar = fig_vector_compare.colorbar(
        shared_scatter,
        ax=axes_vector.tolist(),
        shrink=0.85,
        pad=0.02,
    )
    cbar.set_label("Time (min)")

fig_vector_compare.savefig(
    PLANE_FIT_OUTPUT_DIR
    / "plane_slope_vector_colored_by_time_all_variants.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_vector_compare)


# ==================================================
# Generate all keograms and six-subplot profile figures
# ==================================================
for angle_deg in KEOGRAM_ANGLES_DEG:
    print(f"Processing angle {angle_deg:.0f}°")

    angle_profiles = {}

    # Create and save one keogram for every variant.
    for variant_folder_name, _ in VARIANT_SPECS:
        data_cube = variant_cubes[variant_folder_name]

        keogram, distance_km = make_directional_keogram(
            data_cube,
            angle_deg,
            width_pixels=KEOGRAM_WIDTH_PIXELS,
        )

        angle_profiles[variant_folder_name] = extract_profiles(
            keogram,
            distance_km,
        )

        np.save(
            VARIANT_DIRS[variant_folder_name]
            / "keogram_arrays"
            / f"synthetic_keogram_angle_{angle_deg:03.0f}.npy",
            keogram,
        )

        save_keogram_figure(
            keogram=keogram,
            distance_km=distance_km,
            angle_deg=angle_deg,
            variant_folder_name=variant_folder_name,
        )

        save_profile_array(
            profiles=angle_profiles[variant_folder_name],
            angle_deg=angle_deg,
            variant_folder_name=variant_folder_name,
        )

    # --------------------------------------------------
    # Six stacked profile subplots for this angle
    # --------------------------------------------------
    fig_profile, axes = plt.subplots(
        nrows=6,
        ncols=1,
        figsize=(11, 20),
        sharex=True,
        constrained_layout=True,
    )

    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )

    # Calculate a common y range from all six variants when requested.
    shared_profile_values = []

    if PROFILE_USE_SHARED_Y_LIMITS:
        for variant_folder_name, _ in VARIANT_SPECS:
            for profile in angle_profiles[variant_folder_name].values():
                valid = visible_time_mask & np.isfinite(profile)
                if np.any(valid):
                    shared_profile_values.append(profile[valid])

        if shared_profile_values:
            shared_ymin, shared_ymax = padded_limits(
                np.concatenate(shared_profile_values),
                PROFILE_Y_PADDING_FRACTION,
            )
        else:
            shared_ymin, shared_ymax = -1.0, 1.0

    for ax, (variant_folder_name, variant_label) in zip(
        axes,
        VARIANT_SPECS,
    ):
        profiles = angle_profiles[variant_folder_name]
        subplot_values = []

        for requested_distance in PROFILE_DISTANCES_KM:
            actual_distance = min(
                profiles.keys(),
                key=lambda value: abs(
                    value - requested_distance
                ),
            )
            profile = profiles[actual_distance]

            ax.plot(
                times_min,
                profile,
                linewidth=1.3,
                label=f"{actual_distance:+.1f} km",
            )

            valid = visible_time_mask & np.isfinite(profile)
            if np.any(valid):
                subplot_values.append(profile[valid])

        ax.axhline(
            0.0,
            linewidth=0.8,
            linestyle="--",
            alpha=0.7,
        )

        ax.set_title(variant_label)
        ax.set_ylabel("Amplitude")
        ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", ncol=3)

        if PROFILE_USE_SHARED_Y_LIMITS:
            ax.set_ylim(shared_ymin, shared_ymax)
        elif subplot_values:
            subplot_ymin, subplot_ymax = padded_limits(
                np.concatenate(subplot_values),
                PROFILE_Y_PADDING_FRACTION,
            )
            ax.set_ylim(subplot_ymin, subplot_ymax)

    axes[-1].set_xlabel("Time (min)")

    fig_profile.suptitle(
        "Synthetic Keogram Profiles: Six Data Variants\n"
        f"Directional angle = {angle_deg:.0f}°",
        fontsize=15,
    )

    comparison_path = (
        COMPARISON_PROFILE_DIR
        / (
            "synthetic_keogram_profiles_six_variants_"
            f"angle_{angle_deg:03.0f}.png"
        )
    )

    fig_profile.savefig(
        comparison_path,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_profile)


# ==================================================
# Save common arrays and component arrays
# ==================================================
np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "times_min.npy", times_min)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)
np.save(OUT_DIR / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)
np.save(
    OUT_DIR / "profile_distances_km.npy",
    np.asarray(PROFILE_DISTANCES_KM),
)

component_dir = OUT_DIR / "00_components"
component_dir.mkdir(parents=True, exist_ok=True)

np.save(component_dir / "wave_cube.npy", wave_cube)
np.save(component_dir / "temporal_slope_cube.npy", temporal_slope_cube)
np.save(component_dir / "temporal_slope_values.npy", temporal_slope_values)
np.save(component_dir / "bias_cube.npy", bias_cube)
np.save(component_dir / "bias_angles_deg.npy", bias_angles_deg)
np.save(
    component_dir / "bias_gradients_per_100km.npy",
    bias_gradients_per_100km,
)
np.save(
    component_dir / "bias_x_per_100km.npy",
    bias_x_per_100km,
)
np.save(
    component_dir / "bias_y_per_100km.npy",
    bias_y_per_100km,
)
np.save(component_dir / "noise_cube.npy", noise_cube)

for variant_folder_name, _ in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    np.save(
        component_dir / f"{variant_folder_name}_plane_angle_deg.npy",
        results["angle_deg"],
    )
    np.save(
        component_dir / f"{variant_folder_name}_plane_slope_per_100km.npy",
        results["slope_per_100km"],
    )


# ==================================================
# Save bias interpolation diagnostic plot
# ==================================================
fig_bias, axes_bias = plt.subplots(
    3,
    1,
    figsize=(9, 10),
    sharex=True,
)

ax_components, ax_angle, ax_gradient = axes_bias

ax_components.plot(
    times_min,
    bias_x_per_100km,
    linewidth=2.0,
    label="X component",
)
ax_components.plot(
    times_min,
    bias_y_per_100km,
    linewidth=2.0,
    label="Y component",
)
ax_components.plot(
    bias_curve_time_min,
    bias_curve_x_per_100km,
    linewidth=1.2,
    linestyle="--",
    alpha=0.8,
    label="Rounded path X",
)
ax_components.plot(
    bias_curve_time_min,
    bias_curve_y_per_100km,
    linewidth=1.2,
    linestyle="--",
    alpha=0.8,
    label="Rounded path Y",
)
ax_components.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_X_PER_100KM,
    marker="o",
    zorder=3,
    label="X anchors",
)
ax_components.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_Y_PER_100KM,
    marker="s",
    zorder=3,
    label="Y anchors",
)
ax_components.axhline(
    0.0,
    linewidth=0.8,
    linestyle="--",
    alpha=0.7,
)
ax_components.set_ylabel("Gradient component\n(amp per 100 km)")
ax_components.set_title(
    "Directional bias from rounded-path physical gradient anchors"
)
ax_components.grid(True, alpha=0.3)
ax_components.legend(loc="best", ncol=2)

ax_angle.plot(
    times_min,
    bias_angles_deg,
    linewidth=2.0,
)
ax_angle.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    (
        np.degrees(
            np.arctan2(
                BIAS_ANCHOR_Y_PER_100KM,
                BIAS_ANCHOR_X_PER_100KM,
            )
        )
        + 180.0
    )
    % 360.0
    - 180.0,
    zorder=3,
)
ax_angle.set_ylabel("Direction (deg)")
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(
    [-180, -120, -60, 0, 60, 120, 180]
)
ax_angle.grid(True, alpha=0.3)

ax_gradient.plot(
    times_min,
    bias_gradients_per_100km,
    linewidth=2.0,
)
ax_gradient.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    np.hypot(
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ),
    zorder=3,
)
ax_gradient.set_xlabel("Time (min)")
ax_gradient.set_ylabel("Magnitude\n(amp per 100 km)")
ax_gradient.grid(True, alpha=0.3)

plt.tight_layout()
fig_bias.savefig(
    component_dir / "bias_vector_component_rounded_path.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_bias)


# ==================================================
# Save parameters
# ==================================================
with open(
    OUT_DIR / "synthetic_wave_parameters.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(f"Field size X (km): {FIELD_SIZE_X_KM}\n")
    file.write(f"Field size Y (km): {FIELD_SIZE_Y_KM}\n")
    file.write(f"Grid spacing X (km): {DX_KM}\n")
    file.write(f"Grid spacing Y (km): {DY_KM}\n")
    file.write(f"Time step (s): {DT_SECONDS}\n")
    file.write(f"Total time (min): {TOTAL_TIME_MIN}\n")
    file.write(f"Wavelength (km): {WAVELENGTH_KM}\n")
    file.write(f"Gaussian sigma (km): {GAUSSIAN_WIDTH_KM}\n")
    file.write(f"Maximum amplitude: {MAX_AMPLITUDE}\n")
    file.write(
        f"Propagation angle (deg): {PROPAGATION_ANGLE_DEG}\n"
    )
    file.write(f"Wave speed (m/s): {WAVE_SPEED_M_S}\n")
    file.write(f"Wave time shift (min): {WAVE_TIME_SHIFT_MIN}\n")
    file.write(
        f"Temporal slope start: {TEMPORAL_SLOPE_START}\n"
    )
    file.write(
        f"Temporal slope end: {TEMPORAL_SLOPE_END}\n"
    )
    file.write(f"Noise mean: {NOISE_MEAN}\n")
    file.write(f"Noise standard deviation: {NOISE_STD}\n")
    file.write(f"Noise random seed: {NOISE_RANDOM_SEED}\n")
    file.write(
        "Bias interpolation: natural cubic splines of x and y "
        "gradient components\n"
    )
    file.write(
        "Bias behavior before 140 min: first anchor held constant\n"
    )
    file.write("Bias is zero at the spatial field center\n")
    file.write("Plane-fit direction: increasing perturbation direction\n")
    file.write("Plane-fit angle range: [-180, 180) deg\n")
    file.write(
        "Bias angle derived from interpolated x and y components\n"
    )
    file.write("Plane-fit slope units: perturbation amplitude per 100 km\n")
    file.write(
        "Bias angle convention: 0 deg east, +90 deg south\n"
    )
    file.write(
        "Bias anchors "
        "(time_min, x_per_100km, y_per_100km, "
        "magnitude_per_100km, direction_deg):\n"
    )

    for time_value, x_value, y_value in zip(
        BIAS_ANCHOR_TIMES_MIN,
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ):
        magnitude_value = np.hypot(x_value, y_value)
        direction_value = (
            (
                np.degrees(
                    np.arctan2(y_value, x_value)
                )
                + 180.0
            )
            % 360.0
        ) - 180.0

        file.write(
            f"  {time_value}, {x_value}, {y_value}, "
            f"{magnitude_value}, {direction_value}\n"
        )


# ==================================================
# Print output summary
# ==================================================
print("\nFinished.")
print("Main output folder:", OUT_DIR)
print("\nVariant folders:")
for folder_name, label in VARIANT_SPECS:
    print(f"  {label}: {VARIANT_DIRS[folder_name]}")

print(
    "\nSix-subplot profile comparisons:",
    COMPARISON_PROFILE_DIR,
)
print(
    "Number of angles processed:",
    len(KEOGRAM_ANGLES_DEG),
)
print(
    "Profile distances:",
    PROFILE_DISTANCES_KM,
)
print(
    "Plane-fit outputs:",
    PLANE_FIT_OUTPUT_DIR,
)
print(
    "Slope-vector plots: one per variant plus an all-variant comparison"
)


Synthetic data shape: (251, 501, 501)
Six variants created:
  01_waves_only: Waves only
  02_bias_only: Bias only
  03_waves_plus_slope: Waves + slope
  04_waves_plus_bias: Waves + bias
  05_waves_plus_slope_plus_bias: Waves + slope + bias
  06_waves_plus_slope_plus_bias_plus_noise: Waves + slope + bias + noise

Calculating plane fits for all variants...
  Plane fitting: Waves only
  Plane fitting: Bias only
  Plane fitting: Waves + slope
  Plane fitting: Waves + bias
  Plane fitting: Waves + slope + bias
  Plane fitting: Waves + slope + bias + noise
Processing angle 0°
Processing angle 5°
Processing angle 10°
Processing angle 15°
Processing angle 20°
Processing angle 25°
Processing angle 30°
Processing angle 35°
Processing angle 40°
Processing angle 45°
Processing angle 50°
Processing angle 55°
Processing angle 60°
Processing angle 65°
Processing angle 70°
Processing angle 75°
Processing angle 80°
Processing angle 85°
Processing angle 90°
Processing angle 95°
Processing angle 100°
Pro

## Synthetic Data, 6 Variants, Varying Angles

In [68]:
# ==================================================
# Synthetic-wave directional keograms: six variants
#
# This script creates and saves:
#   1. Waves only
#   2. Bias only
#   3. Waves + temporal slope
#   4. Waves + directional bias
#   5. Waves + temporal slope + directional bias
#   6. Waves + temporal slope + directional bias + noise
#
# Each variant is saved in a separate folder. For every keogram
# angle, one comparison figure is also saved with six vertically
# stacked profile subplots. Each subplot contains the profiles at
# -250, 0, and +250 km.
# ==================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("synthetic_wave_keogram_outputs_six_variants")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Spatial grid
DX_KM = 1.0
DY_KM = 1.0
FIELD_SIZE_X_KM = 500.0
FIELD_SIZE_Y_KM = 500.0

# Temporal grid
DT_SECONDS = 120
TOTAL_TIME_MIN = 500

# Synthetic Gaussian wave packet
WAVELENGTH_KM = 4050.0
GAUSSIAN_WIDTH_KM = 7200.0  # Gaussian sigma
MAX_AMPLITUDE = 0.08
# Time-varying propagation angle.
# The angle is held at 60 deg before 140 min, changes linearly
# from 60 deg to 10 deg between 140 and 230 min, and is held
# at 10 deg afterward.
PROPAGATION_ANGLE_START_TIME_MIN = 154.0
PROPAGATION_ANGLE_END_TIME_MIN = 224.0
PROPAGATION_ANGLE_START_DEG = 70.0
PROPAGATION_ANGLE_END_DEG = 10.0

WAVE_SPEED_M_S = -450.0

# Shift only the wave state in time.
# Positive = advance; negative = delay.
WAVE_TIME_SHIFT_MIN = 90.0

# Spatially uniform temporal slope.
# The same value is added to every pixel in a frame.
TEMPORAL_SLOPE_START = 1.0
TEMPORAL_SLOPE_END = -0.4

# Time-varying directional bias anchor points derived from the
# rounded-path plane-fit vector plot.
#
# Component convention:
#   +x = right/east
#   +y = down/south
#
# Units below are perturbation amplitude per 100 km.
BIAS_PATH_START_TIME_MIN = 120.0
BIAS_PATH_END_TIME_MIN = 500.0

BIAS_ANCHOR_TIMES_MIN = np.array(
    [154.0, 224.0, 294.0, 364.0, 434.0, 500.0],
    dtype=np.float64,
)

BIAS_ANCHOR_X_PER_100KM = np.array(
    [0.003282, -0.014097, -0.012296, -0.014813, -0.012807, -0.003761],
    dtype=np.float64,
)

BIAS_ANCHOR_Y_PER_100KM = np.array(
    [-0.008459, 0.005455, 0.002510, 0.011933, 0.010430, 0.026472],
    dtype=np.float64,
)

# Use a mostly straight rounded polyline, similar to the rounded-path
# vector plot. Smaller corner_fraction makes the path straighter.
BIAS_CORNER_FRACTION = 0.10
BIAS_POINTS_PER_STRAIGHT = 150
BIAS_POINTS_PER_CORNER = 60

# Additive Gaussian noise
NOISE_MEAN = 0.0
NOISE_STD = 0.02
NOISE_RANDOM_SEED = 42

DPI = 200
CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(-90, 91, 5)

# Average a wider strip to reduce noise
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = TOTAL_TIME_MIN
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# Distance rows shown in each profile subplot
PROFILE_DISTANCES_KM = [-250, 0, 250]

# Automatic plot limits
KEOGRAM_COLOR_PERCENTILE = 99.0
PROFILE_Y_PADDING_FRACTION = 0.05

# When True, all six profile subplots use one common y range.
# This makes the amplitudes directly comparable between variants.
# Set False to scale each subplot independently.
PROFILE_USE_SHARED_Y_LIMITS = True

# Plane-fit analysis
# Each frame is fitted as z = intercept + slope_x*x + slope_y*y.
# The direction points toward increasing perturbation.
# Angle convention: 0 deg = +x/east, +90 deg = +y/south.
# Angles are wrapped to [-180, 180).
# Scalar slope units are perturbation amplitude per 100 km.
PLANE_FIT_OUTPUT_DIR = OUT_DIR / "08_plane_fit_analysis"
PLANE_FIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Slope-vector trajectory plots
# Plot every Nth vector arrow to reduce crowding.
SLOPE_VECTOR_ARROW_STEP = 10
SLOPE_VECTOR_CMAP = "viridis"
SLOPE_VECTOR_TIME_MIN = 120.0
SLOPE_VECTOR_TIME_MAX = 500.0


# ==================================================
# Output folders and labels
# ==================================================
# Processing mode:
#   "variant6" = process only Waves + slope + bias + noise
#   "all"      = process all six variants
PROCESSING_MODE = "all"

ALL_VARIANT_SPECS = [
    (
        "01_waves_only",
        "Waves only",
    ),
    (
        "02_bias_only",
        "Bias only",
    ),
    (
        "03_waves_plus_slope",
        "Waves + slope",
    ),
    (
        "04_waves_plus_bias",
        "Waves + bias",
    ),
    (
        "05_waves_plus_slope_plus_bias",
        "Waves + slope + bias",
    ),
    (
        "06_waves_plus_slope_plus_bias_plus_noise",
        "Waves + slope + bias + noise",
    ),
]

if PROCESSING_MODE == "variant6":
    VARIANT_SPECS = [ALL_VARIANT_SPECS[-1]]
elif PROCESSING_MODE == "all":
    VARIANT_SPECS = ALL_VARIANT_SPECS.copy()
else:
    raise ValueError(
        'PROCESSING_MODE must be either "variant6" or "all".'
    )

VARIANT_DIRS = {}
for folder_name, label in VARIANT_SPECS:
    variant_dir = OUT_DIR / folder_name
    (variant_dir / "keogram_images").mkdir(parents=True, exist_ok=True)
    (variant_dir / "keogram_arrays").mkdir(parents=True, exist_ok=True)
    (variant_dir / "profile_arrays").mkdir(parents=True, exist_ok=True)
    VARIANT_DIRS[folder_name] = variant_dir

COMPARISON_PROFILE_DIR = OUT_DIR / "07_profile_comparisons"
COMPARISON_PROFILE_DIR.mkdir(parents=True, exist_ok=True)


# ==================================================
# Coordinate and time grids
# ==================================================
NX = int(round(FIELD_SIZE_X_KM / DX_KM)) + 1
NY = int(round(FIELD_SIZE_Y_KM / DY_KM)) + 1

x_km = np.linspace(0.0, FIELD_SIZE_X_KM, NX)
y_km = np.linspace(0.0, FIELD_SIZE_Y_KM, NY)

times_s = np.arange(
    0,
    TOTAL_TIME_MIN * 60 + DT_SECONDS,
    DT_SECONDS,
)
times_min = times_s / 60.0
nt = len(times_s)

X_KM, Y_KM = np.meshgrid(x_km, y_km)

cx_km = 0.5 * (x_km.min() + x_km.max())
cy_km = 0.5 * (y_km.min() + y_km.max())

X_CENTERED_KM = X_KM - cx_km
Y_CENTERED_KM = Y_KM - cy_km


# ==================================================
# Generate the moving Gaussian wave packet
# ==================================================
def propagation_angle_deg_at_time(time_min):
    """Return the prescribed propagation angle at any time."""
    time_min = np.asarray(time_min, dtype=np.float64)

    interpolation_fraction = (
        time_min - PROPAGATION_ANGLE_START_TIME_MIN
    ) / (
        PROPAGATION_ANGLE_END_TIME_MIN
        - PROPAGATION_ANGLE_START_TIME_MIN
    )
    interpolation_fraction = np.clip(
        interpolation_fraction,
        0.0,
        1.0,
    )

    return (
        PROPAGATION_ANGLE_START_DEG
        + interpolation_fraction
        * (
            PROPAGATION_ANGLE_END_DEG
            - PROPAGATION_ANGLE_START_DEG
        )
    )


wavenumber_rad_per_km = 2.0 * np.pi / WAVELENGTH_KM
wave_speed_km_s = WAVE_SPEED_M_S / 1000.0

# The propagation-angle schedule is tied to displayed simulation time.
propagation_angles_deg = propagation_angle_deg_at_time(times_min)
propagation_angles_rad = np.deg2rad(propagation_angles_deg)

# --------------------------------------------------
# Integrate the curved Gaussian-packet trajectory
# --------------------------------------------------
# The packet center follows the time-varying propagation direction.
# Changing WAVE_SPEED_M_S changes the travel rate but does not shift
# the 140-230 min angle-transition window.
integration_times_s = times_s.astype(np.float64)

velocity_x_km_s = (
    wave_speed_km_s * np.cos(propagation_angles_rad)
)
velocity_y_km_s = (
    wave_speed_km_s * np.sin(propagation_angles_rad)
)

dt_integration_s = np.diff(integration_times_s)
packet_center_x_km = np.zeros(nt, dtype=np.float64)
packet_center_y_km = np.zeros(nt, dtype=np.float64)

packet_center_x_km[1:] = np.cumsum(
    0.5
    * (velocity_x_km_s[:-1] + velocity_x_km_s[1:])
    * dt_integration_s
)
packet_center_y_km[1:] = np.cumsum(
    0.5
    * (velocity_y_km_s[:-1] + velocity_y_km_s[1:])
    * dt_integration_s
)

# --------------------------------------------------
# Integrate the carrier-wave temporal phase separately
# --------------------------------------------------
# omega = k*c. The sign of WAVE_SPEED_M_S controls the phase direction.
# Positive WAVE_TIME_SHIFT_MIN advances the carrier phase; negative values
# delay it. The time shift does not move the angle-transition times.
angular_frequency_rad_s = (
    wavenumber_rad_per_km * wave_speed_km_s
)

temporal_phase_rad = (
    angular_frequency_rad_s * integration_times_s
)
phase_shift_rad = (
    angular_frequency_rad_s
    * WAVE_TIME_SHIFT_MIN
    * 60.0
)
temporal_phase_rad = temporal_phase_rad + phase_shift_rad

# --------------------------------------------------
# Create the wave cube
# --------------------------------------------------
wave_cube = np.empty((nt, NY, NX), dtype=np.float32)

for it in range(nt):
    theta = propagation_angles_rad[it]
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    # Gaussian envelope follows the integrated curved packet path.
    relative_x_km = X_CENTERED_KM - packet_center_x_km[it]
    relative_y_km = Y_CENTERED_KM - packet_center_y_km[it]

    parallel_relative_km = (
        relative_x_km * cos_theta
        + relative_y_km * sin_theta
    )

    gaussian_envelope = np.exp(
        -0.5
        * (parallel_relative_km / GAUSSIAN_WIDTH_KM) ** 2
    )

    # Wavefront orientation is prescribed by the instantaneous angle,
    # while the carrier phase advances only through temporal_phase_rad.
    spatial_phase_rad = wavenumber_rad_per_km * (
        X_CENTERED_KM * cos_theta
        + Y_CENTERED_KM * sin_theta
    )

    wave_phase_rad = spatial_phase_rad - temporal_phase_rad[it]

    wave_cube[it] = (
        MAX_AMPLITUDE
        * gaussian_envelope
        * np.cos(wave_phase_rad)
    ).astype(np.float32)


# ==================================================
# Construct the spatially uniform temporal slope
# ==================================================
temporal_slope_values = np.linspace(
    TEMPORAL_SLOPE_START,
    TEMPORAL_SLOPE_END,
    nt,
    dtype=np.float32,
)

temporal_slope_cube = np.broadcast_to(
    temporal_slope_values[:, None, None],
    (nt, NY, NX),
)


def make_rounded_polyline(
    times,
    x_values,
    y_values,
    corner_fraction=0.15,
    points_per_straight=150,
    points_per_corner=60,
):
    """
    Connect anchor points using mostly straight segments with rounded corners.

    The curve is constructed in (time, x, y) space so the resulting
    time-varying x and y components follow the same rounded-path geometry.
    """
    times = np.asarray(times, dtype=float)
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)

    if len(times) < 2:
        return times.copy(), x_values.copy(), y_values.copy()

    corner_fraction = np.clip(corner_fraction, 0.0, 0.49)

    points = np.column_stack([times, x_values, y_values])
    curve_parts = []
    current_point = points[0]

    for index in range(1, len(points) - 1):
        previous_anchor = points[index - 1]
        corner_anchor = points[index]
        next_anchor = points[index + 1]

        corner_start = corner_anchor - corner_fraction * (
            corner_anchor - previous_anchor
        )
        corner_end = corner_anchor + corner_fraction * (
            next_anchor - corner_anchor
        )

        straight_parameter = np.linspace(
            0.0,
            1.0,
            points_per_straight,
            endpoint=False,
        )
        straight_section = (
            current_point[None, :]
            + straight_parameter[:, None]
            * (corner_start - current_point)[None, :]
        )
        curve_parts.append(straight_section)

        corner_parameter = np.linspace(
            0.0,
            1.0,
            points_per_corner,
            endpoint=False,
        )
        one_minus = 1.0 - corner_parameter

        rounded_corner = (
            one_minus[:, None] ** 2 * corner_start[None, :]
            + 2.0
            * one_minus[:, None]
            * corner_parameter[:, None]
            * corner_anchor[None, :]
            + corner_parameter[:, None] ** 2
            * corner_end[None, :]
        )
        curve_parts.append(rounded_corner)

        current_point = corner_end

    final_parameter = np.linspace(
        0.0,
        1.0,
        points_per_straight,
    )
    final_section = (
        current_point[None, :]
        + final_parameter[:, None]
        * (points[-1] - current_point)[None, :]
    )
    curve_parts.append(final_section)

    full_curve = np.vstack(curve_parts)

    return full_curve[:, 0], full_curve[:, 1], full_curve[:, 2]


# ==================================================
# Construct the time-varying directional bias
# ==================================================
# Build a rounded polyline in (time, x, y) space so the synthetic bias
# follows the same anchor points and similar curvature as the rounded-path
# vector plot.
(
    bias_curve_time_min,
    bias_curve_x_per_100km,
    bias_curve_y_per_100km,
) = make_rounded_polyline(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_X_PER_100KM,
    BIAS_ANCHOR_Y_PER_100KM,
    corner_fraction=BIAS_CORNER_FRACTION,
    points_per_straight=BIAS_POINTS_PER_STRAIGHT,
    points_per_corner=BIAS_POINTS_PER_CORNER,
)

# Hold the first and last anchor values constant outside the anchor interval.
bias_evaluation_times_min = np.clip(
    times_min,
    BIAS_ANCHOR_TIMES_MIN[0],
    BIAS_ANCHOR_TIMES_MIN[-1],
)

bias_x_per_100km = np.interp(
    bias_evaluation_times_min,
    bias_curve_time_min,
    bias_curve_x_per_100km,
)
bias_y_per_100km = np.interp(
    bias_evaluation_times_min,
    bias_curve_time_min,
    bias_curve_y_per_100km,
)

bias_gradients_per_100km = np.hypot(
    bias_x_per_100km,
    bias_y_per_100km,
)

bias_angles_deg = np.degrees(
    np.arctan2(
        bias_y_per_100km,
        bias_x_per_100km,
    )
)
bias_angles_deg = (
    bias_angles_deg + 180.0
) % 360.0 - 180.0

bias_cube = np.empty((nt, NY, NX), dtype=np.float32)

for it, (gradient_x, gradient_y) in enumerate(
    zip(
        bias_x_per_100km,
        bias_y_per_100km,
    )
):
    # A planar field whose gradient components are exactly
    # gradient_x and gradient_y per 100 km.
    bias_cube[it] = (
        gradient_x * X_CENTERED_KM / 100.0
        + gradient_y * Y_CENTERED_KM / 100.0
    ).astype(np.float32)


# ==================================================
# Construct reproducible Gaussian noise
# ==================================================
rng = np.random.default_rng(NOISE_RANDOM_SEED)

noise_cube = rng.normal(
    loc=NOISE_MEAN,
    scale=NOISE_STD,
    size=(nt, NY, NX),
).astype(np.float32)


# ==================================================
# Build the six data variants
# ==================================================
variant_cubes = {
    "01_waves_only": wave_cube,
    "02_bias_only": bias_cube,
    "03_waves_plus_slope": (
        wave_cube + temporal_slope_cube
    ),
    "04_waves_plus_bias": (
        wave_cube + bias_cube
    ),
    "05_waves_plus_slope_plus_bias": (
        wave_cube
        + temporal_slope_cube
        + bias_cube
    ),
    "06_waves_plus_slope_plus_bias_plus_noise": (
        wave_cube
        + temporal_slope_cube
        + bias_cube
        + noise_cube
    ),
}

variant_labels = dict(VARIANT_SPECS)

print("Synthetic data shape:", wave_cube.shape)
print("Active variants:")
for folder_name, label in VARIANT_SPECS:
    print(f"  {folder_name}: {label}")


# ==================================================
# Helper functions
# ==================================================
def symmetric_limits(data, percentile=99.0):
    finite_values = np.asarray(data)[np.isfinite(data)]

    if finite_values.size == 0:
        return -1.0, 1.0

    vmax = np.nanpercentile(
        np.abs(finite_values),
        percentile,
    )

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    return -vmax, vmax


def padded_limits(values, padding_fraction=0.05):
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return -1.0, 1.0

    ymin = np.min(values)
    ymax = np.max(values)
    span = ymax - ymin

    if span == 0:
        span = max(abs(ymax), 1.0)

    padding = padding_fraction * span
    return ymin - padding, ymax + padding


def fit_plane_to_cube(data_cube, x_centered_km, y_centered_km):
    """Fit z = intercept + slope_x*x + slope_y*y to every frame."""
    nt_local = data_cube.shape[0]
    x_flat = np.asarray(x_centered_km, dtype=np.float64).ravel()
    y_flat = np.asarray(y_centered_km, dtype=np.float64).ravel()
    design = np.column_stack([np.ones(x_flat.size), x_flat, y_flat])

    intercept = np.full(nt_local, np.nan)
    slope_x_per_km = np.full(nt_local, np.nan)
    slope_y_per_km = np.full(nt_local, np.nan)

    for it in range(nt_local):
        z_flat = np.asarray(data_cube[it], dtype=np.float64).ravel()
        valid = np.isfinite(x_flat) & np.isfinite(y_flat) & np.isfinite(z_flat)
        if np.count_nonzero(valid) < 3:
            continue
        coefficients, _, _, _ = np.linalg.lstsq(
            design[valid], z_flat[valid], rcond=None
        )
        intercept[it], slope_x_per_km[it], slope_y_per_km[it] = coefficients

    slope_per_100km = 100.0 * np.hypot(slope_x_per_km, slope_y_per_km)
    angle_deg = np.rad2deg(np.arctan2(slope_y_per_km, slope_x_per_km))
    angle_deg = (angle_deg + 180.0) % 360.0 - 180.0

    return {
        "intercept": intercept,
        "slope_x_per_km": slope_x_per_km,
        "slope_y_per_km": slope_y_per_km,
        "slope_x_per_100km": 100.0 * slope_x_per_km,
        "slope_y_per_100km": 100.0 * slope_y_per_km,
        "slope_per_100km": slope_per_100km,
        "angle_deg": angle_deg,
    }


def save_plane_fit_results(plane_fit_results, variant_folder_name):
    """Save plane-fit arrays and angle/slope versus frame plots."""
    variant_label = variant_labels[variant_folder_name]
    variant_output_dir = PLANE_FIT_OUTPUT_DIR / variant_folder_name
    variant_output_dir.mkdir(parents=True, exist_ok=True)
    frame_numbers = np.arange(nt)

    output_array = np.column_stack([
        frame_numbers,
        times_min,
        plane_fit_results["angle_deg"],
        plane_fit_results["slope_per_100km"],
        plane_fit_results["slope_x_per_km"],
        plane_fit_results["slope_y_per_km"],
        plane_fit_results["intercept"],
    ])

    np.save(variant_output_dir / "plane_fit_vs_frame.npy", output_array)
    np.savetxt(
        variant_output_dir / "plane_fit_vs_frame.csv",
        output_array,
        delimiter=",",
        header=(
            "frame,time_min,angle_deg_increasing,slope_per_100km,"
            "slope_x_per_km,slope_y_per_km,intercept"
        ),
        comments="",
    )

    fig, (ax_angle, ax_slope) = plt.subplots(
        2, 1, figsize=(10, 8), sharex=True, constrained_layout=True
    )
    ax_angle.plot(frame_numbers, plane_fit_results["angle_deg"], linewidth=1.5)
    ax_angle.set_ylabel("Plane angle (deg)")
    ax_angle.set_ylim(-180, 180)
    ax_angle.set_yticks([-180, -120, -60, 0, 60, 120, 180])
    ax_angle.grid(True, alpha=0.3)
    ax_angle.set_title(f"{variant_label}: plane fit versus frame")

    ax_slope.plot(frame_numbers, plane_fit_results["slope_per_100km"], linewidth=1.5)
    ax_slope.set_xlabel("Frame")
    ax_slope.set_ylabel("Plane slope per 100 km")
    ax_slope.grid(True, alpha=0.3)

    fig.savefig(
        variant_output_dir / "plane_angle_and_slope_vs_frame.png",
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_slope_vector_plot(
    plane_fit_results,
    variant_folder_name,
):
    """
    Plot the plane-fit slope vector endpoint in component space.

    Coordinates:
        x = slope_x in amplitude per 100 km
        y = slope_y in amplitude per 100 km

    Color represents time. The y-axis is inverted so positive y points
    downward, consistent with the image-coordinate angle convention.
    """
    variant_label = variant_labels[variant_folder_name]
    variant_output_dir = PLANE_FIT_OUTPUT_DIR / variant_folder_name
    variant_output_dir.mkdir(parents=True, exist_ok=True)

    slope_x = plane_fit_results["slope_x_per_100km"]
    slope_y = plane_fit_results["slope_y_per_100km"]

    valid = (
        np.isfinite(slope_x)
        & np.isfinite(slope_y)
        & np.isfinite(times_min)
        & (times_min >= SLOPE_VECTOR_TIME_MIN)
        & (times_min <= SLOPE_VECTOR_TIME_MAX)
    )

    fig, ax = plt.subplots(figsize=(8, 8))

    if np.any(valid):
        x_valid = slope_x[valid]
        y_valid = slope_y[valid]
        time_valid = times_min[valid]

        # Connect consecutive vector endpoints to show time evolution.
        ax.plot(
            x_valid,
            y_valid,
            linewidth=0.9,
            alpha=0.35,
        )

        scatter = ax.scatter(
            x_valid,
            y_valid,
            c=time_valid,
            cmap=SLOPE_VECTOR_CMAP,
            s=28,
            zorder=3,
        )

        cbar = fig.colorbar(
            scatter,
            ax=ax,
            pad=0.02,
        )
        cbar.set_label("Time (min)")

        selected_indices = np.flatnonzero(valid)[
            ::SLOPE_VECTOR_ARROW_STEP
        ]

        if selected_indices.size > 0:
            ax.quiver(
                np.zeros(selected_indices.size),
                np.zeros(selected_indices.size),
                slope_x[selected_indices],
                slope_y[selected_indices],
                angles="xy",
                scale_units="xy",
                scale=1.0,
                color="0.35",
                width=0.003,
                alpha=0.45,
                zorder=2,
            )

        # Overlay the prescribed rounded bias path and anchors for variants
        # that contain bias.
        if "bias" in variant_folder_name:
            ax.plot(
                bias_curve_x_per_100km,
                bias_curve_y_per_100km,
                color="black",
                linewidth=2.0,
                alpha=0.9,
                label="Prescribed rounded bias path",
                zorder=4,
            )

            ax.scatter(
                BIAS_ANCHOR_X_PER_100KM,
                BIAS_ANCHOR_Y_PER_100KM,
                marker="x",
                s=80,
                linewidths=2.0,
                c="black",
                label="Bias anchors",
                zorder=5,
            )

            for anchor_number, (
                anchor_x,
                anchor_y,
                anchor_time,
            ) in enumerate(
                zip(
                    BIAS_ANCHOR_X_PER_100KM,
                    BIAS_ANCHOR_Y_PER_100KM,
                    BIAS_ANCHOR_TIMES_MIN,
                ),
                start=1,
            ):
                ax.annotate(
                    f"{anchor_number}: {anchor_time:.0f} min",
                    xy=(anchor_x, anchor_y),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=8,
                )

            ax.legend(loc="best")

        component_limit = np.nanmax(
            np.abs(
                np.concatenate(
                    [
                        x_valid,
                        y_valid,
                        bias_curve_x_per_100km
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        bias_curve_y_per_100km
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        BIAS_ANCHOR_X_PER_100KM
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                        BIAS_ANCHOR_Y_PER_100KM
                        if "bias" in variant_folder_name
                        else np.array([0.0]),
                    ]
                )
            )
        )
    else:
        component_limit = 1.0

    if not np.isfinite(component_limit) or component_limit <= 0:
        component_limit = 1.0

    component_limit *= 1.10

    ax.axhline(
        0.0,
        color="0.5",
        linewidth=0.8,
        alpha=0.6,
    )
    ax.axvline(
        0.0,
        color="0.5",
        linewidth=0.8,
        alpha=0.6,
    )
    ax.scatter(
        [0.0],
        [0.0],
        c="black",
        s=25,
        zorder=6,
    )

    ax.set_xlim(-component_limit, component_limit)
    ax.set_ylim(component_limit, -component_limit)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(
        f"{variant_label}: Plane-fit Slope Vector Colored by Time"
    )
    ax.set_xlabel("X slope (amplitude per 100 km; right positive)")
    ax.set_ylabel("Y slope (amplitude per 100 km; down positive)")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(
        variant_output_dir
        / "plane_slope_vector_colored_by_time.png",
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def make_directional_keogram(
    data_cube,
    angle_deg,
    width_pixels=3,
):
    """
    Create a directional keogram through the image center.

    angle_deg:
        0 degrees  = E-W slice
        90 degrees = N-S slice
    """
    nt_local, ny_local, nx_local = data_cube.shape

    theta_local = np.deg2rad(angle_deg)

    # Unit vector along the cut
    ux = np.cos(theta_local)
    uy = np.sin(theta_local)

    # Perpendicular unit vector used for strip averaging
    px = -np.sin(theta_local)
    py = np.cos(theta_local)

    cx = (nx_local - 1) / 2.0
    cy = (ny_local - 1) / 2.0

    half_length_pix = min(nx_local, ny_local) // 2
    distances_pix = np.arange(
        -half_length_pix,
        half_length_pix + 1,
    )

    offsets = (
        np.arange(width_pixels)
        - width_pixels // 2
    )

    keogram = np.full(
        (nt_local, len(distances_pix)),
        np.nan,
        dtype=np.float32,
    )

    # Sampling coordinates do not change with time, so calculate
    # them once for each perpendicular offset.
    sampling_coordinates = []

    for off in offsets:
        x_sample = cx + distances_pix * ux + off * px
        y_sample = cy + distances_pix * uy + off * py

        valid = (
            (x_sample >= 0)
            & (x_sample <= nx_local - 1)
            & (y_sample >= 0)
            & (y_sample <= ny_local - 1)
        )

        sampling_coordinates.append(
            (x_sample, y_sample, valid)
        )

    for it in range(nt_local):
        sampled_slices = []

        for x_sample, y_sample, valid in sampling_coordinates:
            values = np.full(
                len(distances_pix),
                np.nan,
                dtype=np.float32,
            )

            values[valid] = map_coordinates(
                data_cube[it],
                [
                    y_sample[valid],
                    x_sample[valid],
                ],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(
            sampled_slices,
            axis=0,
        )

    # DX_KM = DY_KM in this synthetic grid.
    distance_km = distances_pix * DX_KM

    return keogram, distance_km


def extract_profiles(keogram, distance_km):
    """Return profiles at all requested distances."""
    profiles = {}

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )
        actual_distance = float(distance_km[distance_index])
        profiles[actual_distance] = keogram[:, distance_index]

    return profiles


def save_keogram_figure(
    keogram,
    distance_km,
    angle_deg,
    variant_folder_name,
):
    """Save one directional keogram image."""
    variant_dir = VARIANT_DIRS[variant_folder_name]
    variant_label = variant_labels[variant_folder_name]

    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )
    visible_distance_mask = (
        (distance_km >= KEOGRAM_DISTANCE_MIN)
        & (distance_km <= KEOGRAM_DISTANCE_MAX)
    )

    visible_keogram = keogram[
        np.ix_(visible_time_mask, visible_distance_mask)
    ]

    keogram_vmin, keogram_vmax = symmetric_limits(
        visible_keogram,
        percentile=KEOGRAM_COLOR_PERCENTILE,
    )

    fig, ax = plt.subplots(figsize=(9, 5))

    image = ax.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=keogram_vmin,
        vmax=keogram_vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Perturbation amplitude")

    for requested_distance in PROFILE_DISTANCES_KM:
        distance_index = np.argmin(
            np.abs(distance_km - requested_distance)
        )
        actual_distance = distance_km[distance_index]

        ax.axhline(
            actual_distance,
            linewidth=1.0,
            linestyle="--",
            alpha=0.8,
        )

    ax.set_title(
        f"{variant_label} | Directional keogram: "
        f"{angle_deg:.0f}°"
    )
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("Distance along slice (km)")
    ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax.set_ylim(
        KEOGRAM_DISTANCE_MIN,
        KEOGRAM_DISTANCE_MAX,
    )

    plt.tight_layout()

    output_path = (
        variant_dir
        / "keogram_images"
        / f"synthetic_keogram_angle_{angle_deg:03.0f}.png"
    )

    fig.savefig(
        output_path,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_profile_array(
    profiles,
    angle_deg,
    variant_folder_name,
):
    """Save time and the selected profiles in one NumPy array."""
    ordered_profiles = []

    for requested_distance in PROFILE_DISTANCES_KM:
        actual_distance = min(
            profiles.keys(),
            key=lambda value: abs(value - requested_distance),
        )
        ordered_profiles.append(profiles[actual_distance])

    profile_array = np.column_stack(
        [times_min] + ordered_profiles
    )

    output_path = (
        VARIANT_DIRS[variant_folder_name]
        / "profile_arrays"
        / (
            "synthetic_keogram_profiles_"
            f"angle_{angle_deg:03.0f}.npy"
        )
    )

    np.save(output_path, profile_array)


# ==================================================
# Fit one plane to every frame of every data variant
# ==================================================
plane_fit_results_by_variant = {}
print("\nCalculating plane fits for all variants...")

for variant_folder_name, variant_label in VARIANT_SPECS:
    print(f"  Plane fitting: {variant_label}")
    results = fit_plane_to_cube(
        variant_cubes[variant_folder_name],
        X_CENTERED_KM,
        Y_CENTERED_KM,
    )
    plane_fit_results_by_variant[variant_folder_name] = results
    save_plane_fit_results(results, variant_folder_name)
    save_slope_vector_plot(results, variant_folder_name)

# Comparison figure containing all six variants.
frame_numbers = np.arange(nt)
fig_plane_compare, (ax_angle_compare, ax_slope_compare) = plt.subplots(
    2, 1, figsize=(11, 9), sharex=True, constrained_layout=True
)

for variant_folder_name, variant_label in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    ax_angle_compare.plot(
        frame_numbers, results["angle_deg"], linewidth=1.2, label=variant_label
    )
    ax_slope_compare.plot(
        frame_numbers, results["slope_per_100km"], linewidth=1.2, label=variant_label
    )

ax_angle_compare.set_title("Plane-fit direction and slope for active variants")
ax_angle_compare.set_ylabel("Plane angle (deg)")
ax_angle_compare.set_ylim(-180, 180)
ax_angle_compare.set_yticks([-180, -120, -60, 0, 60, 120, 180])
ax_angle_compare.grid(True, alpha=0.3)
ax_angle_compare.legend(loc="best", fontsize=8, ncol=2)

ax_slope_compare.set_xlabel("Frame")
ax_slope_compare.set_ylabel("Plane slope per 100 km")
ax_slope_compare.grid(True, alpha=0.3)
ax_slope_compare.legend(loc="best", fontsize=8, ncol=2)

fig_plane_compare.savefig(
    PLANE_FIT_OUTPUT_DIR / "plane_angle_and_slope_vs_frame_active_variants.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_plane_compare)


# ==================================================
# Slope-vector trajectories for active variants
# ==================================================
number_of_variants = len(VARIANT_SPECS)
number_of_columns = min(3, number_of_variants)
number_of_rows = int(np.ceil(number_of_variants / number_of_columns))

fig_vector_compare, axes_vector = plt.subplots(
    nrows=number_of_rows,
    ncols=number_of_columns,
    figsize=(5 * number_of_columns, 5 * number_of_rows),
    constrained_layout=True,
    squeeze=False,
)
axes_vector = axes_vector.ravel()

vector_time_mask = (
    (times_min >= SLOPE_VECTOR_TIME_MIN)
    & (times_min <= SLOPE_VECTOR_TIME_MAX)
)

all_vector_components = []
for variant_folder_name, _ in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    all_vector_components.extend(
        [
            results["slope_x_per_100km"][vector_time_mask],
            results["slope_y_per_100km"][vector_time_mask],
        ]
    )

all_vector_components.extend(
    [
        bias_curve_x_per_100km,
        bias_curve_y_per_100km,
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ]
)

common_vector_limit = np.nanmax(
    np.abs(np.concatenate(all_vector_components))
)
if not np.isfinite(common_vector_limit) or common_vector_limit <= 0:
    common_vector_limit = 1.0
common_vector_limit *= 1.10

shared_scatter = None

for ax, (variant_folder_name, variant_label) in zip(
    axes_vector,
    VARIANT_SPECS,
):
    results = plane_fit_results_by_variant[variant_folder_name]
    slope_x = results["slope_x_per_100km"]
    slope_y = results["slope_y_per_100km"]

    valid = (
        np.isfinite(slope_x)
        & np.isfinite(slope_y)
        & np.isfinite(times_min)
        & (times_min >= SLOPE_VECTOR_TIME_MIN)
        & (times_min <= SLOPE_VECTOR_TIME_MAX)
    )

    if np.any(valid):
        ax.plot(
            slope_x[valid],
            slope_y[valid],
            linewidth=0.8,
            alpha=0.30,
        )
        shared_scatter = ax.scatter(
            slope_x[valid],
            slope_y[valid],
            c=times_min[valid],
            cmap=SLOPE_VECTOR_CMAP,
            vmin=SLOPE_VECTOR_TIME_MIN,
            vmax=SLOPE_VECTOR_TIME_MAX,
            s=18,
            zorder=3,
        )

    ax.axhline(0.0, color="0.5", linewidth=0.7, alpha=0.6)
    ax.axvline(0.0, color="0.5", linewidth=0.7, alpha=0.6)
    ax.set_xlim(-common_vector_limit, common_vector_limit)
    ax.set_ylim(common_vector_limit, -common_vector_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(variant_label)
    ax.set_xlabel("X slope (amp/100 km)")
    ax.set_ylabel("Y slope (amp/100 km)")
    ax.grid(True, alpha=0.3)

    if "bias" in variant_folder_name:
        ax.plot(
            bias_curve_x_per_100km,
            bias_curve_y_per_100km,
            color="black",
            linewidth=1.8,
            alpha=0.9,
            zorder=4,
        )
        ax.scatter(
            BIAS_ANCHOR_X_PER_100KM,
            BIAS_ANCHOR_Y_PER_100KM,
            marker="x",
            s=55,
            linewidths=1.7,
            c="black",
            zorder=5,
        )

for unused_ax in axes_vector[number_of_variants:]:
    unused_ax.remove()

fig_vector_compare.suptitle(
    "Plane-fit Slope Vector Trajectories Colored by Time",
    fontsize=15,
)

if shared_scatter is not None:
    cbar = fig_vector_compare.colorbar(
        shared_scatter,
        ax=axes_vector[:number_of_variants].tolist(),
        shrink=0.85,
        pad=0.02,
    )
    cbar.set_label("Time (min)")

fig_vector_compare.savefig(
    PLANE_FIT_OUTPUT_DIR
    / "plane_slope_vector_colored_by_time_active_variants.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_vector_compare)

# ==================================================
# Generate all keograms and six-subplot profile figures
# ==================================================
for angle_deg in KEOGRAM_ANGLES_DEG:
    print(f"Processing angle {angle_deg:.0f}°")

    angle_profiles = {}

    # Create and save one keogram for every variant.
    for variant_folder_name, _ in VARIANT_SPECS:
        data_cube = variant_cubes[variant_folder_name]

        keogram, distance_km = make_directional_keogram(
            data_cube,
            angle_deg,
            width_pixels=KEOGRAM_WIDTH_PIXELS,
        )

        angle_profiles[variant_folder_name] = extract_profiles(
            keogram,
            distance_km,
        )

        np.save(
            VARIANT_DIRS[variant_folder_name]
            / "keogram_arrays"
            / f"synthetic_keogram_angle_{angle_deg:03.0f}.npy",
            keogram,
        )

        save_keogram_figure(
            keogram=keogram,
            distance_km=distance_km,
            angle_deg=angle_deg,
            variant_folder_name=variant_folder_name,
        )

        save_profile_array(
            profiles=angle_profiles[variant_folder_name],
            angle_deg=angle_deg,
            variant_folder_name=variant_folder_name,
        )

    # --------------------------------------------------
    # Profile subplots for the active variants
    # --------------------------------------------------
    number_of_variants = len(VARIANT_SPECS)
    fig_profile, axes = plt.subplots(
        nrows=number_of_variants,
        ncols=1,
        figsize=(11, max(5, 3.3 * number_of_variants)),
        sharex=True,
        constrained_layout=True,
        squeeze=False,
    )
    axes = axes.ravel()

    visible_time_mask = (
        (times_min >= 0)
        & (times_min <= KEOGRAM_TIME_MIN_MAX)
    )

    shared_profile_values = []
    if PROFILE_USE_SHARED_Y_LIMITS:
        for variant_folder_name, _ in VARIANT_SPECS:
            for profile in angle_profiles[variant_folder_name].values():
                valid = visible_time_mask & np.isfinite(profile)
                if np.any(valid):
                    shared_profile_values.append(profile[valid])

        if shared_profile_values:
            shared_ymin, shared_ymax = padded_limits(
                np.concatenate(shared_profile_values),
                PROFILE_Y_PADDING_FRACTION,
            )
        else:
            shared_ymin, shared_ymax = -1.0, 1.0

    for ax, (variant_folder_name, variant_label) in zip(
        axes,
        VARIANT_SPECS,
    ):
        profiles = angle_profiles[variant_folder_name]
        subplot_values = []

        for requested_distance in PROFILE_DISTANCES_KM:
            actual_distance = min(
                profiles.keys(),
                key=lambda value: abs(value - requested_distance),
            )
            profile = profiles[actual_distance]

            ax.plot(
                times_min,
                profile,
                linewidth=1.3,
                label=f"{actual_distance:+.1f} km",
            )

            valid = visible_time_mask & np.isfinite(profile)
            if np.any(valid):
                subplot_values.append(profile[valid])

        ax.axhline(0.0, linewidth=0.8, linestyle="--", alpha=0.7)
        ax.set_title(variant_label)
        ax.set_ylabel("Amplitude")
        ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", ncol=3)

        if PROFILE_USE_SHARED_Y_LIMITS:
            ax.set_ylim(shared_ymin, shared_ymax)
        elif subplot_values:
            subplot_ymin, subplot_ymax = padded_limits(
                np.concatenate(subplot_values),
                PROFILE_Y_PADDING_FRACTION,
            )
            ax.set_ylim(subplot_ymin, subplot_ymax)

    axes[-1].set_xlabel("Time (min)")

    fig_profile.suptitle(
        "Synthetic Keogram Profiles: Active Variants\n"
        f"Directional angle = {angle_deg:.0f}°",
        fontsize=15,
    )

    comparison_path = (
        COMPARISON_PROFILE_DIR
        / (
            "synthetic_keogram_profiles_active_variants_"
            f"angle_{angle_deg:03.0f}.png"
        )
    )

    fig_profile.savefig(
        comparison_path,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig_profile)

# ==================================================
# Save common arrays and component arrays
# ==================================================
np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "times_min.npy", times_min)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)
np.save(OUT_DIR / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)
np.save(
    OUT_DIR / "profile_distances_km.npy",
    np.asarray(PROFILE_DISTANCES_KM),
)

component_dir = OUT_DIR / "00_components"
component_dir.mkdir(parents=True, exist_ok=True)

np.save(component_dir / "wave_cube.npy", wave_cube)
np.save(
    component_dir / "propagation_angles_deg.npy",
    propagation_angles_deg,
)
np.save(
    component_dir / "packet_center_x_km.npy",
    packet_center_x_km,
)
np.save(
    component_dir / "packet_center_y_km.npy",
    packet_center_y_km,
)
np.save(
    component_dir / "temporal_phase_rad.npy",
    temporal_phase_rad,
)
np.save(component_dir / "temporal_slope_cube.npy", temporal_slope_cube)
np.save(component_dir / "temporal_slope_values.npy", temporal_slope_values)
np.save(component_dir / "bias_cube.npy", bias_cube)
np.save(component_dir / "bias_angles_deg.npy", bias_angles_deg)
np.save(
    component_dir / "bias_gradients_per_100km.npy",
    bias_gradients_per_100km,
)
np.save(
    component_dir / "bias_x_per_100km.npy",
    bias_x_per_100km,
)
np.save(
    component_dir / "bias_y_per_100km.npy",
    bias_y_per_100km,
)
np.save(component_dir / "noise_cube.npy", noise_cube)

for variant_folder_name, _ in VARIANT_SPECS:
    results = plane_fit_results_by_variant[variant_folder_name]
    np.save(
        component_dir / f"{variant_folder_name}_plane_angle_deg.npy",
        results["angle_deg"],
    )
    np.save(
        component_dir / f"{variant_folder_name}_plane_slope_per_100km.npy",
        results["slope_per_100km"],
    )


# ==================================================
# Save bias interpolation diagnostic plot
# ==================================================
fig_bias, axes_bias = plt.subplots(
    3,
    1,
    figsize=(9, 10),
    sharex=True,
)

ax_components, ax_angle, ax_gradient = axes_bias

ax_components.plot(
    times_min,
    bias_x_per_100km,
    linewidth=2.0,
    label="X component",
)
ax_components.plot(
    times_min,
    bias_y_per_100km,
    linewidth=2.0,
    label="Y component",
)
ax_components.plot(
    bias_curve_time_min,
    bias_curve_x_per_100km,
    linewidth=1.2,
    linestyle="--",
    alpha=0.8,
    label="Rounded path X",
)
ax_components.plot(
    bias_curve_time_min,
    bias_curve_y_per_100km,
    linewidth=1.2,
    linestyle="--",
    alpha=0.8,
    label="Rounded path Y",
)
ax_components.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_X_PER_100KM,
    marker="o",
    zorder=3,
    label="X anchors",
)
ax_components.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    BIAS_ANCHOR_Y_PER_100KM,
    marker="s",
    zorder=3,
    label="Y anchors",
)
ax_components.axhline(
    0.0,
    linewidth=0.8,
    linestyle="--",
    alpha=0.7,
)
ax_components.set_ylabel("Gradient component\n(amp per 100 km)")
ax_components.set_title(
    "Directional bias from rounded-path physical gradient anchors"
)
ax_components.grid(True, alpha=0.3)
ax_components.legend(loc="best", ncol=2)

ax_angle.plot(
    times_min,
    bias_angles_deg,
    linewidth=2.0,
)
ax_angle.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    (
        np.degrees(
            np.arctan2(
                BIAS_ANCHOR_Y_PER_100KM,
                BIAS_ANCHOR_X_PER_100KM,
            )
        )
        + 180.0
    )
    % 360.0
    - 180.0,
    zorder=3,
)
ax_angle.set_ylabel("Direction (deg)")
ax_angle.set_ylim(-180, 180)
ax_angle.set_yticks(
    [-180, -120, -60, 0, 60, 120, 180]
)
ax_angle.grid(True, alpha=0.3)

ax_gradient.plot(
    times_min,
    bias_gradients_per_100km,
    linewidth=2.0,
)
ax_gradient.scatter(
    BIAS_ANCHOR_TIMES_MIN,
    np.hypot(
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ),
    zorder=3,
)
ax_gradient.set_xlabel("Time (min)")
ax_gradient.set_ylabel("Magnitude\n(amp per 100 km)")
ax_gradient.grid(True, alpha=0.3)

plt.tight_layout()
fig_bias.savefig(
    component_dir / "bias_vector_component_rounded_path.png",
    dpi=DPI,
    bbox_inches="tight",
)
plt.close(fig_bias)


# ==================================================
# Save parameters
# ==================================================
with open(
    OUT_DIR / "synthetic_wave_parameters.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(f"Field size X (km): {FIELD_SIZE_X_KM}\n")
    file.write(f"Field size Y (km): {FIELD_SIZE_Y_KM}\n")
    file.write(f"Grid spacing X (km): {DX_KM}\n")
    file.write(f"Grid spacing Y (km): {DY_KM}\n")
    file.write(f"Time step (s): {DT_SECONDS}\n")
    file.write(f"Total time (min): {TOTAL_TIME_MIN}\n")
    file.write(f"Wavelength (km): {WAVELENGTH_KM}\n")
    file.write(f"Gaussian sigma (km): {GAUSSIAN_WIDTH_KM}\n")
    file.write(f"Maximum amplitude: {MAX_AMPLITUDE}\n")
    file.write(f"Processing mode: {PROCESSING_MODE}\n")
    file.write(
        "Propagation angle schedule: "
        f"{PROPAGATION_ANGLE_START_DEG} deg at and before "
        f"{PROPAGATION_ANGLE_START_TIME_MIN} min, changing linearly to "
        f"{PROPAGATION_ANGLE_END_DEG} deg at "
        f"{PROPAGATION_ANGLE_END_TIME_MIN} min, then held constant\n"
    )
    file.write(f"Wave speed (m/s): {WAVE_SPEED_M_S}\n")
    file.write(f"Wave time shift (min): {WAVE_TIME_SHIFT_MIN}\n")
    file.write(
        f"Temporal slope start: {TEMPORAL_SLOPE_START}\n"
    )
    file.write(
        f"Temporal slope end: {TEMPORAL_SLOPE_END}\n"
    )
    file.write(f"Noise mean: {NOISE_MEAN}\n")
    file.write(f"Noise standard deviation: {NOISE_STD}\n")
    file.write(f"Noise random seed: {NOISE_RANDOM_SEED}\n")
    file.write(
        "Bias interpolation: rounded polyline interpolation of x and y "
        "gradient components\n"
    )
    file.write(
        "Bias behavior before 154 min: first anchor held constant\n"
    )
    file.write("Bias is zero at the spatial field center\n")
    file.write("Plane-fit direction: increasing perturbation direction\n")
    file.write("Plane-fit angle range: [-180, 180) deg\n")
    file.write(
        "Bias angle derived from interpolated x and y components\n"
    )
    file.write("Plane-fit slope units: perturbation amplitude per 100 km\n")
    file.write(
        "Bias angle convention: 0 deg east, +90 deg south\n"
    )
    file.write(
        "Bias anchors "
        "(time_min, x_per_100km, y_per_100km, "
        "magnitude_per_100km, direction_deg):\n"
    )

    for time_value, x_value, y_value in zip(
        BIAS_ANCHOR_TIMES_MIN,
        BIAS_ANCHOR_X_PER_100KM,
        BIAS_ANCHOR_Y_PER_100KM,
    ):
        magnitude_value = np.hypot(x_value, y_value)
        direction_value = (
            (
                np.degrees(
                    np.arctan2(y_value, x_value)
                )
                + 180.0
            )
            % 360.0
        ) - 180.0

        file.write(
            f"  {time_value}, {x_value}, {y_value}, "
            f"{magnitude_value}, {direction_value}\n"
        )


# ==================================================
# Print output summary
# ==================================================
print("\nFinished.")
print("Main output folder:", OUT_DIR)
print("\nVariant folders:")
for folder_name, label in VARIANT_SPECS:
    print(f"  {label}: {VARIANT_DIRS[folder_name]}")

print(
    "\nProfile comparisons:",
    COMPARISON_PROFILE_DIR,
)
print(
    "Number of angles processed:",
    len(KEOGRAM_ANGLES_DEG),
)
print(
    "Profile distances:",
    PROFILE_DISTANCES_KM,
)
print(
    "Plane-fit outputs:",
    PLANE_FIT_OUTPUT_DIR,
)
print(
    "Slope-vector plots: one per active variant plus an active-variant comparison"
)


Synthetic data shape: (251, 501, 501)
Active variants:
  01_waves_only: Waves only
  02_bias_only: Bias only
  03_waves_plus_slope: Waves + slope
  04_waves_plus_bias: Waves + bias
  05_waves_plus_slope_plus_bias: Waves + slope + bias
  06_waves_plus_slope_plus_bias_plus_noise: Waves + slope + bias + noise

Calculating plane fits for all variants...
  Plane fitting: Waves only
  Plane fitting: Bias only
  Plane fitting: Waves + slope
  Plane fitting: Waves + bias
  Plane fitting: Waves + slope + bias
  Plane fitting: Waves + slope + bias + noise
Processing angle -90°
Processing angle -85°
Processing angle -80°
Processing angle -75°
Processing angle -70°
Processing angle -65°
Processing angle -60°
Processing angle -55°
Processing angle -50°
Processing angle -45°
Processing angle -40°
Processing angle -35°
Processing angle -30°
Processing angle -25°
Processing angle -20°
Processing angle -15°
Processing angle -10°
Processing angle -5°
Processing angle 0°
Processing angle 5°
Processing an

## Correlation Analysis, Also plot original keogram and smoothed ranged keogram. 

In [70]:
# ==================================================
# Version note
# This version processes the output of the rounded-bias six-variant
# synthetic keogram generator:
#   1. Waves only
#   2. Bias only
#   3. Waves + temporal slope
#   4. Waves + directional bias
#   5. Waves + temporal slope + directional bias
#   6. Waves + temporal slope + directional bias + noise
#   7. Variant 6 with a row-by-row linear temporal detrend
#
# Bias correction and known linear-slope correction are disabled.
# The seventh analysis variant is generated here by independently
# removing a least-squares linear temporal trend from every distance row
# of the sixth generator output.
# ==================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks


# ==================================================
# Settings
# ==================================================
# Folder created by the rounded-bias six-variant generator.
# It contains times_s.npy at the root and one keogram_arrays folder
# inside each saved data-variant folder.
BASE_OUT_DIR = Path(
    r"synthetic_wave_keogram_outputs_six_variants"
)

# Process every directional keogram:
# 0, 5, 10, ..., 175 degrees
KEOGRAM_ANGLES_DEG = np.arange(
    -90,
    91,
    5,
    dtype=int,
)

# Angle range shown in the combined phase-speed comparison plot.
# All angles are still processed and saved; this only limits the
# combined phase-speed figure.
PHASE_SPEED_PLOT_ANGLE_MIN_DEG = -65.0
PHASE_SPEED_PLOT_ANGLE_MAX_DEG = 65.0

# Distances shown in the six-variant profile comparison figures.
PROFILE_COMPARISON_DISTANCES_KM = np.array(
    [0.0, 125.0, 250.0],
    dtype=float,
)

# Time range used for correlation
CORRELATION_TIME_MIN = 200.0
CORRELATION_TIME_MAX = 400.0

#CORRELATION_TIME_MIN = 0.0
#CORRELATION_TIME_MAX = 300.0

# Distance range included in the correlation map
CORRELATION_DISTANCE_MIN_KM = 0.0
CORRELATION_DISTANCE_MAX_KM = 250.0

# Shift range in minutes used for the correlation map and ridge fit.
SHIFT_MIN_MINUTES = -50.0
SHIFT_MAX_MINUTES = 50.0

# The final 0 km correlation-profile subplot uses a wider shift range.
# A factor of 4 changes -50...+50 min to -200...+200 min while
# leaving the correlation map and ridge fit unchanged.
ZERO_KM_PROFILE_SHIFT_FACTOR = 4.0
ZERO_KM_PROFILE_SHIFT_MIN_MINUTES = (
    ZERO_KM_PROFILE_SHIFT_FACTOR * SHIFT_MIN_MINUTES
)
ZERO_KM_PROFILE_SHIFT_MAX_MINUTES = (
    ZERO_KM_PROFILE_SHIFT_FACTOR * SHIFT_MAX_MINUTES
)

# Minimum number of overlapping time samples
MIN_VALID_CORRELATION_POINTS = 20

# Minimum maximum-correlation value used in the slope fit
MIN_CORRELATION_FOR_SLOPE = 0.50

# Remove ridge outliers before the final linear fit. At each iteration,
# points with residuals farther than this many robust standard deviations
# from the fitted line are rejected.
REMOVE_RIDGE_OUTLIERS = True
RIDGE_MAXIMUM_JUMP_MINUTES = 20.0
RIDGE_OUTLIER_SIGMA = 3.0
RIDGE_OUTLIER_MAX_ITERATIONS = 10
RIDGE_OUTLIER_MIN_POINTS = 3

# Distance range used in the ridge fit
SLOPE_FIT_DISTANCE_MIN_KM = 0.0
SLOPE_FIT_DISTANCE_MAX_KM = 250.0

DPI = 200

CORRELATION_CMAP = "RdBu_r"
KEOGRAM_CMAP = "RdBu_r"

# Use symmetric automatic keogram limits based on this percentile.
# Set KEOGRAM_ABS_LIMIT to a number to force fixed limits.
KEOGRAM_ABS_LIMIT = None
KEOGRAM_AUTO_PERCENTILE = 99.0

# Correlation color scale
CORRELATION_VMIN = 0.70
CORRELATION_VMAX = 1.00

# Local maxima search on the 0 km correlation-shift profile.
# The left maximum is searched at negative shifts and the right
# maximum is searched at positive shifts. The zero-shift point is
# therefore excluded automatically.
LOCAL_MAXIMUM_MIN_PROMINENCE = 0.0
LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS = 1


# ==================================================
# Optional 2D smoothing
# ==================================================
APPLY_2D_SMOOTHING = True

# Keogram dimensions:
# axis 0 = time
# axis 1 = distance
SMOOTH_SIGMA_TIME_PIXELS = 1.5
SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0

SMOOTH_MODE = "nearest"
SMOOTH_WEIGHT_TOLERANCE = 1e-12


# ==================================================
# Linear temporal-slope correction settings
# ==================================================
# The synthetic generator adds a spatially uniform value that changes
# linearly from +0.8 at 0 min to -0.2 at 300 min. Enable this correction
# to subtract that known temporal trend from every distance row.
APPLY_LINEAR_SLOPE_CORRECTION = False

TEMPORAL_SLOPE_START = 0.8
TEMPORAL_SLOPE_END = -0.2
TEMPORAL_SLOPE_START_TIME_MIN = 0.0
TEMPORAL_SLOPE_END_TIME_MIN = 300.0

# Outside the defined interval, hold the endpoint value constant.
LINEAR_SLOPE_HOLD_ENDPOINTS = True


# ==================================================
# Filename-safe numeric label
# ==================================================
def safe_number_label(value):
    value_float = float(value)

    if value_float.is_integer():
        return str(
            int(value_float)
        )

    return (
        f"{value_float:g}"
        .replace("-", "m")
        .replace(".", "p")
    )


# ==================================================
# NaN-aware 2D Gaussian smoothing
# ==================================================
def smooth_keogram_2d(
    data,
    sigma_time,
    sigma_distance,
    mode="nearest",
    weight_tolerance=1e-12,
):
    """
    Apply NaN-aware Gaussian smoothing to a 2D keogram.
    """
    data = np.asarray(
        data,
        dtype=float,
    )

    if data.ndim != 2:
        raise ValueError(
            "The keogram must be two-dimensional."
        )

    if sigma_time < 0 or sigma_distance < 0:
        raise ValueError(
            "Smoothing sigma values must be nonnegative."
        )

    if sigma_time == 0 and sigma_distance == 0:
        return data.copy()

    valid_mask = np.isfinite(
        data
    )

    data_filled = np.where(
        valid_mask,
        data,
        0.0,
    )

    weights = valid_mask.astype(
        float
    )

    sigma = (
        sigma_time,
        sigma_distance,
    )

    smoothed_values = gaussian_filter(
        data_filled,
        sigma=sigma,
        mode=mode,
    )

    smoothed_weights = gaussian_filter(
        weights,
        sigma=sigma,
        mode=mode,
    )

    smoothed = np.full(
        data.shape,
        np.nan,
        dtype=float,
    )

    valid_output = (
        smoothed_weights
        > weight_tolerance
    )

    smoothed[valid_output] = (
        smoothed_values[valid_output]
        / smoothed_weights[valid_output]
    )

    return smoothed


# ==================================================
# NaN-aware linear detrending along time
# ==================================================
def detrend_keogram_rows(
    data,
    times_min,
    minimum_points=3,
):
    """
    Independently remove a least-squares linear temporal trend from
    every distance row of a time x distance keogram.

    For each distance column, fit:

        value(t) = slope * t + intercept

    using only finite samples, then subtract the fitted line. This is
    data-driven detrending, not subtraction of the known synthetic slope.
    """
    data = np.asarray(data, dtype=float)
    times_min = np.asarray(times_min, dtype=float)

    if data.ndim != 2:
        raise ValueError(
            "The keogram must be two-dimensional."
        )

    if data.shape[0] != len(times_min):
        raise ValueError(
            "The number of keogram time samples must match times_min."
        )

    detrended = np.full_like(
        data,
        np.nan,
        dtype=float,
    )

    fitted_trends = np.full_like(
        data,
        np.nan,
        dtype=float,
    )

    slopes = np.full(
        data.shape[1],
        np.nan,
        dtype=float,
    )

    intercepts = np.full(
        data.shape[1],
        np.nan,
        dtype=float,
    )

    for distance_index in range(data.shape[1]):
        profile = data[:, distance_index]

        valid = (
            np.isfinite(profile)
            & np.isfinite(times_min)
        )

        if np.count_nonzero(valid) < minimum_points:
            continue

        slope, intercept = np.polyfit(
            times_min[valid],
            profile[valid],
            deg=1,
        )

        trend = slope * times_min + intercept

        detrended[valid, distance_index] = (
            profile[valid] - trend[valid]
        )

        fitted_trends[valid, distance_index] = trend[valid]
        slopes[distance_index] = slope
        intercepts[distance_index] = intercept

    return (
        detrended,
        fitted_trends,
        slopes,
        intercepts,
    )


# ==================================================
# Create spatially uniform linear temporal-slope correction
# ==================================================
def create_linear_slope_correction_keogram(
    times_min,
    n_distance,
    slope_start,
    slope_end,
    start_time_min,
    end_time_min,
    hold_endpoints=True,
):
    """
    Create the correction that removes the known uniform temporal slope.

    The synthetic field contains:
        temporal_slope(t)

    Therefore the correction added to the keogram is:
        -temporal_slope(t)

    The same correction is applied to every distance row.
    """
    times_min = np.asarray(times_min, dtype=float)

    if end_time_min <= start_time_min:
        raise ValueError(
            "TEMPORAL_SLOPE_END_TIME_MIN must exceed "
            "TEMPORAL_SLOPE_START_TIME_MIN."
        )

    normalized_time = (
        (times_min - start_time_min)
        / (end_time_min - start_time_min)
    )

    if hold_endpoints:
        normalized_time = np.clip(normalized_time, 0.0, 1.0)

    temporal_slope_values = (
        slope_start
        + normalized_time * (slope_end - slope_start)
    )

    correction_vs_time = -temporal_slope_values

    correction_keogram = np.broadcast_to(
        correction_vs_time[:, np.newaxis],
        (len(times_min), int(n_distance)),
    ).copy()

    return (
        correction_keogram,
        temporal_slope_values,
        correction_vs_time,
    )


# ==================================================
# NaN-aware Pearson correlation
# ==================================================
def nan_pearson_correlation(
    reference,
    comparison,
    minimum_points=20,
):
    """
    Calculate Pearson correlation using finite overlapping values.
    """
    reference = np.asarray(
        reference,
        dtype=float,
    )

    comparison = np.asarray(
        comparison,
        dtype=float,
    )

    if reference.shape != comparison.shape:
        raise ValueError(
            "The two arrays must have the same shape."
        )

    valid = (
        np.isfinite(reference)
        & np.isfinite(comparison)
    )

    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    reference_valid = reference[
        valid
    ]

    comparison_valid = comparison[
        valid
    ]

    reference_centered = (
        reference_valid
        - np.mean(reference_valid)
    )

    comparison_centered = (
        comparison_valid
        - np.mean(comparison_valid)
    )

    denominator = np.sqrt(
        np.sum(reference_centered ** 2)
        * np.sum(comparison_centered ** 2)
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 0
    ):
        return np.nan

    correlation = (
        np.sum(
            reference_centered
            * comparison_centered
        )
        / denominator
    )

    return np.clip(
        correlation,
        -1.0,
        1.0,
    )


# ==================================================
# Correlation for one integer-pixel shift
# ==================================================
def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    """
    Shift comparison_row and correlate it with reference_row.

    Positive shift:
        comparison row moves toward later times.

    Negative shift:
        comparison row moves toward earlier times.

    No circular wrapping is used.
    """
    reference_row = np.asarray(
        reference_row,
        dtype=float,
    )

    comparison_row = np.asarray(
        comparison_row,
        dtype=float,
    )

    if reference_row.shape != comparison_row.shape:
        raise ValueError(
            "Reference and comparison rows must have "
            "the same shape."
        )

    ntime = len(
        reference_row
    )

    if abs(shift_pixels) >= ntime:
        return np.nan

    if shift_pixels > 0:
        reference_overlap = reference_row[
            shift_pixels:
        ]

        comparison_overlap = comparison_row[
            :-shift_pixels
        ]

    elif shift_pixels < 0:
        shift_abs = abs(
            shift_pixels
        )

        reference_overlap = reference_row[
            :-shift_abs
        ]

        comparison_overlap = comparison_row[
            shift_abs:
        ]

    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference=reference_overlap,
        comparison=comparison_overlap,
        minimum_points=minimum_points,
    )


# ==================================================
# Calculate complete correlation map
# ==================================================
def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    """
    Correlate every selected distance row with the reference row
    for every requested shift.
    """
    keogram = np.asarray(
        keogram,
        dtype=float,
    )

    distance_km = np.asarray(
        distance_km,
        dtype=float,
    )

    shifts_pixels = np.asarray(
        shifts_pixels,
        dtype=int,
    )

    reference_index = np.argmin(
        np.abs(
            distance_km
            - reference_distance_km
        )
    )

    selected_indices = np.where(
        selected_distance_mask
    )[0]

    selected_distances = distance_km[
        selected_indices
    ]

    reference_row = keogram[
        :,
        reference_index,
    ]

    correlation_map = np.full(
        (
            len(selected_indices),
            len(shifts_pixels),
        ),
        np.nan,
        dtype=float,
    )

    for output_row_index, keogram_row_index in enumerate(
        selected_indices
    ):
        comparison_row = keogram[
            :,
            keogram_row_index,
        ]

        for shift_index, shift_pixels in enumerate(
            shifts_pixels
        ):
            correlation_map[
                output_row_index,
                shift_index,
            ] = calculate_shifted_correlation(
                reference_row=reference_row,
                comparison_row=comparison_row,
                shift_pixels=int(shift_pixels),
                minimum_points=minimum_points,
            )

    return (
        correlation_map,
        selected_distances,
        reference_index,
    )


# ==================================================
# Maximum-correlation shift for each row
# ==================================================
def find_best_shifts(
    correlation_map,
    shifts_minutes,
):
    """
    Find the shift in minutes having the largest correlation
    for every distance row.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    best_shift_minutes = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    maximum_correlation = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    for row_index in range(
        correlation_map.shape[0]
    ):
        correlation_row = correlation_map[
            row_index,
            :
        ]

        finite = np.isfinite(
            correlation_row
        )

        if not np.any(finite):
            continue

        finite_indices = np.where(
            finite
        )[0]

        local_max_index = np.argmax(
            correlation_row[finite]
        )

        full_max_index = finite_indices[
            local_max_index
        ]

        best_shift_minutes[row_index] = (
            shifts_minutes[full_max_index]
        )

        maximum_correlation[row_index] = (
            correlation_row[full_max_index]
        )

    return (
        best_shift_minutes,
        maximum_correlation,
    )


# ==================================================
# Extract the correlation-shift profile at 0 km
# ==================================================
def extract_zero_km_correlation_profile(
    correlation_map,
    selected_distances,
):
    """
    Return the correlation-versus-shift profile for the selected
    distance row nearest 0 km.
    """
    correlation_map = np.asarray(
        correlation_map,
        dtype=float,
    )

    selected_distances = np.asarray(
        selected_distances,
        dtype=float,
    )

    if correlation_map.ndim != 2:
        raise ValueError(
            "correlation_map must be two-dimensional."
        )

    if correlation_map.shape[0] != len(selected_distances):
        raise ValueError(
            "The number of correlation-map rows must equal "
            "the number of selected distances."
        )

    zero_distance_row_index = int(
        np.argmin(
            np.abs(selected_distances)
        )
    )

    zero_distance_km = float(
        selected_distances[zero_distance_row_index]
    )

    zero_km_profile = correlation_map[
        zero_distance_row_index,
        :
    ].copy()

    return (
        zero_km_profile,
        zero_distance_row_index,
        zero_distance_km,
    )


# ==================================================
# Find strongest local maximum on each side of zero
# ==================================================
def find_left_and_right_local_maxima(
    correlation_profile,
    shifts_minutes,
    minimum_prominence=0.0,
    minimum_distance_pixels=1,
):
    """
    Find the strongest finite local maximum at negative shift
    and the strongest finite local maximum at positive shift.

    The point at zero shift is not included in either search.

    If scipy.signal.find_peaks finds no strict local maximum on one
    side, the largest finite value on that side is used as a fallback.
    """
    correlation_profile = np.asarray(
        correlation_profile,
        dtype=float,
    )

    shifts_minutes = np.asarray(
        shifts_minutes,
        dtype=float,
    )

    if correlation_profile.shape != shifts_minutes.shape:
        raise ValueError(
            "correlation_profile and shifts_minutes must "
            "have the same shape."
        )

    def find_one_side(side_mask):
        side_indices = np.where(
            side_mask
            & np.isfinite(correlation_profile)
        )[0]

        if len(side_indices) == 0:
            return {
                "index": -1,
                "shift_minutes": np.nan,
                "correlation": np.nan,
                "used_fallback": False,
            }

        side_values = correlation_profile[
            side_indices
        ]

        local_indices, properties = find_peaks(
            side_values,
            prominence=minimum_prominence,
            distance=minimum_distance_pixels,
        )

        used_fallback = False

        if len(local_indices) > 0:
            local_values = side_values[
                local_indices
            ]

            strongest_local_position = local_indices[
                int(np.nanargmax(local_values))
            ]

            full_index = int(
                side_indices[strongest_local_position]
            )
        else:
            used_fallback = True

            strongest_side_position = int(
                np.nanargmax(side_values)
            )

            full_index = int(
                side_indices[strongest_side_position]
            )

        return {
            "index": full_index,
            "shift_minutes": float(
                shifts_minutes[full_index]
            ),
            "correlation": float(
                correlation_profile[full_index]
            ),
            "used_fallback": used_fallback,
        }

    left_result = find_one_side(
        shifts_minutes < 0.0
    )

    right_result = find_one_side(
        shifts_minutes > 0.0
    )

    return {
        "left": left_result,
        "right": right_result,
    }


# ==================================================
# Fit maximum-correlation ridge
# ==================================================
def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
    minimum_correlation=0.5,
    distance_min_km=None,
    distance_max_km=None,
    remove_outliers=True,
    maximum_ridge_jump_minutes=20.0,
    outlier_sigma=3.0,
    outlier_max_iterations=10,
    outlier_min_points=3,
):
    """
    Fit:

        shift_minutes = slope * distance_km + intercept
    """
    distances_km = np.asarray(
        distances_km,
        dtype=float,
    )

    best_shift_minutes = np.asarray(
        best_shift_minutes,
        dtype=float,
    )

    maximum_correlation = np.asarray(
        maximum_correlation,
        dtype=float,
    )

    fit_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= minimum_correlation
        )
    )

    if distance_min_km is not None:
        fit_mask &= (
            distances_km
            >= distance_min_km
        )

    if distance_max_km is not None:
        fit_mask &= (
            distances_km
            <= distance_max_km
        )

    number_of_candidate_points = np.count_nonzero(
        fit_mask
    )

    if number_of_candidate_points < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": number_of_candidate_points,
            "number_of_outliers_removed": 0,
            "fit_mask": fit_mask,
            "outlier_mask": np.zeros_like(fit_mask, dtype=bool),
        }

    # First retain the continuous ridge branch beginning at the smallest
    # fitted distance. A large shift jump usually means that the maximum-
    # correlation search has switched to a different correlation peak.
    retained_mask = fit_mask.copy()

    if remove_outliers:
        candidate_indices = np.flatnonzero(fit_mask)
        sorted_indices = candidate_indices[
            np.argsort(distances_km[candidate_indices])
        ]

        continuity_mask = np.zeros_like(fit_mask, dtype=bool)
        first_index = sorted_indices[0]
        continuity_mask[first_index] = True
        previous_shift = best_shift_minutes[first_index]

        for index in sorted_indices[1:]:
            current_shift = best_shift_minutes[index]
            ridge_jump = abs(current_shift - previous_shift)

            if ridge_jump <= maximum_ridge_jump_minutes:
                continuity_mask[index] = True
                previous_shift = current_shift

        # Do not apply continuity rejection if it would leave too few
        # points for a meaningful linear fit.
        if np.count_nonzero(continuity_mask) >= max(2, outlier_min_points):
            retained_mask = continuity_mask

    # Then iteratively reject points far from the preliminary ridge line.
    # MAD is used because it is much less affected by outliers than the
    # ordinary residual standard deviation.
    if remove_outliers:
        for _ in range(outlier_max_iterations):
            if np.count_nonzero(retained_mask) < max(3, outlier_min_points):
                break

            trial_distances = distances_km[retained_mask]
            trial_shifts = best_shift_minutes[retained_mask]
            trial_slope, trial_intercept = np.polyfit(
                trial_distances,
                trial_shifts,
                deg=1,
            )
            residuals = trial_shifts - (
                trial_slope * trial_distances + trial_intercept
            )
            residual_median = np.median(residuals)
            mad = np.median(np.abs(residuals - residual_median))

            if not np.isfinite(mad) or np.isclose(mad, 0.0):
                break

            robust_sigma = 1.4826 * mad
            keep_local = (
                np.abs(residuals - residual_median)
                <= outlier_sigma * robust_sigma
            )

            if np.all(keep_local):
                break

            retained_indices = np.flatnonzero(retained_mask)
            proposed_mask = retained_mask.copy()
            proposed_mask[retained_indices[~keep_local]] = False

            if np.count_nonzero(proposed_mask) < max(2, outlier_min_points):
                break

            retained_mask = proposed_mask

    outlier_mask = fit_mask & ~retained_mask
    fit_mask = retained_mask
    number_of_points = np.count_nonzero(fit_mask)

    fit_distances = distances_km[
        fit_mask
    ]

    fit_shifts = best_shift_minutes[
        fit_mask
    ]

    slope_min_per_km, intercept_minutes = np.polyfit(
        fit_distances,
        fit_shifts,
        deg=1,
    )

    predicted_shifts = (
        slope_min_per_km
        * fit_distances
        + intercept_minutes
    )

    residual_sum_squares = np.sum(
        (
            fit_shifts
            - predicted_shifts
        ) ** 2
    )

    total_sum_squares = np.sum(
        (
            fit_shifts
            - np.mean(fit_shifts)
        ) ** 2
    )

    if total_sum_squares > 0:
        r_squared = (
            1.0
            - residual_sum_squares
            / total_sum_squares
        )
    else:
        r_squared = np.nan

    if np.isclose(
        slope_min_per_km,
        0.0,
    ):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        speed_km_per_min = (
            1.0
            / slope_min_per_km
        )

        speed_m_per_s = (
            speed_km_per_min
            * 1000.0
            / 60.0
        )

    return {
        "slope_min_per_km": slope_min_per_km,
        "intercept_minutes": intercept_minutes,
        "speed_km_per_min": speed_km_per_min,
        "speed_m_per_s": speed_m_per_s,
        "r_squared": r_squared,
        "number_of_fit_points": number_of_points,
        "number_of_outliers_removed": int(np.count_nonzero(outlier_mask)),
        "fit_mask": fit_mask,
        "outlier_mask": outlier_mask,
    }


# ==================================================
# Keogram color limit
# ==================================================
def determine_keogram_abs_limit(
    full_keogram,
    limited_keogram,
):
    """
    Return one symmetric absolute color limit shared by both
    keogram panels.
    """
    if KEOGRAM_ABS_LIMIT is not None:
        abs_limit = float(
            KEOGRAM_ABS_LIMIT
        )

        if (
            not np.isfinite(abs_limit)
            or abs_limit <= 0
        ):
            raise ValueError(
                "KEOGRAM_ABS_LIMIT must be positive and finite."
            )

        return abs_limit

    finite_values = []

    for data in [
        full_keogram,
        limited_keogram,
    ]:
        data = np.asarray(
            data,
            dtype=float,
        )

        finite = np.abs(
            data[np.isfinite(data)]
        )

        if finite.size > 0:
            finite_values.append(
                finite
            )

    if len(finite_values) == 0:
        return 1.0

    all_finite = np.concatenate(
        finite_values
    )

    abs_limit = np.nanpercentile(
        all_finite,
        KEOGRAM_AUTO_PERCENTILE,
    )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = np.nanmax(
            all_finite
        )

    if (
        not np.isfinite(abs_limit)
        or abs_limit <= 0
    ):
        abs_limit = 1.0

    return float(
        abs_limit
    )


# ==================================================
# Plot keograms, correlation map, ridge, and profile
# ==================================================
def plot_correlation_map(
    angle_deg,
    full_keogram,
    full_times_min,
    full_distance_km,
    limited_keogram,
    limited_times_min,
    limited_distance_km,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_km_correlation_profile,
    zero_km_profile_shifts_minutes,
    zero_km_distance,
    zero_km_local_maxima,
    dataset_name,
    output_file,
):
    """
    Plot:
      1. full-range original keogram;
      2. limited-range smoothed/corrected keogram;
      3. time profiles at the start, middle, and end distances;
      4. correlation versus time shift and distance;
      5. correlation versus shift for the 0 km row.
    """
    fig, (
        ax_full,
        ax_limited,
        ax_keogram_profiles,
        ax_map,
        ax_profile,
    ) = plt.subplots(
        nrows=5,
        ncols=1,
        figsize=(11, 19),
        gridspec_kw={
            "height_ratios": [
                1.55,
                1.55,
                1.15,
                2.5,
                1.15,
            ],
        },
    )

    keogram_abs_limit = determine_keogram_abs_limit(
        full_keogram=full_keogram,
        limited_keogram=limited_keogram,
    )

    # ==================================================
    # Full-range original keogram
    # ==================================================
    full_mesh = ax_full.pcolormesh(
        full_times_min,
        full_distance_km,
        full_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        vmin=-keogram_abs_limit,
        vmax=keogram_abs_limit,
    )

    full_cbar = fig.colorbar(
        full_mesh,
        ax=ax_full,
        fraction=0.025,
        pad=0.02,
    )

    full_cbar.set_label(
        "Keogram value"
    )

    ax_full.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_full.set_xlim(
        np.nanmin(full_times_min),
        np.nanmax(full_times_min),
    )

    ax_full.set_ylim(
        np.nanmin(full_distance_km),
        np.nanmax(full_distance_km),
    )

    ax_full.set_title(
        f"{angle_deg}° Full-Range Original Keogram"
    )

    ax_full.set_xlabel(
        "Time (min)"
    )

    ax_full.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Limited-range processed keogram
    # ==================================================
    limited_mesh = ax_limited.pcolormesh(
        limited_times_min,
        limited_distance_km,
        limited_keogram.T,
        shading="auto",
        cmap=KEOGRAM_CMAP,
        #vmin=-keogram_abs_limit,
        #vmax=keogram_abs_limit,
    )

    limited_cbar = fig.colorbar(
        limited_mesh,
        ax=ax_limited,
        fraction=0.025,
        pad=0.02,
    )

    limited_cbar.set_label(
        "Keogram value"
    )

    ax_limited.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ax_limited.set_xlim(
        CORRELATION_TIME_MIN,
        CORRELATION_TIME_MAX,
    )

    ax_limited.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_limited.set_title(
        "Limited-Range Processed Keogram\n"
        f"{dataset_name}"
    )

    ax_limited.set_xlabel(
        "Time (min)"
    )

    ax_limited.set_ylabel(
        "Distance along slice (km)"
    )

    # ==================================================
    # Profiles at start, middle, and end distance rows
    # ==================================================
    profile_target_distances = np.array(
        [
            limited_distance_km[0],
            0.5 * (limited_distance_km[0] + limited_distance_km[-1]),
            limited_distance_km[-1],
        ],
        dtype=float,
    )

    profile_labels = ["Start", "Middle", "End"]
    profile_colors = ["tab:blue", "tab:orange", "tab:green"]

    for target_distance, label, color in zip(
        profile_target_distances,
        profile_labels,
        profile_colors,
    ):
        row_index = int(np.nanargmin(
            np.abs(limited_distance_km - target_distance)
        ))
        actual_distance = limited_distance_km[row_index]
        ax_keogram_profiles.plot(
            limited_times_min,
            limited_keogram[:, row_index],
            color=color,
            linewidth=1.2,
            label=f"{label}: {actual_distance:g} km",
        )

    ax_keogram_profiles.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=0.9,
    )
    ax_keogram_profiles.set_xlim(
        CORRELATION_TIME_MIN,
        CORRELATION_TIME_MAX,
    )
    ax_keogram_profiles.set_title(
        "Processed Keogram Profiles at Start, Middle, and End Rows"
    )
    ax_keogram_profiles.set_xlabel("Time (min)")
    ax_keogram_profiles.set_ylabel("Keogram value")
    ax_keogram_profiles.grid(True, alpha=0.3)
    ax_keogram_profiles.legend(loc="best", fontsize=8)

    # ==================================================
    # Correlation map
    # ==================================================
    if len(shifts_minutes) > 1:
        shift_spacing = np.median(
            np.diff(shifts_minutes)
        )
    else:
        shift_spacing = 1.0

    if len(selected_distances) > 1:
        distance_spacing = np.median(
            np.diff(selected_distances)
        )
    else:
        distance_spacing = 1.0

    im = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=CORRELATION_CMAP,
        vmin=CORRELATION_VMIN,
        vmax=CORRELATION_VMAX,
        extent=[
            shifts_minutes[0]
            - shift_spacing / 2.0,
            shifts_minutes[-1]
            + shift_spacing / 2.0,
            selected_distances[0]
            - distance_spacing / 2.0,
            selected_distances[-1]
            + distance_spacing / 2.0,
        ],
    )

    cbar = fig.colorbar(
        im,
        ax=ax_map,
        fraction=0.025,
        pad=0.02,
    )

    cbar.set_label(
        "Pearson correlation with 0 km row"
    )

    ax_map.axvline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.2,
        label="Zero shift",
    )

    ax_map.axhline(
        zero_km_distance,
        color="white",
        linestyle=":",
        linewidth=1.5,
        label=f"Profile row ({zero_km_distance:g} km)",
    )

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= MIN_CORRELATION_FOR_SLOPE
        )
    )

    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            linewidth=1.2,
            label="Maximum correlation",
        )

        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            s=8,
        )

    outlier_mask = slope_results.get(
        "outlier_mask",
        np.zeros_like(ridge_mask, dtype=bool),
    )

    if np.any(outlier_mask):
        ax_map.scatter(
            best_shift_minutes[outlier_mask],
            selected_distances[outlier_mask],
            color="yellow",
            edgecolor="black",
            marker="x",
            s=35,
            linewidth=1.2,
            zorder=6,
            label="Rejected ridge outlier",
        )

    slope = slope_results[
        "slope_min_per_km"
    ]

    intercept = slope_results[
        "intercept_minutes"
    ]

    r_squared = slope_results[
        "r_squared"
    ]

    speed_m_per_s = slope_results[
        "speed_m_per_s"
    ]

    if np.isfinite(slope):
        fit_distances = np.linspace(
            SLOPE_FIT_DISTANCE_MIN_KM,
            SLOPE_FIT_DISTANCE_MAX_KM,
            300,
        )

        fit_shifts = (
            slope
            * fit_distances
            + intercept
        )

        ax_map.plot(
            fit_shifts,
            fit_distances,
            color="lime",
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        fit_text = (
            f"Shift slope = {slope:.5f} min/km\n"
            f"Equivalent speed = {speed_m_per_s:.2f} m/s\n"
            f"$R^2$ = {r_squared:.3f}\n"
            f"Outliers removed = "
            f"{slope_results.get('number_of_outliers_removed', 0)}"
        )

        ax_map.text(
            0.02,
            0.03,
            fit_text,
            transform=ax_map.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox={
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.85,
            },
        )

    ax_map.set_xlim(
        SHIFT_MIN_MINUTES,
        SHIFT_MAX_MINUTES,
    )

    ax_map.set_ylim(
        CORRELATION_DISTANCE_MIN_KM,
        CORRELATION_DISTANCE_MAX_KM,
    )

    ax_map.set_title(
        f"{angle_deg}° Shifted Row Correlation\n"
        f"{dataset_name}"
    )

    ax_map.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_map.set_ylabel(
        "Distance along slice (km)"
    )

    ax_map.legend(
        loc="upper right",
        fontsize=8,
    )

    # ==================================================
    # 0 km correlation-shift line
    # ==================================================
    ax_profile.plot(
        zero_km_profile_shifts_minutes,
        zero_km_correlation_profile,
        color="black",
        linewidth=1.4,
        label=f"{zero_km_distance:g} km correlation profile",
    )

    ax_profile.axvline(
        0.0,
        color="gray",
        linestyle="--",
        linewidth=1.0,
    )

    left_peak = zero_km_local_maxima[
        "left"
    ]

    right_peak = zero_km_local_maxima[
        "right"
    ]

    if np.isfinite(
        left_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            left_peak["shift_minutes"],
            left_peak["correlation"],
            s=55,
            marker="o",
            zorder=5,
            label=(
                "Left local max: "
                f"{left_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            left_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{left_peak['shift_minutes']:.2f} min\n"
                f"r={left_peak['correlation']:.3f}"
            ),
            xy=(
                left_peak["shift_minutes"],
                left_peak["correlation"],
            ),
            xytext=(8, 15),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    if np.isfinite(
        right_peak["shift_minutes"]
    ):
        ax_profile.scatter(
            right_peak["shift_minutes"],
            right_peak["correlation"],
            s=55,
            marker="s",
            zorder=5,
            label=(
                "Right local max: "
                f"{right_peak['shift_minutes']:.2f} min"
            ),
        )

        ax_profile.axvline(
            right_peak["shift_minutes"],
            linestyle=":",
            linewidth=1.2,
        )

        ax_profile.annotate(
            (
                f"{right_peak['shift_minutes']:.2f} min\n"
                f"r={right_peak['correlation']:.3f}"
            ),
            xy=(
                right_peak["shift_minutes"],
                right_peak["correlation"],
            ),
            xytext=(-8, 15),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8,
            arrowprops={
                "arrowstyle": "->",
                "linewidth": 0.8,
            },
        )

    ax_profile.set_xlim(
        ZERO_KM_PROFILE_SHIFT_MIN_MINUTES,
        ZERO_KM_PROFILE_SHIFT_MAX_MINUTES,
    )

    ax_profile.set_xlabel(
        "Shift applied to row (min)"
    )

    ax_profile.set_ylabel(
        "Correlation"
    )

    ax_profile.set_title(
        f"Correlation-Shift Line at {zero_km_distance:g} km"
    )

    ax_profile.grid(
        True,
        alpha=0.3,
    )

    ax_profile.legend(
        loc="best",
        fontsize=8,
    )

    plt.tight_layout()

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )

    plt.close(fig)


# ==================================================
# Plot component and six-variant profile comparison for one angle
# ==================================================
def plot_nine_panel_profile_comparison(
    angle_deg,
    variant_profile_data,
    output_file,
):
    """
    Create a nine-panel profile comparison for the rounded-bias output.

    Panel order:
      1. Waves only
      2. Bias only
      3. Temporal slope only, calculated as (waves + slope) - waves
      4. Waves + temporal slope
      5. Waves + directional bias
      6. Waves + temporal slope + directional bias
      7. Waves + temporal slope + directional bias + noise
      8. Variant 7 with row-wise linear temporal detrending
      9. Derived noise only, calculated as noisy - noise-free combined

    Every panel shows the processed profiles nearest 0, 125, and
    250 km.
    """
    waves_key = "01_waves_only"
    bias_only_key = "02_bias_only"
    waves_slope_key = "03_waves_plus_slope"
    waves_bias_key = "04_waves_plus_bias"
    combined_key = "05_waves_plus_slope_plus_bias"
    noisy_key = "06_waves_plus_slope_plus_bias_plus_noise"
    detrended_key = (
        "07_waves_plus_slope_plus_bias_plus_noise_detrended"
    )

    panel_specs = [
        {"key": waves_key, "label": "Waves only"},
        {"key": bias_only_key, "label": "Bias only"},
        {"key": "derived_slope_only", "label": "Temporal slope only"},
        {"key": waves_slope_key, "label": "Waves + temporal slope"},
        {"key": waves_bias_key, "label": "Waves + directional bias"},
        {"key": combined_key,
         "label": "Waves + temporal slope + directional bias"},
        {"key": noisy_key,
         "label": "Waves + temporal slope + directional bias + noise"},
        {"key": detrended_key,
         "label": "Noisy combined variant + linear detrend"},
        {"key": "derived_noise_only", "label": "Noise only"},
    ]

    plot_data = dict(variant_profile_data)

    waves_entry = variant_profile_data.get(waves_key)
    waves_slope_entry = variant_profile_data.get(waves_slope_key)
    combined_entry = variant_profile_data.get(combined_key)
    noisy_entry = variant_profile_data.get(noisy_key)

    def create_difference_entry(combined_entry, base_entry):
        if combined_entry is None or base_entry is None:
            return None

        combined_times = np.asarray(combined_entry["times_min"], dtype=float)
        base_times = np.asarray(base_entry["times_min"], dtype=float)
        combined_distances = np.asarray(
            combined_entry["distance_km"], dtype=float
        )
        base_distances = np.asarray(
            base_entry["distance_km"], dtype=float
        )

        if (
            combined_entry["keogram"].shape != base_entry["keogram"].shape
            or not np.allclose(combined_times, base_times, equal_nan=True)
            or not np.allclose(
                combined_distances, base_distances, equal_nan=True
            )
        ):
            return None

        return {
            "times_min": combined_times.copy(),
            "distance_km": combined_distances.copy(),
            "keogram": (
                np.asarray(combined_entry["keogram"], dtype=float)
                - np.asarray(base_entry["keogram"], dtype=float)
            ),
        }

    plot_data["derived_slope_only"] = create_difference_entry(
        waves_slope_entry,
        waves_entry,
    )
    plot_data["derived_noise_only"] = create_difference_entry(
        noisy_entry,
        combined_entry,
    )

    if not any(plot_data.get(spec["key"]) is not None for spec in panel_specs):
        return

    fig, axes = plt.subplots(
        nrows=5,
        ncols=2,
        figsize=(15, 19),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.asarray(axes).ravel()

    profile_colors = [
        "tab:blue",
        "tab:orange",
        "tab:green",
    ]

    for axis, panel_spec in zip(axes, panel_specs):
        profile_entry = plot_data.get(panel_spec["key"])

        if profile_entry is None:
            axis.text(
                0.5,
                0.5,
                "Profile data unavailable",
                transform=axis.transAxes,
                ha="center",
                va="center",
            )
            axis.set_title(panel_spec["label"])
            axis.grid(True, alpha=0.3)
            continue

        profile_times_min = np.asarray(
            profile_entry["times_min"], dtype=float
        )
        profile_keogram = np.asarray(
            profile_entry["keogram"], dtype=float
        )
        profile_distances_km = np.asarray(
            profile_entry["distance_km"], dtype=float
        )

        for target_distance, color in zip(
            PROFILE_COMPARISON_DISTANCES_KM,
            profile_colors,
        ):
            distance_index = int(
                np.nanargmin(
                    np.abs(profile_distances_km - target_distance)
                )
            )
            actual_distance = float(
                profile_distances_km[distance_index]
            )
            axis.plot(
                profile_times_min,
                profile_keogram[:, distance_index],
                color=color,
                linewidth=1.3,
                label=f"{actual_distance:g} km",
            )

        axis.axhline(
            0.0,
            color="black",
            linestyle="--",
            linewidth=0.8,
        )
        axis.set_xlim(
            CORRELATION_TIME_MIN,
            CORRELATION_TIME_MAX,
        )
        axis.set_ylabel("Value")
        axis.set_title(panel_spec["label"])
        axis.grid(True, alpha=0.3)
        axis.legend(loc="best", fontsize=8, ncol=3)

    for axis in axes[-2:]:
        axis.set_xlabel("Time (min)")

    fig.suptitle(
        f"{angle_deg}° Processed Keogram Profiles and Components\n"
        f"Profiles near 0, 125, and 250 km; "
        f"time range {CORRELATION_TIME_MIN:g}–"
        f"{CORRELATION_TIME_MAX:g} min",
        fontsize=14,
    )

    fig.savefig(
        output_file,
        dpi=DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


# ==================================================
# Print slope results
# ==================================================
def print_slope_results(
    angle_deg,
    label,
    results,
):
    print(
        f"\n{angle_deg}°{label}:"
    )

    print(
        f"  Shift slope: "
        f"{results['slope_min_per_km']:.8f} min/km"
    )

    print(
        f"  Intercept: "
        f"{results['intercept_minutes']:.6f} min"
    )

    print(
        f"  Equivalent speed: "
        f"{results['speed_m_per_s']:.4f} m/s"
    )

    print(
        f"  R²: "
        f"{results['r_squared']:.6f}"
    )

    print(
        f"  Points used: "
        f"{results['number_of_fit_points']}"
    )



# ==================================================
# Six phase-speed variants plus one profile-only bias variant
# ==================================================
# All six generator folders are read. Bias only is retained for profile and
# component diagnostics but is excluded from correlation-ridge and phase-speed
# calculations because it contains no propagating wave. The final analysis
# variant reuses the noisy combined output and applies row-by-row detrending.
VARIANT_SPECS = [
    {
        "folder": "01_waves_only",
        "source_folder": "01_waves_only",
        "label": "Waves only",
        "short_label": "waves_only",
        "apply_detrend": False,
        "calculate_phase_speed": True,
    },
    {
        "folder": "02_bias_only",
        "source_folder": "02_bias_only",
        "label": "Bias only",
        "short_label": "bias_only",
        "apply_detrend": False,
        "calculate_phase_speed": False,
    },
    {
        "folder": "03_waves_plus_slope",
        "source_folder": "03_waves_plus_slope",
        "label": "Waves + temporal slope",
        "short_label": "waves_plus_slope",
        "apply_detrend": False,
        "calculate_phase_speed": True,
    },
    {
        "folder": "04_waves_plus_bias",
        "source_folder": "04_waves_plus_bias",
        "label": "Waves + directional bias",
        "short_label": "waves_plus_bias",
        "apply_detrend": False,
        "calculate_phase_speed": True,
    },
    {
        "folder": "05_waves_plus_slope_plus_bias",
        "source_folder": "05_waves_plus_slope_plus_bias",
        "label": "Waves + temporal slope + directional bias",
        "short_label": "waves_plus_slope_plus_bias",
        "apply_detrend": False,
        "calculate_phase_speed": True,
    },
    {
        "folder": "06_waves_plus_slope_plus_bias_plus_noise",
        "source_folder": "06_waves_plus_slope_plus_bias_plus_noise",
        "label": "Waves + temporal slope + directional bias + noise",
        "short_label": "waves_plus_slope_plus_bias_plus_noise",
        "apply_detrend": False,
        "calculate_phase_speed": True,
    },
    {
        "folder": "07_waves_plus_slope_plus_bias_plus_noise_detrended",
        "source_folder": "06_waves_plus_slope_plus_bias_plus_noise",
        "label": (
            "Waves + temporal slope + directional bias + noise "
            "+ linear detrend"
        ),
        "short_label": (
            "waves_plus_slope_plus_bias_plus_noise_detrended"
        ),
        "apply_detrend": True,
        "calculate_phase_speed": True,
    },
]

# Explicitly disable all correction steps.
APPLY_BIAS_CORRECTION = False
APPLY_LINEAR_SLOPE_CORRECTION = False


# ==================================================
# Common output paths
# ==================================================
# Organize the analysis by the correlation time interval instead of
# the shift range. Smoothing sigma values are intentionally omitted
# from the folder hierarchy.
distance_folder = (
    f"r_"
    f"{safe_number_label(CORRELATION_DISTANCE_MIN_KM)}_"
    f"{safe_number_label(CORRELATION_DISTANCE_MAX_KM)}_km"
)

correlation_time_folder = (
    f"correlation_time_"
    f"{safe_number_label(CORRELATION_TIME_MIN)}_"
    f"{safe_number_label(CORRELATION_TIME_MAX)}_min"
)

OUT_DIR = (
    BASE_OUT_DIR
    / "correlation_analysis_seven_variants"
    / "no_bias_correction"
    / distance_folder
    / correlation_time_folder
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================
# Read common time array
# ==================================================
TIMES_FILE = BASE_OUT_DIR / "times_s.npy"

if not BASE_OUT_DIR.exists():
    raise FileNotFoundError(
        f"Generator output folder not found: {BASE_OUT_DIR.resolve()}"
    )

if not TIMES_FILE.exists():
    raise FileNotFoundError(
        f"Common time file not found: {TIMES_FILE.resolve()}"
    )

times_s = np.load(
    TIMES_FILE
).astype(float)

times_min = times_s / 60.0

if len(times_min) < 2:
    raise ValueError(
        "At least two time samples are required."
    )


# ==================================================
# Determine minutes per time pixel
# ==================================================
time_differences_min = np.diff(
    times_min
)

finite_time_differences = time_differences_min[
    np.isfinite(time_differences_min)
]

if len(finite_time_differences) == 0:
    raise ValueError(
        "Could not determine the time spacing."
    )

MINUTES_PER_PIXEL = np.median(
    finite_time_differences
)

if (
    not np.isfinite(MINUTES_PER_PIXEL)
    or MINUTES_PER_PIXEL <= 0
):
    raise ValueError(
        "The calculated minutes-per-pixel value is invalid."
    )

maximum_time_spacing_deviation = np.max(
    np.abs(
        finite_time_differences
        - MINUTES_PER_PIXEL
    )
)

print("Seven analysis variants to process:")
for variant in VARIANT_SPECS:
    print(
        f"  {variant['folder']}: "
        f"{variant['label']}"
        + (" [profile only]" if not variant["calculate_phase_speed"] else "")
    )

print("\nAngles to process:")
print(KEOGRAM_ANGLES_DEG)
print(
    f"Number of angles per variant: "
    f"{len(KEOGRAM_ANGLES_DEG)}"
)

print("\nCorrections:")
print(
    f"  Bias correction: "
    f"{APPLY_BIAS_CORRECTION}"
)
print(
    f"  Linear slope correction: "
    f"{APPLY_LINEAR_SLOPE_CORRECTION}"
)

print("\nTime sampling:")
print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL:.6f} minutes"
)
print(
    f"One time pixel = "
    f"{MINUTES_PER_PIXEL * 60.0:.3f} seconds"
)
print(
    f"Maximum spacing deviation = "
    f"{maximum_time_spacing_deviation:.6e} minutes"
)


# ==================================================
# Convert minute shifts to integer pixel shifts
# ==================================================
shift_min_pixels = int(
    np.ceil(
        SHIFT_MIN_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shift_max_pixels = int(
    np.floor(
        SHIFT_MAX_MINUTES
        / MINUTES_PER_PIXEL
    )
)

shifts_pixels = np.arange(
    shift_min_pixels,
    shift_max_pixels + 1,
    dtype=int,
)

shifts_minutes = (
    shifts_pixels
    * MINUTES_PER_PIXEL
)

print("\nShift range:")
print(
    f"Requested: "
    f"{SHIFT_MIN_MINUTES:.2f} to "
    f"{SHIFT_MAX_MINUTES:.2f} min"
)
print(
    f"Actual: "
    f"{shifts_minutes[0]:.2f} to "
    f"{shifts_minutes[-1]:.2f} min"
)
print(
    f"Pixel shifts: "
    f"{shifts_pixels[0]} to "
    f"{shifts_pixels[-1]}"
)

# Separate, four-times-wider shift array used only by the final
# 0 km correlation-profile subplot.
zero_km_shift_min_pixels = int(
    np.ceil(
        ZERO_KM_PROFILE_SHIFT_MIN_MINUTES
        / MINUTES_PER_PIXEL
    )
)

zero_km_shift_max_pixels = int(
    np.floor(
        ZERO_KM_PROFILE_SHIFT_MAX_MINUTES
        / MINUTES_PER_PIXEL
    )
)

zero_km_profile_shifts_pixels = np.arange(
    zero_km_shift_min_pixels,
    zero_km_shift_max_pixels + 1,
    dtype=int,
)

zero_km_profile_shifts_minutes = (
    zero_km_profile_shifts_pixels
    * MINUTES_PER_PIXEL
)

print("\n0 km profile shift range:")
print(
    f"Requested: "
    f"{ZERO_KM_PROFILE_SHIFT_MIN_MINUTES:.2f} to "
    f"{ZERO_KM_PROFILE_SHIFT_MAX_MINUTES:.2f} min"
)
print(
    f"Actual: "
    f"{zero_km_profile_shifts_minutes[0]:.2f} to "
    f"{zero_km_profile_shifts_minutes[-1]:.2f} min"
)


# ==================================================
# Select common time range
# ==================================================
time_mask = (
    (times_min >= CORRELATION_TIME_MIN)
    & (times_min <= CORRELATION_TIME_MAX)
)

if np.count_nonzero(time_mask) < MIN_VALID_CORRELATION_POINTS:
    raise ValueError(
        "The selected time interval has too few samples."
    )

times_correlation_min = times_min[
    time_mask
]


# ==================================================
# Save common arrays
# ==================================================
np.save(
    OUT_DIR / "angles_deg.npy",
    KEOGRAM_ANGLES_DEG,
)
np.save(
    OUT_DIR / "shifts_minutes.npy",
    shifts_minutes,
)
np.save(
    OUT_DIR / "shifts_pixels.npy",
    shifts_pixels,
)
np.save(
    OUT_DIR / "zero_km_profile_shifts_minutes.npy",
    zero_km_profile_shifts_minutes,
)
np.save(
    OUT_DIR / "zero_km_profile_shifts_pixels.npy",
    zero_km_profile_shifts_pixels,
)
np.save(
    OUT_DIR / "correlation_times_min.npy",
    times_correlation_min,
)


# ==================================================
# Process all seven analysis variants
# ==================================================
all_variant_results = {}
all_variant_peak_results = {}
all_failures = {}

# Store processed limited-range keograms so that, after all variants
# are complete, one nine-panel profile comparison can be made for
# every angle.
all_angle_variant_profiles = {
    int(angle_deg): {}
    for angle_deg in KEOGRAM_ANGLES_DEG
}

for variant_number, variant in enumerate(
    VARIANT_SPECS,
    start=1,
):
    variant_folder = variant["folder"]
    source_folder = variant["source_folder"]
    variant_label = variant["label"]
    variant_short_label = variant["short_label"]
    apply_detrend = bool(variant["apply_detrend"])
    calculate_phase_speed = bool(variant["calculate_phase_speed"])

    print("\n" + "#" * 76)
    print(
        f"VARIANT {variant_number}/{len(VARIANT_SPECS)}: "
        f"{variant_label}"
    )
    print("#" * 76)

    keogram_input_dir = (
        BASE_OUT_DIR
        / source_folder
        / "keogram_arrays"
    )

    variant_out_dir = (
        OUT_DIR
        / variant_folder
    )
    plots_out_dir = variant_out_dir / "plots"
    arrays_out_dir = variant_out_dir / "arrays"
    reports_out_dir = variant_out_dir / "slope_reports"

    for directory in [
        variant_out_dir,
        plots_out_dir,
        arrays_out_dir,
        reports_out_dir,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    variant_results = []
    variant_peak_results = []
    successful_angles = []
    failed_angles = []

    for angle_number, angle_deg in enumerate(
        KEOGRAM_ANGLES_DEG,
        start=1,
    ):
        print("\n" + "=" * 70)
        print(
            f"{variant_label}: angle {angle_deg}° "
            f"({angle_number}/{len(KEOGRAM_ANGLES_DEG)})"
        )
        print("=" * 70)

        keogram_file = (
            keogram_input_dir
            / f"synthetic_keogram_angle_{angle_deg:03d}.npy"
        )

        if not keogram_file.exists():
            reason = f"file not found: {keogram_file}"
            print(f"WARNING: {reason}")
            failed_angles.append((int(angle_deg), reason))
            continue

        try:
            keogram_original = np.load(
                keogram_file
            ).astype(float)

            if keogram_original.ndim != 2:
                raise ValueError(
                    f"Expected a 2D keogram, found "
                    f"{keogram_original.shape}."
                )

            nt, nspace = keogram_original.shape

            if len(times_min) != nt:
                raise ValueError(
                    f"Keogram has {nt} time samples, but "
                    f"times_s.npy has {len(times_min)}."
                )

            # The generator uses 1 km spacing and centers the cut at zero.
            # Prefer the exact saved distance coordinates when available.
            saved_distance_file = (
                BASE_OUT_DIR / "00_components" / "keogram_distance_km.npy"
            )
            if saved_distance_file.exists():
                distance_km = np.load(saved_distance_file).astype(float)
                if len(distance_km) != nspace:
                    raise ValueError(
                        "Saved keogram_distance_km.npy length does not match "
                        "the keogram distance dimension."
                    )
            else:
                distance_km = (
                    np.arange(nspace, dtype=float)
                    - nspace // 2
                )

            available_distance_min = np.nanmin(
                distance_km
            )
            available_distance_max = np.nanmax(
                distance_km
            )

            if (
                CORRELATION_DISTANCE_MIN_KM
                < available_distance_min
                or CORRELATION_DISTANCE_MAX_KM
                > available_distance_max
            ):
                raise ValueError(
                    f"Requested distance range "
                    f"{CORRELATION_DISTANCE_MIN_KM} to "
                    f"{CORRELATION_DISTANCE_MAX_KM} km "
                    f"is outside available range "
                    f"{available_distance_min} to "
                    f"{available_distance_max} km."
                )

            keogram_time_selected = keogram_original[
                time_mask,
                :,
            ]

            if APPLY_2D_SMOOTHING:
                keogram_processed = smooth_keogram_2d(
                    data=keogram_time_selected,
                    sigma_time=SMOOTH_SIGMA_TIME_PIXELS,
                    sigma_distance=SMOOTH_SIGMA_DISTANCE_PIXELS,
                    mode=SMOOTH_MODE,
                    weight_tolerance=SMOOTH_WEIGHT_TOLERANCE,
                )
                processing_label = (
                    f"smoothed σt={SMOOTH_SIGMA_TIME_PIXELS:g}, "
                    f"σd={SMOOTH_SIGMA_DISTANCE_PIXELS:g}"
                )
            else:
                keogram_processed = (
                    keogram_time_selected.copy()
                )
                processing_label = "unsmoothed"

            detrend_trend = None
            detrend_slopes = None
            detrend_intercepts = None

            if apply_detrend:
                (
                    keogram_processed,
                    detrend_trend,
                    detrend_slopes,
                    detrend_intercepts,
                ) = detrend_keogram_rows(
                    data=keogram_processed,
                    times_min=times_correlation_min,
                    minimum_points=3,
                )

                dataset_name = (
                    f"{variant_label}; {processing_label}; "
                    "independent linear time trend removed from each row"
                )
            else:
                dataset_name = (
                    f"{variant_label}; no correction; {processing_label}"
                )

            selected_distance_mask = (
                (distance_km >= CORRELATION_DISTANCE_MIN_KM)
                & (distance_km <= CORRELATION_DISTANCE_MAX_KM)
            )

            selected_distances = distance_km[
                selected_distance_mask
            ]

            if len(selected_distances) == 0:
                raise ValueError(
                    "No rows lie inside the selected distance range."
                )

            all_angle_variant_profiles[int(angle_deg)][variant_folder] = {
                "times_min": times_correlation_min.copy(),
                "distance_km": selected_distances.copy(),
                "keogram": keogram_processed[
                    :,
                    selected_distance_mask,
                ].copy(),
            }

            if not calculate_phase_speed:
                print(
                    "Skipping correlation ridge and phase-speed calculation: "
                    "this variant contains no propagating wave."
                )
                successful_angles.append(int(angle_deg))
                continue

            angle_plot_dir = (
                plots_out_dir
                / f"angle_{angle_deg:03d}"
            )
            angle_array_dir = (
                arrays_out_dir
                / f"angle_{angle_deg:03d}"
            )
            angle_plot_dir.mkdir(
                parents=True,
                exist_ok=True,
            )
            angle_array_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            (
                correlation_map,
                selected_distances_map,
                reference_index,
            ) = calculate_correlation_map(
                keogram=keogram_processed,
                distance_km=distance_km,
                reference_distance_km=0.0,
                selected_distance_mask=selected_distance_mask,
                shifts_pixels=shifts_pixels,
                minimum_points=MIN_VALID_CORRELATION_POINTS,
            )

            (
                best_shift_minutes,
                maximum_correlation,
            ) = find_best_shifts(
                correlation_map=correlation_map,
                shifts_minutes=shifts_minutes,
            )

            slope_results = fit_maximum_correlation_slope(
                distances_km=selected_distances_map,
                best_shift_minutes=best_shift_minutes,
                maximum_correlation=maximum_correlation,
                minimum_correlation=MIN_CORRELATION_FOR_SLOPE,
                distance_min_km=SLOPE_FIT_DISTANCE_MIN_KM,
                distance_max_km=SLOPE_FIT_DISTANCE_MAX_KM,
                remove_outliers=REMOVE_RIDGE_OUTLIERS,
                maximum_ridge_jump_minutes=RIDGE_MAXIMUM_JUMP_MINUTES,
                outlier_sigma=RIDGE_OUTLIER_SIGMA,
                outlier_max_iterations=RIDGE_OUTLIER_MAX_ITERATIONS,
                outlier_min_points=RIDGE_OUTLIER_MIN_POINTS,
            )

            # Calculate the 0 km autocorrelation separately using the
            # four-times-wider shift range. This affects only the final
            # correlation-profile subplot and its saved profile arrays.
            zero_km_full_index = int(
                np.argmin(np.abs(distance_km - 0.0))
            )
            zero_km_distance = float(
                distance_km[zero_km_full_index]
            )
            zero_km_time_series = keogram_processed[
                :, zero_km_full_index
            ]

            zero_km_profile = np.array(
                [
                    calculate_shifted_correlation(
                        reference_row=zero_km_time_series,
                        comparison_row=zero_km_time_series,
                        shift_pixels=int(shift_pixels),
                        minimum_points=MIN_VALID_CORRELATION_POINTS,
                    )
                    for shift_pixels in zero_km_profile_shifts_pixels
                ],
                dtype=float,
            )

            zero_km_peaks = find_left_and_right_local_maxima(
                correlation_profile=zero_km_profile,
                shifts_minutes=zero_km_profile_shifts_minutes,
                minimum_prominence=LOCAL_MAXIMUM_MIN_PROMINENCE,
                minimum_distance_pixels=(
                    LOCAL_MAXIMUM_MIN_DISTANCE_PIXELS
                ),
            )

            plot_file = (
                angle_plot_dir
                / (
                    f"correlation_minutes_"
                    f"{variant_short_label}_"
                    f"{angle_deg:03d}.png"
                )
            )

            plot_correlation_map(
                angle_deg=angle_deg,
                full_keogram=keogram_original,
                full_times_min=times_min,
                full_distance_km=distance_km,
                limited_keogram=keogram_processed[
                    :,
                    selected_distance_mask,
                ],
                limited_times_min=times_correlation_min,
                limited_distance_km=selected_distances_map,
                correlation_map=correlation_map,
                shifts_minutes=shifts_minutes,
                selected_distances=selected_distances_map,
                best_shift_minutes=best_shift_minutes,
                maximum_correlation=maximum_correlation,
                slope_results=slope_results,
                zero_km_correlation_profile=zero_km_profile,
                zero_km_profile_shifts_minutes=(
                    zero_km_profile_shifts_minutes
                ),
                zero_km_distance=zero_km_distance,
                zero_km_local_maxima=zero_km_peaks,
                dataset_name=dataset_name,
                output_file=plot_file,
            )

            np.save(
                angle_array_dir / "correlation_map.npy",
                correlation_map,
            )
            np.save(
                angle_array_dir / "best_shift_minutes.npy",
                best_shift_minutes,
            )
            np.save(
                angle_array_dir / "maximum_correlation.npy",
                maximum_correlation,
            )
            np.save(
                angle_array_dir / "distances_km.npy",
                selected_distances_map,
            )
            np.save(
                angle_array_dir / "zero_km_correlation_profile.npy",
                zero_km_profile,
            )
            np.save(
                angle_array_dir / "zero_km_profile_shifts_minutes.npy",
                zero_km_profile_shifts_minutes,
            )
            np.save(
                angle_array_dir / "zero_km_profile_shifts_pixels.npy",
                zero_km_profile_shifts_pixels,
            )

            if apply_detrend:
                np.save(
                    angle_array_dir / "detrended_keogram.npy",
                    keogram_processed,
                )
                np.save(
                    angle_array_dir / "fitted_linear_trend.npy",
                    detrend_trend,
                )
                np.save(
                    angle_array_dir / "detrend_slopes_per_distance.npy",
                    detrend_slopes,
                )
                np.save(
                    angle_array_dir / "detrend_intercepts_per_distance.npy",
                    detrend_intercepts,
                )

            np.savetxt(
                angle_array_dir / "zero_km_correlation_profile.csv",
                np.column_stack(
                    [zero_km_profile_shifts_minutes, zero_km_profile]
                ),
                delimiter=",",
                header="shift_minutes,correlation",
                comments="",
            )

            np.savetxt(
                angle_array_dir / "ridge.csv",
                np.column_stack(
                    [
                        selected_distances_map,
                        best_shift_minutes,
                        maximum_correlation,
                        slope_results["fit_mask"].astype(int),
                        slope_results["outlier_mask"].astype(int),
                    ]
                ),
                delimiter=",",
                header=(
                    "distance_km,best_shift_minutes,"
                    "maximum_correlation,used_in_fit,rejected_outlier"
                ),
                comments="",
            )

            slope_report_file = (
                reports_out_dir
                / f"correlation_slope_{angle_deg:03d}.txt"
            )

            with open(
                slope_report_file,
                "w",
                encoding="utf-8",
            ) as file:
                file.write(f"Variant: {variant_label}\n")
                file.write(f"Angle: {angle_deg} degrees\n")
                file.write(
                    f"Bias correction: "
                    f"{APPLY_BIAS_CORRECTION}\n"
                )
                file.write(
                    f"Row-wise linear detrend: "
                    f"{apply_detrend}\n"
                )
                file.write(
                    f"Linear slope correction: "
                    f"{APPLY_LINEAR_SLOPE_CORRECTION}\n"
                )
                file.write(
                    f"Minutes per pixel: "
                    f"{MINUTES_PER_PIXEL:.10f}\n\n"
                )

                file.write("RIDGE FIT\n")
                for key, value in slope_results.items():
                    if key not in ["fit_mask", "outlier_mask"]:
                        file.write(f"{key}: {value}\n")

                file.write(
                    "\n0 KM CORRELATION-PROFILE LOCAL MAXIMA\n"
                )
                file.write(
                    "Left shift (min): "
                    f"{zero_km_peaks['left']['shift_minutes']}\n"
                )
                file.write(
                    "Left correlation: "
                    f"{zero_km_peaks['left']['correlation']}\n"
                )
                file.write(
                    "Right shift (min): "
                    f"{zero_km_peaks['right']['shift_minutes']}\n"
                )
                file.write(
                    "Right correlation: "
                    f"{zero_km_peaks['right']['correlation']}\n"
                )

            variant_results.append(
                [
                    float(angle_deg),
                    slope_results["slope_min_per_km"],
                    slope_results["intercept_minutes"],
                    slope_results["speed_km_per_min"],
                    slope_results["speed_m_per_s"],
                    slope_results["r_squared"],
                    slope_results["number_of_fit_points"],
                    slope_results["number_of_outliers_removed"],
                ]
            )

            variant_peak_results.append(
                [
                    float(angle_deg),
                    zero_km_peaks["left"]["shift_minutes"],
                    zero_km_peaks["left"]["correlation"],
                    zero_km_peaks["right"]["shift_minutes"],
                    zero_km_peaks["right"]["correlation"],
                ]
            )

            successful_angles.append(int(angle_deg))

            print_slope_results(
                angle_deg=angle_deg,
                label=(
                    f" {variant_label} maximum-correlation ridge"
                ),
                results=slope_results,
            )

        except Exception as error:
            print(
                f"ERROR processing {variant_label}, "
                f"{angle_deg}°: {error}"
            )
            failed_angles.append(
                (int(angle_deg), str(error))
            )

    variant_results = np.asarray(
        variant_results,
        dtype=float,
    )
    variant_peak_results = np.asarray(
        variant_peak_results,
        dtype=float,
    )

    summary_header = (
        "angle_deg,slope_min_per_km,intercept_minutes,"
        "speed_km_per_min,speed_m_per_s,r_squared,"
        "number_of_fit_points,number_of_outliers_removed"
    )

    peak_header = (
        "angle_deg,left_shift_minutes,left_correlation,"
        "right_shift_minutes,right_correlation"
    )

    if variant_results.size > 0:
        np.savetxt(
            variant_out_dir / "all_angle_slope_summary.csv",
            variant_results,
            delimiter=",",
            header=summary_header,
            comments="",
        )
        np.save(
            variant_out_dir / "all_angle_slope_summary.npy",
            variant_results,
        )

        angles_processed = variant_results[:, 0]
        speed_m_per_s = variant_results[:, 4]
        r_squared = variant_results[:, 5]

        fig_speed, ax_speed = plt.subplots(
            figsize=(10, 5)
        )
        ax_speed.plot(
            angles_processed,
            speed_m_per_s,
            marker="o",
            linewidth=1.5,
        )
        ax_speed.axhline(
            0.0,
            color="black",
            linestyle="--",
            linewidth=1.0,
        )
        ax_speed.set_xlim(
            KEOGRAM_ANGLES_DEG.min(),
            KEOGRAM_ANGLES_DEG.max(),
        )
        ax_speed.set_xlabel("Keogram angle (degrees)")
        ax_speed.set_ylabel("Equivalent signed speed (m/s)")
        ax_speed.set_title(
            f"Maximum-Correlation Ridge Speed versus Angle\n"
            f"{variant_label}"
        )
        ax_speed.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_speed.savefig(
            variant_out_dir / "speed_vs_angle.png",
            dpi=DPI,
            bbox_inches="tight",
        )
        plt.close(fig_speed)

        fig_r2, ax_r2 = plt.subplots(
            figsize=(10, 5)
        )
        ax_r2.plot(
            angles_processed,
            r_squared,
            marker="o",
            linewidth=1.5,
        )
        ax_r2.set_xlim(
            KEOGRAM_ANGLES_DEG.min(),
            KEOGRAM_ANGLES_DEG.max(),
        )
        ax_r2.set_ylim(0.0, 1.0)
        ax_r2.set_xlabel("Keogram angle (degrees)")
        ax_r2.set_ylabel("R²")
        ax_r2.set_title(
            f"Maximum-Correlation Ridge Fit R² versus Angle\n"
            f"{variant_label}"
        )
        ax_r2.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_r2.savefig(
            variant_out_dir / "r_squared_vs_angle.png",
            dpi=DPI,
            bbox_inches="tight",
        )
        plt.close(fig_r2)

    if variant_peak_results.size > 0:
        np.savetxt(
            variant_out_dir / "all_angle_zero_km_local_maxima.csv",
            variant_peak_results,
            delimiter=",",
            header=peak_header,
            comments="",
        )
        np.save(
            variant_out_dir / "all_angle_zero_km_local_maxima.npy",
            variant_peak_results,
        )

    with open(
        variant_out_dir / "processing_log.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(f"Variant: {variant_label}\n")
        file.write(
            f"Bias correction: {APPLY_BIAS_CORRECTION}\n"
        )
        file.write(
            f"Row-wise linear detrend: {apply_detrend}\n"
        )
        file.write(
            "Linear slope correction: "
            f"{APPLY_LINEAR_SLOPE_CORRECTION}\n"
        )
        file.write(
            f"Number requested: {len(KEOGRAM_ANGLES_DEG)}\n"
        )
        file.write(
            f"Number successful: {len(successful_angles)}\n"
        )
        file.write(
            f"Number failed: {len(failed_angles)}\n\n"
        )
        file.write("Successful angles:\n")
        file.write(
            ", ".join(str(x) for x in successful_angles)
            + "\n\n"
        )
        file.write("Failed angles:\n")
        if failed_angles:
            for failed_angle, reason in failed_angles:
                file.write(
                    f"{failed_angle}: {reason}\n"
                )
        else:
            file.write("None\n")

    all_variant_results[variant_folder] = variant_results
    all_variant_peak_results[variant_folder] = variant_peak_results
    all_failures[variant_folder] = failed_angles


# ==================================================
# Nine-panel profile and component comparison for every angle
# ==================================================
profile_comparison_out_dir = (
    OUT_DIR
    / "nine_panel_profile_comparisons"
)
profile_comparison_out_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for angle_deg in KEOGRAM_ANGLES_DEG:
    angle_profile_data = all_angle_variant_profiles.get(
        int(angle_deg),
        {},
    )

    if len(angle_profile_data) == 0:
        print(
            f"WARNING: no profile data available for "
            f"{angle_deg}°."
        )
        continue

    profile_comparison_file = (
        profile_comparison_out_dir
        / (
            f"profiles_components_and_seven_analysis_variants_"
            f"angle_{int(angle_deg):03d}.png"
        )
    )

    plot_nine_panel_profile_comparison(
        angle_deg=int(angle_deg),
        variant_profile_data=angle_profile_data,
        output_file=profile_comparison_file,
    )


# ==================================================
# Combined seven-variant comparison plots
# ==================================================
# Plot phase speed for all variants together, restricted to 0–90°.
fig_speed_all, ax_speed_all = plt.subplots(
    figsize=(11, 6)
)

have_speed_data = False

phase_speed_plot_styles = [
    {"color": "tab:blue", "linestyle": "-", "marker": "o"},
    {"color": "tab:orange", "linestyle": "--", "marker": "x"},
    {"color": "tab:green", "linestyle": "-", "marker": "o"},
    {"color": "tab:purple", "linestyle": "-", "marker": "s"},
]

# Variants eligible for phase-speed plotting, in their original order:
#   1. Waves only
#   2. Waves + temporal slope
#   3. Waves + directional bias
#   4. Waves + temporal slope + directional bias
#   5. Waves + temporal slope + directional bias + noise
#   6. Noisy combined variant + linear detrend
#
# Skip the 4th and 6th curves in the combined phase-speed plot to
# reduce clutter. They are still fully processed and saved elsewhere.
phase_speed_variants_all = [
    variant for variant in VARIANT_SPECS
    if variant["calculate_phase_speed"]
]

PHASE_SPEED_PLOT_SKIP_INDICES = {
    4,
    6,
}

phase_speed_variants = [
    variant
    for plot_index, variant in enumerate(
        phase_speed_variants_all,
        start=1,
    )
    if plot_index not in PHASE_SPEED_PLOT_SKIP_INDICES
]

for variant, plot_style in zip(
    phase_speed_variants,
    phase_speed_plot_styles,
):
    results = all_variant_results.get(
        variant["folder"],
        np.empty((0, 8)),
    )

    if results.size == 0:
        continue

    phase_speed_angle_mask = (
        (results[:, 0] >= PHASE_SPEED_PLOT_ANGLE_MIN_DEG)
        & (results[:, 0] <= PHASE_SPEED_PLOT_ANGLE_MAX_DEG)
    )

    phase_speed_results = results[
        phase_speed_angle_mask
    ]

    if phase_speed_results.size == 0:
        continue

    have_speed_data = True
    ax_speed_all.plot(
        phase_speed_results[:, 0],
        phase_speed_results[:, 4],
        color=plot_style["color"],
        linestyle=plot_style["linestyle"],
        marker=plot_style["marker"],
        linewidth=1.3,
        markersize=5,
        label=variant["label"],
    )

if have_speed_data:
    ax_speed_all.axhline(
        450.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
        label="450 m/s",
    )
    ax_speed_all.set_xlim(
        PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
        PHASE_SPEED_PLOT_ANGLE_MAX_DEG,
    )
    ax_speed_all.set_xticks(
        np.arange(
            PHASE_SPEED_PLOT_ANGLE_MIN_DEG,
            PHASE_SPEED_PLOT_ANGLE_MAX_DEG + 1.0,
            10.0,
        )
    )
    ax_speed_all.set_xlabel("Keogram angle (degrees)")
    ax_speed_all.set_ylabel("Signed phase speed (m/s)")
    ax_speed_all.set_title(
        "Phase Speed versus Angle: Selected Analysis Variants\n"
        f"Correlation time range: "
        f"{CORRELATION_TIME_MIN:g}–{CORRELATION_TIME_MAX:g} min"
    )
    ax_speed_all.grid(True, alpha=0.3)
    ax_speed_all.legend(loc="best", fontsize=8)
    plt.tight_layout()
    fig_speed_all.savefig(
        OUT_DIR / "phase_speed_vs_angle_minus75_to_plus75_selected_variants.png",
        dpi=DPI,
        bbox_inches="tight",
    )

plt.close(fig_speed_all)


fig_r2_all, ax_r2_all = plt.subplots(
    figsize=(11, 6)
)

have_r2_data = False

for variant in phase_speed_variants:
    results = all_variant_results.get(
        variant["folder"],
        np.empty((0, 8)),
    )

    if results.size == 0:
        continue

    have_r2_data = True
    ax_r2_all.plot(
        results[:, 0],
        results[:, 5],
        marker="o",
        linewidth=1.3,
        label=variant["label"],
    )

if have_r2_data:
    ax_r2_all.set_xlim(
        KEOGRAM_ANGLES_DEG.min(),
        KEOGRAM_ANGLES_DEG.max(),
    )
    ax_r2_all.set_ylim(0.0, 1.0)
    ax_r2_all.set_xlabel("Keogram angle (degrees)")
    ax_r2_all.set_ylabel("R²")
    ax_r2_all.set_title(
        "Maximum-Correlation Ridge Fit R²: Seven Analysis Variants\n"
        "No bias or linear-slope correction"
    )
    ax_r2_all.grid(True, alpha=0.3)
    ax_r2_all.legend(loc="best", fontsize=8)
    plt.tight_layout()
    fig_r2_all.savefig(
        OUT_DIR / "r_squared_vs_angle_seven_variants.png",
        dpi=DPI,
        bbox_inches="tight",
    )

plt.close(fig_r2_all)


# ==================================================
# Save master summary combining all seven variants
# ==================================================
master_rows = []

for variant_index, variant in enumerate(
    VARIANT_SPECS,
    start=1,
):
    results = all_variant_results.get(
        variant["folder"],
        np.empty((0, 8)),
    )

    for row in results:
        master_rows.append(
            [float(variant_index), *row.tolist()]
        )

master_rows = np.asarray(
    master_rows,
    dtype=float,
)

if master_rows.size > 0:
    np.savetxt(
        OUT_DIR / "all_variants_all_angles_summary.csv",
        master_rows,
        delimiter=",",
        header=(
            "variant_index,angle_deg,slope_min_per_km,"
            "intercept_minutes,speed_km_per_min,speed_m_per_s,"
            "r_squared,number_of_fit_points,"
            "number_of_outliers_removed"
        ),
        comments="",
    )
    np.save(
        OUT_DIR / "all_variants_all_angles_summary.npy",
        master_rows,
    )

with open(
    OUT_DIR / "variant_key.txt",
    "w",
    encoding="utf-8",
) as file:
    for variant_index, variant in enumerate(
        VARIANT_SPECS,
        start=1,
    ):
        file.write(
            f"{variant_index}: {variant['folder']} = "
            f"{variant['label']}; "
            f"source={variant['source_folder']}; "
            f"detrend={variant['apply_detrend']}\n"
        )


# ==================================================
# Final summary
# ==================================================
print("\n" + "=" * 76)
print("SEVEN-VARIANT SYNTHETIC-KEOGRAM PROCESSING COMPLETE")
print("=" * 76)
print(f"Bias correction: {APPLY_BIAS_CORRECTION}")
print(
    "Linear slope correction: "
    f"{APPLY_LINEAR_SLOPE_CORRECTION}"
)

for variant in VARIANT_SPECS:
    results = all_variant_results.get(
        variant["folder"],
        np.empty((0, 8)),
    )
    failures = all_failures.get(
        variant["folder"],
        [],
    )
    print(
        f"{variant['label']}: "
        f"{len(results)} successful, "
        f"{len(failures)} failed"
    )

print("\nOutput folder:")
print(OUT_DIR.resolve())


Seven analysis variants to process:
  01_waves_only: Waves only
  02_bias_only: Bias only [profile only]
  03_waves_plus_slope: Waves + temporal slope
  04_waves_plus_bias: Waves + directional bias
  05_waves_plus_slope_plus_bias: Waves + temporal slope + directional bias
  06_waves_plus_slope_plus_bias_plus_noise: Waves + temporal slope + directional bias + noise
  07_waves_plus_slope_plus_bias_plus_noise_detrended: Waves + temporal slope + directional bias + noise + linear detrend

Angles to process:
[-90 -85 -80 -75 -70 -65 -60 -55 -50 -45 -40 -35 -30 -25 -20 -15 -10  -5
   0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90]
Number of angles per variant: 37

Corrections:
  Bias correction: False
  Linear slope correction: False

Time sampling:
One time pixel = 2.000000 minutes
One time pixel = 120.000 seconds
Maximum spacing deviation = 0.000000e+00 minutes

Shift range:
Requested: -50.00 to 50.00 min
Actual: -50.00 to 50.00 min
Pixel shifts: -25 to 25

0 km 

/var/folders/q8/62bq5d2s56b67cdzjyvxn4vc0000gn/T/ipykernel_15705/294507663.py:1493: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax_map.scatter(



-90° Waves + temporal slope + directional bias + noise maximum-correlation ridge:
  Shift slope: 0.00000000 min/km
  Intercept: 0.000000 min
  Equivalent speed: nan m/s
  R²: nan
  Points used: 250

Waves + temporal slope + directional bias + noise: angle -85° (2/37)

-85° Waves + temporal slope + directional bias + noise maximum-correlation ridge:
  Shift slope: 0.00000000 min/km
  Intercept: 0.000000 min
  Equivalent speed: nan m/s
  R²: nan
  Points used: 251

Waves + temporal slope + directional bias + noise: angle -80° (3/37)

-80° Waves + temporal slope + directional bias + noise maximum-correlation ridge:
  Shift slope: 0.00829141 min/km
  Intercept: -0.510529 min
  Equivalent speed: 2010.1135 m/s
  R²: 0.465573
  Points used: 251

Waves + temporal slope + directional bias + noise: angle -75° (4/37)

-75° Waves + temporal slope + directional bias + noise maximum-correlation ridge:
  Shift slope: 0.01122671 min/km
  Intercept: -0.184216 min
  Equivalent speed: 1484.5545 m/s
  R²